In [ ]:
# Read-only checks. Run in a fresh Colab notebook in the account that owns A3IDPS_COLAB.
from google.colab import drive
drive.mount('/content/drive')

import json, joblib, glob
import numpy as np, pandas as pd

BASE = "/content/drive/MyDrive/A3IDPS_COLAB"
NSL = f"{BASE}/data_processed/nslkdd"

# 1) Is the label an input column in NSL-KDD?
cols = json.load(open(f"{NSL}/nsl_feature_list.json"))
print("NSL input dim:", len(cols), "| label_5class in inputs:", "label_5class" in cols)
if "label_5class" in cols:
    i = cols.index("label_5class")
    sc = joblib.load(f"{NSL}/nsl_scaler.joblib")
    X_te = np.load(f"{NSL}/X_test.npy"); y_te = np.load(f"{NSL}/y_test.npy")
    recovered = np.round(X_te[:, i] * sc.scale_[i] + sc.mean_[i])
    print("column index:", i, "| test rows where that column != true label:",
          int((recovered != y_te).sum()), "of", len(y_te))

# 2) mailbomb rows in raw KDDTest+ (mapped to U2R by the notebook's label function)
te = pd.read_csv(f"{BASE}/data_raw/NSL-KDD/KDDTest+.txt", header=None)
print("mailbomb rows in KDDTest+:", int((te[41] == "mailbomb").sum()))

# 3) Are CICIDS features in [0,1]? Read one processed shard directly.
feat = json.load(open(f"{BASE}/artifacts/feature_list.json"))
shard = sorted(glob.glob(f"{BASE}/data_processed/cicids_clean_shards_full/*.csv"))[0]
X = pd.read_csv(shard, usecols=feat, nrows=200000).values.astype("float32")
print("CICIDS shard min/max:", float(X.min()), float(X.max()),
      "| share of values outside [0,1]:", float(((X < 0) | (X > 1)).mean()))

Mounted at /content/drive
NSL input dim: 123 | label_5class in inputs: True
column index: 38 | test rows where that column != true label: 0 of 22544
mailbomb rows in KDDTest+: 293
CICIDS shard min/max: -12.0 2070000000.0 | share of values outside [0,1]: 0.5484920007867532


In [ ]:
# Stage 1 - NSL-KDD corrected pipeline (preprocessing + undefended baseline + attack sanity checks)
# Paste into Colab cell by cell (cells are separated by "# %%"). One seed. Writes only to *_v2 folders.

# %% [0] Setup
from google.colab import drive
drive.mount('/content/drive')

import os, json, random, hashlib
import numpy as np, pandas as pd, joblib
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

BASE = "/content/drive/MyDrive/A3IDPS_COLAB"
RAW  = f"{BASE}/data_raw/NSL-KDD"
OUT  = f"{BASE}/data_processed/nslkdd_v2"
MOD  = f"{BASE}/models/nsl_v2"
RES  = f"{BASE}/results/nsl_v2"
for p in (OUT, MOD, RES):
    os.makedirs(p, exist_ok=True)

SPLIT_SEED = 42          # fixed data split (same for every model and seed)
SEED = 0                 # model seed (later: 0..4)
CLASS_NAMES = ["Normal", "DoS", "Probe", "R2L", "U2R"]
CLASS_WEIGHT_CAP = 20.0  # same cap for every model
BATCH = 512

def set_seed(s):
    random.seed(s); np.random.seed(s); tf.keras.utils.set_random_seed(s)

def jsonable(o):
    if isinstance(o, dict):  return {k: jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [jsonable(v) for v in o]
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    return o

# %% [1] Load official KDDTrain+ / KDDTest+ and map labels (fails loudly on unknown labels)
COLS = ["duration","protocol_type","service","flag","src_bytes","dst_bytes","land","wrong_fragment",
        "urgent","hot","num_failed_logins","logged_in","num_compromised","root_shell","su_attempted",
        "num_root","num_file_creations","num_shells","num_access_files","num_outbound_cmds",
        "is_host_login","is_guest_login","count","srv_count","serror_rate","srv_serror_rate",
        "rerror_rate","srv_rerror_rate","same_srv_rate","diff_srv_rate","srv_diff_host_rate",
        "dst_host_count","dst_host_srv_count","dst_host_same_srv_rate","dst_host_diff_srv_rate",
        "dst_host_same_src_port_rate","dst_host_srv_diff_host_rate","dst_host_serror_rate",
        "dst_host_srv_serror_rate","dst_host_rerror_rate","dst_host_srv_rerror_rate",
        "label","difficulty"]

GROUPS = {
    0: {"normal"},
    1: {"back","land","neptune","pod","smurf","teardrop","apache2","udpstorm","processtable","worm","mailbomb"},
    2: {"satan","ipsweep","nmap","portsweep","mscan","saint"},
    3: {"guess_passwd","ftp_write","imap","phf","multihop","warezmaster","warezclient","spy",
        "xlock","xsnoop","snmpguess","snmpgetattack","httptunnel","sendmail","named"},
    4: {"buffer_overflow","loadmodule","rootkit","perl","sqlattack","xterm","ps"},
}
LOOKUP = {lbl: c for c, s in GROUPS.items() for lbl in s}

def to_5class(l):
    if l not in LOOKUP:
        raise ValueError(f"unmapped attack label: {l}")
    return LOOKUP[l]

df_tr = pd.read_csv(f"{RAW}/KDDTrain+.txt", header=None, names=COLS)
df_te = pd.read_csv(f"{RAW}/KDDTest+.txt",  header=None, names=COLS)
y_all = df_tr["label"].map(to_5class).values.astype("int32")
y_te  = df_te["label"].map(to_5class).values.astype("int32")

print("train counts:", np.bincount(y_all), "| test counts:", np.bincount(y_te))
assert list(np.bincount(y_all)) == [67343, 45927, 11656, 995, 52],  "train class counts differ from expected"
assert list(np.bincount(y_te))  == [9711, 7460, 2421, 2885, 67],    "test class counts differ from expected"

# %% [2] Features: labels/difficulty are NEVER inputs; official split; min-max fit on TRAIN only
CAT = ["protocol_type", "service", "flag"]
feat_tr = df_tr.drop(columns=["label", "difficulty"])
feat_te = df_te.drop(columns=["label", "difficulty"])
assert not any(c.lower().startswith("label") or c == "difficulty" for c in feat_tr.columns)

cats = {c: sorted(feat_tr[c].unique()) for c in CAT}          # vocabulary from KDDTrain+ only
print({c: len(v) for c, v in cats.items()})

def encode(df):
    d = df.copy()
    for c in CAT:
        d[c] = pd.Categorical(d[c], categories=cats[c])
    return pd.get_dummies(d, columns=CAT, dtype=np.float32)

X_all_df, X_te_df = encode(feat_tr), encode(feat_te)
assert list(X_all_df.columns) == list(X_te_df.columns)
print("input dim:", X_all_df.shape[1])
assert X_all_df.shape[1] == 122, "expected 38 numeric + 3 + 70 + 11 = 122"

unseen = int((feat_te[CAT].apply(lambda s: ~s.isin(cats[s.name]))).any(axis=1).sum())
print("test rows with a category unseen in train:", unseen)

idx_tr, idx_va = train_test_split(np.arange(len(y_all)), test_size=0.10,
                                  stratify=y_all, random_state=SPLIT_SEED)
Xtr_raw = X_all_df.values[idx_tr]; Xva_raw = X_all_df.values[idx_va]; Xte_raw = X_te_df.values
ytr, yva = y_all[idx_tr], y_all[idx_va]

scaler = MinMaxScaler().fit(Xtr_raw)
Xtr = scaler.transform(Xtr_raw).astype("float32")
Xva = np.clip(scaler.transform(Xva_raw), 0, 1).astype("float32")
te_scaled = scaler.transform(Xte_raw)
print("share of test values outside train range (clipped to [0,1]):",
      float(((te_scaled < 0) | (te_scaled > 1)).mean()))
Xte = np.clip(te_scaled, 0, 1).astype("float32")

def hashes(X): return {hashlib.md5(r.tobytes()).hexdigest() for r in np.round(X, 6)}
overlap = len(hashes(Xte) & (hashes(Xtr) | hashes(Xva)))
print("exact test rows also present in train/val (informational):", overlap, "of", len(Xte))

np.savez_compressed(f"{OUT}/nsl_v2_arrays.npz", Xtr=Xtr, ytr=ytr, Xva=Xva, yva=yva, Xte=Xte, yte=y_te)
joblib.dump(scaler, f"{OUT}/minmax_scaler.joblib")
json.dump(list(X_all_df.columns), open(f"{OUT}/feature_names.json", "w"), indent=1)
print("splits:", Xtr.shape, Xva.shape, Xte.shape)

# %% [3] Shared architecture and training recipe (every model in the paper uses these)
def build_model(d, k=5):
    inp = keras.Input(shape=(d,))
    x = layers.Dense(256)(inp); x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)
    x = layers.Dense(128)(x);   x = layers.Dropout(0.3)(x);          x = layers.Activation("relu")(x)
    x = layers.Dense(64)(x);    x = layers.Dropout(0.3)(x);          x = layers.Activation("relu")(x)
    return keras.Model(inp, layers.Dense(k, activation="softmax")(x))

def class_weights(y):
    w = compute_class_weight("balanced", classes=np.arange(5), y=y)
    return {i: float(min(v, CLASS_WEIGHT_CAP)) for i, v in enumerate(w)}

def train(model, X, y, Xv, yv, epochs=60):
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    model.fit(X, y, validation_data=(Xv, yv), epochs=epochs, batch_size=BATCH,
              class_weight=class_weights(y), verbose=2,
              callbacks=[keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True),
                         keras.callbacks.ReduceLROnPlateau(patience=4, factor=0.5)])
    return model

def evaluate(model, X, y):
    p = np.argmax(model.predict(X, batch_size=4096, verbose=0), axis=1)
    return {"acc": float(accuracy_score(y, p)),
            "macro_f1": float(f1_score(y, p, average="macro", labels=range(5), zero_division=0)),
            "per_class_f1": dict(zip(CLASS_NAMES, f1_score(y, p, average=None, labels=range(5), zero_division=0).tolist())),
            "confusion": confusion_matrix(y, p, labels=range(5)).tolist()}

# %% [4] Attacks with the box constraint [0,1] (sanity versions; full suite comes in Stage 2)
def _loss_and_grad(model, x, y):
    with tf.GradientTape() as t:
        t.watch(x)
        l = keras.losses.sparse_categorical_crossentropy(y, model(x, training=False))
        s = tf.reduce_sum(l)
    return l, t.gradient(s, x)

def fgsm(model, X, y, eps, bs=4096):
    out = []
    for i in range(0, len(X), bs):
        xb = tf.constant(X[i:i+bs]); yb = tf.constant(y[i:i+bs])
        _, g = _loss_and_grad(model, xb, yb)
        out.append(tf.clip_by_value(xb + eps * tf.sign(g), 0., 1.).numpy())
    return np.vstack(out)

def pgd(model, X, y, eps, steps=10, restarts=1, bs=4096):
    alpha = 2.5 * eps / steps
    out = []
    for i in range(0, len(X), bs):
        x0 = tf.constant(X[i:i+bs]); yb = tf.constant(y[i:i+bs])
        best, best_loss = x0, -np.inf * tf.ones(len(x0))
        for _ in range(restarts):
            x = tf.clip_by_value(x0 + tf.random.uniform(x0.shape, -eps, eps), 0., 1.)
            for _ in range(steps):
                _, g = _loss_and_grad(model, x, yb)
                x = tf.clip_by_value(tf.clip_by_value(x + alpha * tf.sign(g), x0 - eps, x0 + eps), 0., 1.)
            l = keras.losses.sparse_categorical_crossentropy(yb, model(x, training=False))
            better = l > best_loss
            best = tf.where(better[:, None], x, best); best_loss = tf.where(better, l, best_loss)
        out.append(best.numpy())
    return np.vstack(out)

# %% [5] Train the undefended baseline (test set is never touched during training)
set_seed(SEED)
model = build_model(Xtr.shape[1])
print("parameters (total):", model.count_params())
train(model, Xtr, ytr, Xva, yva)
model.save(f"{MOD}/undefended_seed{SEED}.keras")

clean = evaluate(model, Xte, y_te)
print("CLEAN test:", {k: clean[k] for k in ("acc", "macro_f1")}, clean["per_class_f1"])

# %% [6] Attack sanity checks (Carlini et al. red flags)
EPS = [0.0, 0.01, 0.02, 0.05, 0.10, 0.20, 0.50, 1.0]
rows = []
for e in EPS:
    a_f = accuracy_score(y_te, np.argmax(model.predict(fgsm(model, Xte, y_te, e), batch_size=4096, verbose=0), 1)) if e else clean["acc"]
    a_p = accuracy_score(y_te, np.argmax(model.predict(pgd(model, Xte, y_te, e, steps=10, restarts=1), batch_size=4096, verbose=0), 1)) if e else clean["acc"]
    rows.append({"eps": e, "fgsm_acc": float(a_f), "pgd10_acc": float(a_p)})
    print(f"eps={e:<5} FGSM acc={a_f:.4f}   PGD-10 acc={a_p:.4f}")

flags = []
for i in range(len(rows) - 1):
    for j in range(i + 1, len(rows)):
        if rows[j]["fgsm_acc"] > rows[i]["fgsm_acc"] + 0.01:
            flags.append(f"FGSM accuracy RISES from eps={rows[i]['eps']} to {rows[j]['eps']}")
            break
for r in rows:
    if r["pgd10_acc"] > r["fgsm_acc"] + 0.02:
        flags.append(f"PGD weaker than FGSM at eps={r['eps']}")
if rows[-1]["pgd10_acc"] > 0.05:
    flags.append("PGD at eps=1.0 does not drive accuracy near 0")
print("RED FLAGS:", flags if flags else "none")

json.dump(jsonable({"seed": SEED, "params": model.count_params(), "clean": clean,
                    "attack_sanity": rows, "flags": flags,
                    "test_rows_also_in_train": overlap, "unseen_category_rows": unseen}),
          open(f"{RES}/stage1_undefended_seed{SEED}.json", "w"), indent=1)
print("saved:", f"{RES}/stage1_undefended_seed{SEED}.json")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
train counts: [67343 45927 11656   995    52] | test counts: [9711 7460 2421 2885   67]
{'protocol_type': 3, 'service': 70, 'flag': 11}
input dim: 122
test rows with a category unseen in train: 0
share of test values outside train range (clipped to [0,1]): 2.545113962931506e-06
exact test rows also present in train/val (informational): 1975 of 22544
splits: (113375, 122) (12598, 122) (22544, 122)
parameters (total): 73989
Epoch 1/60
222/222 - 23s - 103ms/step - accuracy: 0.9262 - loss: 0.1720 - val_accuracy: 0.9613 - val_loss: 0.1672 - learning_rate: 0.0010
Epoch 2/60
222/222 - 1s - 3ms/step - accuracy: 0.9646 - loss: 0.0791 - val_accuracy: 0.9722 - val_loss: 0.0726 - learning_rate: 0.0010
Epoch 3/60
222/222 - 1s - 3ms/step - accuracy: 0.9685 - loss: 0.0627 - val_accuracy: 0.9729 - val_loss: 0.0707 - learning_rate: 0.0010
Epoch 4/60
222/222 - 1s - 3ms/step - 

In [ ]:
# Stage 1b - stronger attack sanity check (no retraining: loads the saved Stage-1 baseline from Drive)
# Fixes in the attack code: logits-based losses, best-iterate tracking, restarts, worst-case over attacks,
# optional mask that keeps one-hot columns fixed. Paste into Colab.

# %% [0] Load
from google.colab import drive
drive.mount('/content/drive')
import json, numpy as np, tensorflow as tf
from tensorflow import keras
from sklearn.metrics import accuracy_score

BASE = "/content/drive/MyDrive/A3IDPS_COLAB"
d = np.load(f"{BASE}/data_processed/nslkdd_v2/nsl_v2_arrays.npz")
Xte, yte = d["Xte"].astype("float32"), d["yte"].astype("int32")
names = json.load(open(f"{BASE}/data_processed/nslkdd_v2/feature_names.json"))
NUMERIC_MASK = np.array([0. if n.startswith(("protocol_type_", "service_", "flag_")) else 1. for n in names], "float32")
print("perturbable numeric features:", int(NUMERIC_MASK.sum()), "of", len(names))
model = keras.models.load_model(f"{BASE}/models/nsl_v2/undefended_seed0.keras")

# %% [1] Logits function (avoids softmax saturation / zero gradients) and losses
def make_logits_fn(m):
    W, b = [tf.constant(w) for w in m.layers[-1].get_weights()]
    body = keras.Model(m.input, m.layers[-2].output)
    return lambda x: tf.matmul(body(x, training=False), W) + b

logits_fn = make_logits_fn(model)
assert np.mean(np.argmax(logits_fn(tf.constant(Xte)).numpy(), 1) == np.argmax(model.predict(Xte, verbose=0), 1)) == 1.0

def ce_loss(z, y):
    return tf.nn.sparse_softmax_cross_entropy_with_logits(labels=y, logits=z)

def margin_loss(z, y):                     # >0 means misclassified
    oh = tf.one_hot(y, z.shape[-1])
    return tf.reduce_max(z - 1e9 * oh, 1) - tf.reduce_sum(z * oh, 1)

# %% [2] Attacks (all respect the [0,1] box and the per-feature budget eps * mask)
def fgsm(X, y, eps, mask, bs=4096):
    out = []
    for i in range(0, len(X), bs):
        x0 = tf.constant(X[i:i+bs]); yb = tf.constant(y[i:i+bs])
        with tf.GradientTape() as t:
            t.watch(x0); s = tf.reduce_sum(ce_loss(logits_fn(x0), yb))
        g = t.gradient(s, x0)
        out.append(tf.clip_by_value(x0 + eps * mask * tf.sign(g), 0., 1.).numpy())
    return np.vstack(out)

def pgd(X, y, eps, mask, loss_fn, steps=20, restarts=3, bs=4096):
    E = tf.constant(eps * mask)
    out = []
    for i in range(0, len(X), bs):
        x0 = tf.constant(X[i:i+bs]); yb = tf.constant(y[i:i+bs])
        best_x, best_l = x0, tf.fill([x0.shape[0]], -1e30)
        for r in range(restarts):
            x = x0 if r == 0 else tf.clip_by_value(x0 + tf.random.uniform(x0.shape, -1., 1.) * E, 0., 1.)
            for t_ in range(steps):
                alpha = (1.0 if t_ < steps // 2 else 0.25) * E / 4.0
                with tf.GradientTape() as t:
                    t.watch(x); s = tf.reduce_sum(loss_fn(logits_fn(x), yb))
                g = t.gradient(s, x)
                x = tf.clip_by_value(tf.clip_by_value(x + alpha * tf.sign(g), x0 - E, x0 + E), 0., 1.)
                l_new = loss_fn(logits_fn(x), yb)
                upd = l_new > best_l
                best_x = tf.where(upd[:, None], x, best_x); best_l = tf.where(upd, l_new, best_l)
        out.append(best_x.numpy())
    return np.vstack(out)

def correct_mask(Xa, y):
    return np.argmax(logits_fn(tf.constant(Xa)).numpy(), 1) == y

# %% [3] Sanity table, with and without the categorical mask
EPS = [0.0, 0.01, 0.02, 0.05, 0.10, 0.20, 0.50, 1.0]
results = {}
for use_mask in (False, True):
    mask = NUMERIC_MASK if use_mask else np.ones_like(NUMERIC_MASK)
    tag = "numeric_only" if use_mask else "all_features"
    rows, flags = [], []
    print(f"\n=== perturbation on: {tag} ===")
    for e in EPS:
        if e == 0:
            a = float(accuracy_score(yte, np.argmax(logits_fn(tf.constant(Xte)).numpy(), 1)))
            rows.append({"eps": e, "fgsm": a, "pgd_ce": a, "pgd_margin": a, "worst_case": a}); continue
        c_f = correct_mask(fgsm(Xte, yte, e, mask), yte)
        c_c = correct_mask(pgd(Xte, yte, e, mask, ce_loss), yte)
        c_m = correct_mask(pgd(Xte, yte, e, mask, margin_loss), yte)
        worst = c_f & c_c & c_m
        rows.append({"eps": e, "fgsm": float(c_f.mean()), "pgd_ce": float(c_c.mean()),
                     "pgd_margin": float(c_m.mean()), "worst_case": float(worst.mean())})
        r = rows[-1]
        print(f"eps={e:<5} FGSM={r['fgsm']:.4f}  PGD-CE={r['pgd_ce']:.4f}  PGD-margin={r['pgd_margin']:.4f}  worst-case={r['worst_case']:.4f}")
        if r["pgd_ce"] > r["fgsm"] + 0.02:
            flags.append(f"PGD-CE weaker than FGSM at eps={e}")
    for i in range(len(rows) - 1):
        if rows[i + 1]["worst_case"] > rows[i]["worst_case"] + 0.01:
            flags.append(f"worst-case accuracy rises from eps={rows[i]['eps']} to {rows[i+1]['eps']}")
    if use_mask is False and rows[-1]["worst_case"] > 0.05:
        flags.append("worst-case at eps=1.0 (all features) is not near 0")
    print("RED FLAGS:", flags if flags else "none")
    results[tag] = {"rows": rows, "flags": flags}

json.dump(results, open(f"{BASE}/results/nsl_v2/stage1b_attack_sanity_seed0.json", "w"), indent=1)
print("saved stage1b_attack_sanity_seed0.json")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
perturbable numeric features: 38 of 122

=== perturbation on: all_features ===
eps=0.01  FGSM=0.5940  PGD-CE=0.5863  PGD-margin=0.5864  worst-case=0.5862
eps=0.02  FGSM=0.5214  PGD-CE=0.4848  PGD-margin=0.4853  worst-case=0.4847
eps=0.05  FGSM=0.2334  PGD-CE=0.0857  PGD-margin=0.0735  worst-case=0.0684
eps=0.1   FGSM=0.0495  PGD-CE=0.0000  PGD-margin=0.0000  worst-case=0.0000
eps=0.2   FGSM=0.0004  PGD-CE=0.0000  PGD-margin=0.0000  worst-case=0.0000
eps=0.5   FGSM=0.0000  PGD-CE=0.0000  PGD-margin=0.0000  worst-case=0.0000
eps=1.0   FGSM=0.0001  PGD-CE=0.0000  PGD-margin=0.0000  worst-case=0.0000
RED FLAGS: none

=== perturbation on: numeric_only ===
eps=0.01  FGSM=0.7025  PGD-CE=0.6977  PGD-margin=0.6977  worst-case=0.6974
eps=0.02  FGSM=0.5962  PGD-CE=0.5867  PGD-margin=0.5868  worst-case=0.5867
eps=0.05  FGSM=0.4709  PGD-CE=0.3406  PGD-margin=0.3421  worst

In [ ]:
# Stage 2 - NSL-KDD: shared logits architecture, undefended / FGSM-AT / PGD-AT with an epsilon_train sweep,
# validation-based selection (rule fixed in advance), and full test evaluation with worst-case attacks.
# Paste into Colab cell by cell. Resumable: finished models are loaded from Drive, not retrained.
# Protocol (frozen):
#   - main protocol perturbs the 38 numeric features only; all 122 features is a stress test
#   - epsilon_train in {0.02, 0.05, 0.10}, chosen per method on VALIDATION data only:
#       highest mean validation worst-case accuracy over eps in {0.01, 0.02, 0.05, 0.10},
#       among candidates whose clean validation accuracy is within 5 pp of the undefended model (ties -> smaller eps)
#   - evaluation eps {0.01, 0.02, 0.05, 0.10, 0.15}; 0.15 lies outside the training grid

# %% [0] Setup
from google.colab import drive
drive.mount('/content/drive')
import os, json, random, time
import numpy as np, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight

BASE = "/content/drive/MyDrive/A3IDPS_COLAB"
DATA = f"{BASE}/data_processed/nslkdd_v2"
MOD  = f"{BASE}/models/nsl_v2"
RES  = f"{BASE}/results/nsl_v2"
os.makedirs(MOD, exist_ok=True); os.makedirs(RES, exist_ok=True)

d = np.load(f"{DATA}/nsl_v2_arrays.npz")
Xtr, Xva, Xte = [d[k].astype("float32") for k in ("Xtr", "Xva", "Xte")]
ytr, yva, yte = [d[k].astype("int32") for k in ("ytr", "yva", "yte")]
names = json.load(open(f"{DATA}/feature_names.json"))
D, NCLS = Xtr.shape[1], 5
CLASS_NAMES = ["Normal", "DoS", "Probe", "R2L", "U2R"]
MASK_NUM = np.array([0. if n.startswith(("protocol_type_", "service_", "flag_")) else 1. for n in names], "float32")
MASK_ALL = np.ones(D, "float32")
assert int(MASK_NUM.sum()) == 38

SEED = 0
EPS_EVAL   = [0.01, 0.02, 0.05, 0.10, 0.15]
EPS_SELECT = EPS_EVAL[:4]
EPS_TRAIN  = [0.02, 0.05, 0.10]
CLASS_WEIGHT_CAP, VAL_CLEAN_TOL, BATCH, MAX_EPOCHS = 20.0, 0.05, 512, 40
cw = np.minimum(compute_class_weight("balanced", classes=np.arange(NCLS), y=ytr), CLASS_WEIGHT_CAP).astype("float32")

def set_seed(s):
    random.seed(s); np.random.seed(s); tf.keras.utils.set_random_seed(s)

def jsonable(o):
    if isinstance(o, dict): return {k: jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [jsonable(v) for v in o]
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return float(o)
    return o

# %% [1] Model (outputs LOGITS; softmax is folded into the loss) and attacks
def build_model(d):
    inp = keras.Input(shape=(d,))
    x = layers.Dense(256)(inp); x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)
    x = layers.Dense(128)(x);   x = layers.Dropout(0.3)(x);          x = layers.Activation("relu")(x)
    x = layers.Dense(64)(x);    x = layers.Dropout(0.3)(x);          x = layers.Activation("relu")(x)
    return keras.Model(inp, layers.Dense(NCLS)(x))

def make_attacks(model, mask):
    M = tf.constant(mask, tf.float32)

    def loss_fn(z, y, margin):
        if margin:
            oh = tf.one_hot(y, NCLS)
            return tf.reduce_max(z - 1e9 * oh, axis=1) - tf.reduce_sum(z * oh, axis=1)
        return tf.nn.sparse_softmax_cross_entropy_with_logits(labels=y, logits=z)

    @tf.function
    def fgsm(x0, y, eps):
        with tf.GradientTape() as t:
            t.watch(x0)
            s = tf.reduce_sum(loss_fn(model(x0, training=False), y, False))
        g = t.gradient(s, x0)
        return tf.clip_by_value(x0 + eps * M * tf.sign(g), 0., 1.)

    @tf.function
    def pgd(x0, y, eps, steps, restarts, margin):
        E = eps * M
        best_x = x0
        best_l = tf.fill(tf.shape(y), -1e30)
        for r in range(restarts):
            if r == 0:
                x = x0
            else:
                x = tf.clip_by_value(x0 + tf.random.uniform(tf.shape(x0), -1., 1.) * E, 0., 1.)
            for s_ in range(steps):
                alpha = (1.0 if s_ < steps // 2 else 0.25) * E / 4.0
                with tf.GradientTape() as t:
                    t.watch(x)
                    s = tf.reduce_sum(loss_fn(model(x, training=False), y, margin))
                g = t.gradient(s, x)
                x = tf.clip_by_value(tf.clip_by_value(x + alpha * tf.sign(g), x0 - E, x0 + E), 0., 1.)
                l_new = loss_fn(model(x, training=False), y, margin)
                upd = l_new > best_l
                best_x = tf.where(upd[:, None], x, best_x)
                best_l = tf.where(upd, l_new, best_l)
        return best_x
    return {"fgsm": fgsm, "pgd": pgd}

def run_attack(fn, X, y, bs=4096, **kw):
    return np.vstack([fn(tf.constant(X[i:i+bs]), tf.constant(y[i:i+bs]), **kw).numpy()
                      for i in range(0, len(X), bs)])

def predict(model, X):
    return np.argmax(model.predict(X, batch_size=4096, verbose=0), axis=1)

def clean_eval(model, X, y):
    p = predict(model, X)
    return {"acc": float(accuracy_score(y, p)),
            "macro_f1": float(f1_score(y, p, average="macro", labels=range(NCLS), zero_division=0)),
            "per_class_f1": dict(zip(CLASS_NAMES, f1_score(y, p, average=None, labels=range(NCLS), zero_division=0).tolist()))}

def robust_eval(model, X, y, mask, eps_list, steps, restarts):
    A = make_attacks(model, mask); out = {}
    for e in eps_list:
        et = tf.constant(e, tf.float32)
        c_f = predict(model, run_attack(A["fgsm"], X, y, eps=et)) == y
        c_c = predict(model, run_attack(A["pgd"], X, y, eps=et, steps=steps, restarts=restarts, margin=False)) == y
        c_m = predict(model, run_attack(A["pgd"], X, y, eps=et, steps=steps, restarts=restarts, margin=True)) == y
        w = c_f & c_c & c_m
        out[str(e)] = {"fgsm": float(c_f.mean()), "pgd_ce": float(c_c.mean()), "pgd_margin": float(c_m.mean()),
                       "worst": float(w.mean()),
                       "worst_recall": {CLASS_NAMES[k]: float(w[y == k].mean()) for k in range(NCLS)}}
    return out

# %% [2] Training: identical recipe for every model (undefended, FGSM-AT, PGD-AT)
def train_model(kind, eps_train, seed, mask=MASK_NUM):
    tag = kind if kind == "none" else f"{kind}_eps{eps_train}"
    path = f"{MOD}/s2_{tag}_seed{seed}.keras"
    if os.path.exists(path):
        print("loading", path); return keras.models.load_model(path, compile=False), tag
    set_seed(seed)
    model = build_model(D)
    opt = keras.optimizers.Adam(1e-3)
    A = make_attacks(model, mask) if kind != "none" else None
    cwt = tf.constant(cw)
    wce = lambda z, y: tf.reduce_mean(tf.nn.sparse_softmax_cross_entropy_with_logits(labels=y, logits=z) * tf.gather(cwt, y))
    et = tf.constant(eps_train or 0., tf.float32)
    best, best_w, wait, lr = 1e9, None, 0, 1e-3
    rng = np.random.RandomState(seed)
    for ep in range(MAX_EPOCHS):
        t0 = time.time(); perm = rng.permutation(len(Xtr))
        for i in range(0, len(perm), BATCH):
            idx = perm[i:i+BATCH]
            xb, yb = tf.constant(Xtr[idx]), tf.constant(ytr[idx])
            if kind == "fgsm":
                xa = A["fgsm"](xb, yb, eps=et)
            elif kind == "pgd":
                xa = A["pgd"](xb, yb, eps=et, steps=10, restarts=1, margin=False)
            if kind != "none":                      # clean + adversarial version of the same batch
                xb, yb = tf.concat([xb, xa], 0), tf.concat([yb, yb], 0)
            with tf.GradientTape() as t:
                loss = wce(model(xb, training=True), yb)
            opt.apply_gradients(zip(t.gradient(loss, model.trainable_variables), model.trainable_variables))
        vl = float(wce(model(tf.constant(Xva), training=False), tf.constant(yva)))
        if kind != "none":
            xv = run_attack(A["fgsm"], Xva, yva, eps=et)
            vl += float(wce(model(tf.constant(xv), training=False), tf.constant(yva)))
        if vl < best - 1e-4:
            best, best_w, wait = vl, model.get_weights(), 0
        else:
            wait += 1
            if wait % 4 == 0:
                lr *= 0.5; opt.learning_rate = lr
            if wait >= 8:
                break
        print(f"[{tag}] epoch {ep+1} val_loss={vl:.4f} best={best:.4f} ({time.time()-t0:.1f}s)")
    model.set_weights(best_w); model.save(path)
    return model, tag

# %% [3] Train everything (seed 0) and score on VALIDATION data
SPEC = [("none", None)] + [("fgsm", e) for e in EPS_TRAIN] + [("pgd", e) for e in EPS_TRAIN]
models, val_res = {}, {}
for kind, e in SPEC:
    model, tag = train_model(kind, e, SEED)
    models[tag] = model
    vc = clean_eval(model, Xva, yva)
    vr = robust_eval(model, Xva, yva, MASK_NUM, EPS_SELECT, steps=10, restarts=1)
    val_res[tag] = {"val_clean": vc, "val_rob": vr,
                    "val_rob_mean": float(np.mean([vr[str(x)]["worst"] for x in EPS_SELECT]))}
    print(f"VAL {tag:<12} clean acc={vc['acc']:.4f}  mean worst-case robust acc={val_res[tag]['val_rob_mean']:.4f}")
    json.dump(jsonable(val_res), open(f"{RES}/stage2_val_seed{SEED}.json", "w"), indent=1)

# %% [4] Select epsilon_train per method with the pre-stated rule (validation data only)
und = val_res["none"]["val_clean"]["acc"]
selected = {}
for method in ("fgsm", "pgd"):
    cands = [(e, val_res[f"{method}_eps{e}"]) for e in EPS_TRAIN]
    elig = [(e, v) for e, v in cands if und - v["val_clean"]["acc"] <= VAL_CLEAN_TOL]
    pool = elig if elig else cands
    if not elig: print(f"WARNING: no {method} candidate within {VAL_CLEAN_TOL:.0%} clean-accuracy tolerance")
    e_best = max(pool, key=lambda ev: (round(ev[1]["val_rob_mean"], 6), -ev[0]))[0]
    selected[method] = e_best
    for e, v in cands:
        print(f"{method:<5} eps_train={e:<5} val clean={v['val_clean']['acc']:.4f} mean robust={v['val_rob_mean']:.4f} "
              f"eligible={(e, v) in elig} {'<== selected' if e == e_best else ''}")
json.dump(selected, open(f"{RES}/stage2_selected_seed{SEED}.json", "w"))

# %% [5] TEST evaluation (worst-case over FGSM, PGD-CE, PGD-margin; 20 steps x 3 restarts)
test_res = {}
for tag, model in models.items():
    test_res[tag] = {"clean": clean_eval(model, Xte, yte),
                     "numeric_only": robust_eval(model, Xte, yte, MASK_NUM, EPS_EVAL, steps=20, restarts=3),
                     "all_features": robust_eval(model, Xte, yte, MASK_ALL, EPS_EVAL, steps=20, restarts=3)}
    json.dump(jsonable({"selected": selected, "test": test_res}), open(f"{RES}/stage2_test_seed{SEED}.json", "w"), indent=1)

show = ["none", f"fgsm_eps{selected['fgsm']}", f"pgd_eps{selected['pgd']}"]
for prot in ("numeric_only", "all_features"):
    print(f"\n=== TEST, worst-case robust accuracy, {prot} (selected models) ===")
    print(f"{'model':<14}{'clean':>8}{'macroF1':>9}" + "".join(f"{('e='+str(e)):>9}" for e in EPS_EVAL))
    for tag in show:
        r, c = test_res[tag][prot], test_res[tag]["clean"]
        print(f"{tag:<14}{c['acc']:>8.4f}{c['macro_f1']:>9.4f}" + "".join(f"{r[str(e)]['worst']:>9.4f}" for e in EPS_EVAL))
print("\nAppendix grid (all eps_train) is in", f"{RES}/stage2_test_seed{SEED}.json")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[none] epoch 1 val_loss=0.1471 best=0.1471 (18.0s)
[none] epoch 2 val_loss=0.0538 best=0.0538 (16.8s)
[none] epoch 3 val_loss=0.0451 best=0.0451 (17.0s)
[none] epoch 4 val_loss=0.0456 best=0.0451 (16.7s)
[none] epoch 5 val_loss=0.0485 best=0.0451 (16.6s)
[none] epoch 6 val_loss=0.0411 best=0.0411 (16.6s)
[none] epoch 7 val_loss=0.0417 best=0.0411 (16.8s)
[none] epoch 8 val_loss=0.0438 best=0.0411 (16.6s)
[none] epoch 9 val_loss=0.0361 best=0.0361 (16.6s)
[none] epoch 10 val_loss=0.0321 best=0.0321 (16.7s)
[none] epoch 11 val_loss=0.0433 best=0.0321 (16.8s)
[none] epoch 12 val_loss=0.0439 best=0.0321 (16.7s)
[none] epoch 13 val_loss=0.0493 best=0.0321 (16.8s)
[none] epoch 14 val_loss=0.0429 best=0.0321 (16.6s)
[none] epoch 15 val_loss=0.0500 best=0.0321 (16.7s)
[none] epoch 16 val_loss=0.0499 best=0.0321 (16.6s)
[none] epoch 17 val_loss=0.0463 best=0.0321 (16.

VAL fgsm_eps0.02 clean acc=0.9682  mean worst-case robust acc=0.9164
[fgsm_eps0.05] epoch 1 val_loss=0.3392 best=0.3392 (18.0s)
[fgsm_eps0.05] epoch 2 val_loss=0.1388 best=0.1388 (17.8s)
[fgsm_eps0.05] epoch 3 val_loss=0.1177 best=0.1177 (17.5s)
[fgsm_eps0.05] epoch 4 val_loss=0.1233 best=0.1177 (17.4s)
[fgsm_eps0.05] epoch 5 val_loss=0.1192 best=0.1177 (17.5s)
[fgsm_eps0.05] epoch 6 val_loss=0.1318 best=0.1177 (17.5s)
[fgsm_eps0.05] epoch 7 val_loss=0.1219 best=0.1177 (17.5s)
[fgsm_eps0.05] epoch 8 val_loss=0.1195 best=0.1177 (17.4s)
[fgsm_eps0.05] epoch 9 val_loss=0.1149 best=0.1149 (17.3s)
[fgsm_eps0.05] epoch 10 val_loss=0.1200 best=0.1149 (17.2s)
[fgsm_eps0.05] epoch 11 val_loss=0.1197 best=0.1149 (17.1s)
[fgsm_eps0.05] epoch 12 val_loss=0.1169 best=0.1149 (17.0s)
[fgsm_eps0.05] epoch 13 val_loss=0.1162 best=0.1149 (17.0s)
[fgsm_eps0.05] epoch 14 val_loss=0.1221 best=0.1149 (17.1s)
[fgsm_eps0.05] epoch 15 val_loss=0.1203 best=0.1149 (16.9s)
[fgsm_eps0.05] epoch 16 val_loss=0.1198 

VAL fgsm_eps0.05 clean acc=0.9698  mean worst-case robust acc=0.9608
[fgsm_eps0.1] epoch 1 val_loss=0.3706 best=0.3706 (17.8s)
[fgsm_eps0.1] epoch 2 val_loss=0.1676 best=0.1676 (17.4s)
[fgsm_eps0.1] epoch 3 val_loss=0.1460 best=0.1460 (17.4s)
[fgsm_eps0.1] epoch 4 val_loss=0.1415 best=0.1415 (17.4s)
[fgsm_eps0.1] epoch 5 val_loss=0.1353 best=0.1353 (17.3s)
[fgsm_eps0.1] epoch 6 val_loss=0.1437 best=0.1353 (17.3s)
[fgsm_eps0.1] epoch 7 val_loss=0.1293 best=0.1293 (17.4s)
[fgsm_eps0.1] epoch 8 val_loss=0.1394 best=0.1293 (17.1s)
[fgsm_eps0.1] epoch 9 val_loss=0.1180 best=0.1180 (17.2s)
[fgsm_eps0.1] epoch 10 val_loss=0.1259 best=0.1180 (17.2s)
[fgsm_eps0.1] epoch 11 val_loss=0.1258 best=0.1180 (17.2s)
[fgsm_eps0.1] epoch 12 val_loss=0.1146 best=0.1146 (17.2s)
[fgsm_eps0.1] epoch 13 val_loss=0.1219 best=0.1146 (17.2s)
[fgsm_eps0.1] epoch 14 val_loss=0.1214 best=0.1146 (17.2s)
[fgsm_eps0.1] epoch 15 val_loss=0.1327 best=0.1146 (17.0s)
[fgsm_eps0.1] epoch 16 val_loss=0.1127 best=0.1127 (17.

In [ ]:
# Stage 2b - ART AutoAttack smoke test. Run AFTER nsl_stage2.py has trained the undefended model.
# Purpose: does ART's AutoAttack run cleanly here, and is it at least as strong as our PGD worst-case?
# If anything below errors, paste the traceback: the fallback is our own APGD-CE + APGD-DLR in TensorFlow.

# %% [0] Install and load
import subprocess
subprocess.run(["pip", "install", "-q", "adversarial-robustness-toolbox"], check=True)

from google.colab import drive
drive.mount('/content/drive')
import json, time, numpy as np, tensorflow as tf
from tensorflow import keras

BASE = "/content/drive/MyDrive/A3IDPS_COLAB"
d = np.load(f"{BASE}/data_processed/nslkdd_v2/nsl_v2_arrays.npz")
Xte, yte = d["Xte"].astype("float32"), d["yte"].astype("int32")
names = json.load(open(f"{BASE}/data_processed/nslkdd_v2/feature_names.json"))
MASK_NUM = np.array([0. if n.startswith(("protocol_type_", "service_", "flag_")) else 1. for n in names], "float32")
model = keras.models.load_model(f"{BASE}/models/nsl_v2/s2_none_seed0.keras", compile=False)   # outputs logits
D = Xte.shape[1]

rng = np.random.RandomState(0)
sub = rng.choice(len(Xte), 1000, replace=False)
X, y = Xte[sub], yte[sub]

# %% [1] Wrap the model for ART
import art
from art.estimators.classification import TensorFlowV2Classifier
from art.attacks.evasion import AutoAttack
print("ART version:", art.__version__)

loss_obj = tf.keras.losses.CategoricalCrossentropy(from_logits=True)
clf = TensorFlowV2Classifier(model=model, nb_classes=5, input_shape=(D,), loss_object=loss_obj,
                             clip_values=(0.0, 1.0))
p_art = np.argmax(clf.predict(X), 1)
p_our = np.argmax(model.predict(X, verbose=0), 1)
print("ART vs our predictions agree:", bool((p_art == p_our).all()), "| clean acc on subset:", float((p_our == y).mean()))
Y1 = keras.utils.to_categorical(y, 5)

# %% [2] AutoAttack, all features (ART does not know about our mask)
ref = json.load(open(f"{BASE}/results/nsl_v2/stage1b_attack_sanity_seed0.json"))["all_features"]["rows"]
for eps in (0.02, 0.05):
    t0 = time.time()
    try:
        aa = AutoAttack(estimator=clf, norm=np.inf, eps=eps, eps_step=eps / 4, batch_size=256)
        Xa = aa.generate(x=X, y=Y1)
        acc = float((np.argmax(clf.predict(Xa), 1) == y).mean())
        linf = float(np.abs(Xa - X).max())
        ok_box = bool((Xa >= 0).all() and (Xa <= 1).all())
        ref_w = [r["worst_case"] for r in ref if abs(r["eps"] - eps) < 1e-9][0]
        print(f"eps={eps}: AutoAttack acc={acc:.4f} | max|delta|={linf:.4f} (<= eps: {linf <= eps + 1e-6}) | in [0,1]: {ok_box} "
              f"| our Stage-1b worst-case on full test (undefended v1 model)={ref_w:.4f} | {time.time()-t0:.0f}s")
    except Exception as ex:
        print(f"eps={eps}: AutoAttack FAILED: {type(ex).__name__}: {ex}")

# %% [3] Does AutoAttack accept a perturbation mask (numeric features only)?
try:
    aa = AutoAttack(estimator=clf, norm=np.inf, eps=0.05, eps_step=0.0125, batch_size=256)
    Xa = aa.generate(x=X[:200], y=Y1[:200], mask=np.tile(MASK_NUM, (200, 1)))
    frozen = float(np.abs((Xa - X[:200])[:, MASK_NUM == 0]).max())
    print("mask accepted; max change on frozen (one-hot) columns:", frozen)
except Exception as ex:
    print("mask NOT supported by AutoAttack.generate:", type(ex).__name__, ex)
print("Done.")


Mounted at /content/drive
ART version: 1.20.1
ART vs our predictions agree: True | clean acc on subset: 0.809


AutoPGD - restart:   0%|          | 0/5 [00:00<?, ?it/s]

AutoPGD - batch:   0%|          | 0/4 [00:00<?, ?it/s]

AutoPGD - iteration:   0%|          | 0/100 [00:00<?, ?it/s]

AutoPGD - iteration:   0%|          | 0/100 [00:00<?, ?it/s]

...

AutoPGD - batch:   0%|          | 0/1 [00:00<?, ?it/s]

AutoPGD - iteration:   0%|          | 0/100 [00:00<?, ?it/s]

AutoPGD - batch:   0%|          | 0/1 [00:00<?, ?it/s]

AutoPGD - iteration:   0%|          | 0/100 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# Stage 2c - ART APGD-only cross-check (replaces the default AutoAttack suite, which fails on tabular data).
# Runs ART's APGD-CE and APGD-DLR on a 300-row test subset and compares them with our own worst-case PGD attacks
# on the SAME rows. Also tests whether ART's APGD accepts a per-feature mask. Run after restarting the runtime.

# %% [0] Install and load
import subprocess
subprocess.run(["pip", "install", "-q", "adversarial-robustness-toolbox"], check=True)

from google.colab import drive
drive.mount('/content/drive')
import json, time, numpy as np, tensorflow as tf
from tensorflow import keras

BASE = "/content/drive/MyDrive/A3IDPS_COLAB"
d = np.load(f"{BASE}/data_processed/nslkdd_v2/nsl_v2_arrays.npz")
Xte, yte = d["Xte"].astype("float32"), d["yte"].astype("int32")
names = json.load(open(f"{BASE}/data_processed/nslkdd_v2/feature_names.json"))
MASK_NUM = np.array([0. if n.startswith(("protocol_type_", "service_", "flag_")) else 1. for n in names], "float32")
MASK_ALL = np.ones_like(MASK_NUM)
model = keras.models.load_model(f"{BASE}/models/nsl_v2/s2_none_seed0.keras", compile=False)   # logits output
D, NCLS = Xte.shape[1], 5

N = 300
sub = np.random.RandomState(0).choice(len(Xte), N, replace=False)
X, y = Xte[sub], yte[sub]
Y1 = keras.utils.to_categorical(y, NCLS)
print("clean acc on subset:", float((np.argmax(model.predict(X, verbose=0), 1) == y).mean()))

# %% [1] Our own attacks (same logic as Stage 2) on the same subset
def loss_fn(z, yy, margin):
    if margin:
        oh = tf.one_hot(yy, NCLS)
        return tf.reduce_max(z - 1e9 * oh, axis=1) - tf.reduce_sum(z * oh, axis=1)
    return tf.nn.sparse_softmax_cross_entropy_with_logits(labels=yy, logits=z)

def our_pgd(X, y, eps, mask, margin, steps=20, restarts=3):
    E = tf.constant(eps * mask); x0 = tf.constant(X); yb = tf.constant(y)
    best_x, best_l = x0, tf.fill(tf.shape(yb), -1e30)
    for r in range(restarts):
        x = x0 if r == 0 else tf.clip_by_value(x0 + tf.random.uniform(tf.shape(x0), -1., 1.) * E, 0., 1.)
        for s_ in range(steps):
            alpha = (1.0 if s_ < steps // 2 else 0.25) * E / 4.0
            with tf.GradientTape() as t:
                t.watch(x); s = tf.reduce_sum(loss_fn(model(x, training=False), yb, margin))
            x = tf.clip_by_value(tf.clip_by_value(x + alpha * tf.sign(t.gradient(s, x)), x0 - E, x0 + E), 0., 1.)
            l_new = loss_fn(model(x, training=False), yb, margin)
            upd = l_new > best_l
            best_x = tf.where(upd[:, None], x, best_x); best_l = tf.where(upd, l_new, best_l)
    return best_x.numpy()

def acc_correct(Xa):
    return np.argmax(model.predict(Xa, verbose=0), 1) == y

# %% [2] ART APGD-CE and APGD-DLR (all features), compared with our worst-case
import art
from art.estimators.classification import TensorFlowV2Classifier
from art.attacks.evasion import AutoProjectedGradientDescent
print("ART version:", art.__version__)

clf = TensorFlowV2Classifier(model=model, nb_classes=NCLS, input_shape=(D,),
                             loss_object=tf.keras.losses.CategoricalCrossentropy(from_logits=True), clip_values=(0.0, 1.0))
results = {}
for eps in (0.02, 0.05):
    row = {}
    ours_ce = acc_correct(our_pgd(X, y, eps, MASK_ALL, False)); ours_m = acc_correct(our_pgd(X, y, eps, MASK_ALL, True))
    row["ours_pgd_ce"], row["ours_pgd_margin"], row["ours_worst"] = float(ours_ce.mean()), float(ours_m.mean()), float((ours_ce & ours_m).mean())
    art_correct = {}
    for loss_type in ("cross_entropy", "difference_logits_ratio"):
        t0 = time.time()
        try:
            atk = AutoProjectedGradientDescent(estimator=clf, norm=np.inf, eps=eps, eps_step=eps / 4, max_iter=100,
                                               targeted=False, nb_random_init=2, batch_size=150, loss_type=loss_type, verbose=False)
            Xa = atk.generate(x=X, y=Y1)
            c = acc_correct(Xa); art_correct[loss_type] = c
            linf = float(np.abs(Xa - X).max())
            row[f"art_apgd_{loss_type}"] = float(c.mean())
            print(f"eps={eps} ART APGD-{loss_type}: acc={c.mean():.4f} | max|delta|={linf:.4f} (<=eps: {linf <= eps + 1e-6}) | {time.time()-t0:.0f}s")
        except Exception as ex:
            print(f"eps={eps} ART APGD-{loss_type} FAILED: {type(ex).__name__}: {ex}")
    if len(art_correct) == 2:
        row["art_worst"] = float((art_correct["cross_entropy"] & art_correct["difference_logits_ratio"]).mean())
    results[str(eps)] = row
    print(f"eps={eps} SUMMARY:", {k: round(v, 4) for k, v in row.items()})

# %% [3] Does ART's APGD accept a perturbation mask (numeric features only)?
try:
    atk = AutoProjectedGradientDescent(estimator=clf, norm=np.inf, eps=0.05, eps_step=0.0125, max_iter=50, targeted=False,
                                       nb_random_init=1, batch_size=100, loss_type="cross_entropy", verbose=False)
    n_ = 100
    Xa = atk.generate(x=X[:n_], y=Y1[:n_], mask=np.tile(MASK_NUM, (n_, 1)))
    print("mask accepted; max change on frozen (one-hot) columns:", float(np.abs((Xa - X[:n_])[:, MASK_NUM == 0]).max()))
except Exception as ex:
    print("mask NOT accepted by APGD.generate:", type(ex).__name__, ex)

json.dump(results, open(f"{BASE}/results/nsl_v2/stage2c_art_apgd_check.json", "w"), indent=1)
print("saved stage2c_art_apgd_check.json")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
clean acc on subset: 0.8
ART version: 1.20.1


eps=0.02 ART APGD-cross_entropy: acc=0.5467 | max|delta|=0.0200 (<=eps: True) | 13s


eps=0.02 ART APGD-difference_logits_ratio: acc=0.6500 | max|delta|=0.0200 (<=eps: True) | 146s
eps=0.02 SUMMARY: {'ours_pgd_ce': 0.5467, 'ours_pgd_margin': 0.55, 'ours_worst': 0.5467, 'art_apgd_cross_entropy': 0.5467, 'art_apgd_difference_logits_ratio': 0.65, 'art_worst': 0.5467}
eps=0.05 ART APGD-cross_entropy: acc=0.1633 | max|delta|=0.0500 (<=eps: True) | 10s
eps=0.05 ART APGD-difference_logits_ratio: acc=0.2333 | max|delta|=0.0500 (<=eps: True) | 105s
eps=0.05 SUMMARY: {'ours_pgd_ce': 0.1867, 'ours_pgd_margin': 0.07, 'ours_worst': 0.07, 'art_apgd_cross_entropy': 0.1633, 'art_apgd_difference_logits_ratio': 0.2333, 'art_worst': 0.1633}
mask NOT accepted by APGD.generate: ValueError operands could not be broadcast together with shapes (79,122) (100,122) 
saved stage2c_art_apgd_check.json


In [ ]:
# ART APGD cross-check (Option A): independent published implementation vs our worst-case attacks,
# on a FIXED 300-row test subset, for any list of models. Resumable (results cached in Drive).
# Reused later for the A3IDPS models: just add them to MODELS below and re-run.
# In the paper this is described as "APGD-CE and APGD-DLR (components of AutoAttack)", not "AutoAttack".

# %% [0] Install, load, configuration
import subprocess
subprocess.run(["pip", "install", "-q", "adversarial-robustness-toolbox"], check=True)

from google.colab import drive
drive.mount('/content/drive')
import json, os, time, numpy as np, tensorflow as tf
from tensorflow import keras

BASE = "/content/drive/MyDrive/A3IDPS_COLAB"
RES = f"{BASE}/results/nsl_v2"; MOD = f"{BASE}/models/nsl_v2"
d = np.load(f"{BASE}/data_processed/nslkdd_v2/nsl_v2_arrays.npz")
Xte, yte = d["Xte"].astype("float32"), d["yte"].astype("int32")
names = json.load(open(f"{BASE}/data_processed/nslkdd_v2/feature_names.json"))
MASK_NUM = np.array([0. if n.startswith(("protocol_type_", "service_", "flag_")) else 1. for n in names], "float32")
MASK_ALL = np.ones_like(MASK_NUM)
D, NCLS = Xte.shape[1], 5

sel = json.load(open(f"{RES}/stage2_selected_seed0.json"))
MODELS = {
    "undefended": f"{MOD}/s2_none_seed0.keras",
    f"fgsm_eps{sel['fgsm']}": f"{MOD}/s2_fgsm_eps{sel['fgsm']}_seed0.keras",
    f"pgd_eps{sel['pgd']}": f"{MOD}/s2_pgd_eps{sel['pgd']}_seed0.keras",
}
# Later, after Stage 3, add e.g.:
# MODELS["a3idps_kd_feedback_r2"] = f"{MOD}/s3_kd_feedback_eps0.05_r2_seed0.keras"   (use the eps_max Stage 3 selected)
EPS_LIST = (0.05, 0.10)
PROTOCOLS = ("all_features", "numeric_only")
N = 300
sub = np.random.RandomState(0).choice(len(Xte), N, replace=False)     # same rows as the earlier ART check
json.dump(sub.tolist(), open(f"{RES}/art_crosscheck_subset_idx.json", "w"))
X, y = Xte[sub], yte[sub]
Y1 = keras.utils.to_categorical(y, NCLS)
OUT = f"{RES}/art_crosscheck_results.json"
results = json.load(open(OUT)) if os.path.exists(OUT) else {}

# %% [1] Our attack (same logic as Stage 2: worst case over PGD-CE and PGD-margin, 20 steps x 3 restarts)
def loss_fn(z, yy, margin):
    if margin:
        oh = tf.one_hot(yy, NCLS)
        return tf.reduce_max(z - 1e9 * oh, axis=1) - tf.reduce_sum(z * oh, axis=1)
    return tf.nn.sparse_softmax_cross_entropy_with_logits(labels=yy, logits=z)

def our_pgd(model, X, y, eps, mask, margin, steps=20, restarts=3):
    E = tf.constant(eps * mask); x0 = tf.constant(X); yb = tf.constant(y)
    best_x, best_l = x0, tf.fill(tf.shape(yb), -1e30)
    for r in range(restarts):
        x = x0 if r == 0 else tf.clip_by_value(x0 + tf.random.uniform(tf.shape(x0), -1., 1.) * E, 0., 1.)
        for s_ in range(steps):
            alpha = (1.0 if s_ < steps // 2 else 0.25) * E / 4.0
            with tf.GradientTape() as t:
                t.watch(x); s = tf.reduce_sum(loss_fn(model(x, training=False), yb, margin))
            x = tf.clip_by_value(tf.clip_by_value(x + alpha * tf.sign(t.gradient(s, x)), x0 - E, x0 + E), 0., 1.)
            l_new = loss_fn(model(x, training=False), yb, margin)
            upd = l_new > best_l
            best_x = tf.where(upd[:, None], x, best_x); best_l = tf.where(upd, l_new, best_l)
    return best_x.numpy()

def correct(model, Xa):
    return np.argmax(model.predict(Xa, verbose=0), 1) == y

# %% [2] ART wrapper; find out which mask argument (if any) ART's APGD accepts
import art
from art.estimators.classification import TensorFlowV2Classifier
from art.attacks.evasion import AutoProjectedGradientDescent
print("ART version:", art.__version__)

def make_art(model):
    return TensorFlowV2Classifier(model=model, nb_classes=NCLS, input_shape=(D,),
                                  loss_object=tf.keras.losses.CategoricalCrossentropy(from_logits=True), clip_values=(0.0, 1.0))

def make_apgd(clf, eps, loss_type, n_init=2, iters=100):
    return AutoProjectedGradientDescent(estimator=clf, norm=np.inf, eps=eps, eps_step=eps / 4, max_iter=iters, targeted=False,
                                        nb_random_init=n_init, batch_size=150, loss_type=loss_type, verbose=False)

def find_art_mask(model):
    clf = make_art(model); last = None
    for cand, label in ((MASK_NUM, "1-D (122,)"), (MASK_NUM[None, :], "(1,122)")):
        try:
            Xa = make_apgd(clf, 0.05, "cross_entropy", n_init=1, iters=20).generate(x=X[:60], y=Y1[:60], mask=cand)
            frozen = float(np.abs((Xa - X[:60])[:, MASK_NUM == 0]).max())
            print(f"mask form {label}: accepted, max change on frozen columns = {frozen}")
            if frozen == 0.0: return cand
        except Exception as ex:
            last = ex; print(f"mask form {label}: rejected ({type(ex).__name__}: {ex})")
    return None

first_model = keras.models.load_model(next(iter(MODELS.values())), compile=False)
ART_MASK = find_art_mask(first_model)
print("numeric-only cross-check available:", ART_MASK is not None)

# %% [3] Run the comparison
for name, path in MODELS.items():
    model = keras.models.load_model(path, compile=False); clf = make_art(model)
    print(f"\n##### {name}  (clean acc on subset: {correct(model, X).mean():.4f})")
    for eps in EPS_LIST:
        for prot in PROTOCOLS:
            key = f"{name}|{eps}|{prot}"
            if key in results: continue
            if prot == "numeric_only" and ART_MASK is None:
                print(f"skip {key}: ART mask unavailable"); continue
            mask = MASK_NUM if prot == "numeric_only" else MASK_ALL
            ce = correct(model, our_pgd(model, X, y, eps, mask, False)); mg = correct(model, our_pgd(model, X, y, eps, mask, True))
            row = {"ours_pgd_ce": float(ce.mean()), "ours_pgd_margin": float(mg.mean()), "ours_worst": float((ce & mg).mean())}
            art_c = {}
            for lt in ("cross_entropy", "difference_logits_ratio"):
                t0 = time.time()
                kw = {"mask": ART_MASK} if prot == "numeric_only" else {}
                Xa = make_apgd(clf, eps, lt).generate(x=X, y=Y1, **kw)
                art_c[lt] = correct(model, Xa)
                row[f"art_{lt}"] = float(art_c[lt].mean())
                row[f"art_{lt}_max_delta"] = float(np.abs(Xa - X).max())
                if prot == "numeric_only": row[f"art_{lt}_frozen_change"] = float(np.abs((Xa - X)[:, MASK_NUM == 0]).max())
                print(f"  {key} ART {lt}: acc={art_c[lt].mean():.4f} ({time.time()-t0:.0f}s)")
            row["art_worst"] = float((art_c["cross_entropy"] & art_c["difference_logits_ratio"]).mean())
            row["combined_worst"] = float((ce & mg & art_c["cross_entropy"] & art_c["difference_logits_ratio"]).mean())
            row["art_stronger_by"] = row["ours_worst"] - row["art_worst"]
            results[key] = row
            json.dump(results, open(OUT, "w"), indent=1)

# %% [4] Summary (lower accuracy = stronger attack; N=300 so differences under ~3-4 pp are noise)
print(f"\n{'model|eps|protocol':<34}{'ours worst':>11}{'ART worst':>11}{'combined':>10}  note")
for key, r in results.items():
    note = "ART clearly stronger - our attack may be weak here" if r["art_stronger_by"] > 0.04 else ""
    print(f"{key:<34}{r['ours_worst']:>11.4f}{r['art_worst']:>11.4f}{r['combined_worst']:>10.4f}  {note}")
print("saved:", OUT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ART version: 1.20.1
mask form 1-D (122,): accepted, max change on frozen columns = 0.04997677356004715
mask form (1,122): accepted, max change on frozen columns = 0.0499562993645668
numeric-only cross-check available: False

##### undefended  (clean acc on subset: 0.8000)
  undefended|0.05|all_features ART cross_entropy: acc=0.1667 (10s)
  undefended|0.05|all_features ART difference_logits_ratio: acc=0.2333 (104s)
skip undefended|0.05|numeric_only: ART mask unavailable
  undefended|0.1|all_features ART cross_entropy: acc=0.0133 (7s)
  undefended|0.1|all_features ART difference_logits_ratio: acc=0.0000 (78s)
skip undefended|0.1|numeric_only: ART mask unavailable

##### fgsm_eps0.05  (clean acc on subset: 0.7767)
  fgsm_eps0.05|0.05|all_features ART cross_entropy: acc=0.5633 (13s)
  fgsm_eps0.05|0.05|all_features ART difference_logits_ratio: acc=0.5667 (135s)
s

In [ ]:
# Numeric-only cross-check for ART's APGD. ART accepted the `mask` argument but did not apply it (frozen columns changed by eps).
# Workaround: wrap each model so its one-hot columns are re-snapped to {0,1} before the network. For eps < 0.5 this erases any
# perturbation ART puts on those columns, so the attack is effectively numeric-only. Results merge into art_crosscheck_results.json.

# %% [0] Setup (same subset and models as nsl_art_crosscheck.py)
import subprocess
subprocess.run(["pip", "install", "-q", "adversarial-robustness-toolbox"], check=True)

from google.colab import drive
drive.mount('/content/drive')
import json, os, time, numpy as np, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

BASE = "/content/drive/MyDrive/A3IDPS_COLAB"
RES = f"{BASE}/results/nsl_v2"; MOD = f"{BASE}/models/nsl_v2"
d = np.load(f"{BASE}/data_processed/nslkdd_v2/nsl_v2_arrays.npz")
Xte, yte = d["Xte"].astype("float32"), d["yte"].astype("int32")
names = json.load(open(f"{BASE}/data_processed/nslkdd_v2/feature_names.json"))
MASK_NUM = np.array([0. if n.startswith(("protocol_type_", "service_", "flag_")) else 1. for n in names], "float32")
D, NCLS = Xte.shape[1], 5
FROZEN = MASK_NUM == 0

sel = json.load(open(f"{RES}/stage2_selected_seed0.json"))
MODELS = {"undefended": f"{MOD}/s2_none_seed0.keras",
          f"fgsm_eps{sel['fgsm']}": f"{MOD}/s2_fgsm_eps{sel['fgsm']}_seed0.keras",
          f"pgd_eps{sel['pgd']}": f"{MOD}/s2_pgd_eps{sel['pgd']}_seed0.keras"}
# Later add the A3IDPS models here as well.
EPS_LIST = (0.05, 0.10)
sub = np.array(json.load(open(f"{RES}/art_crosscheck_subset_idx.json")))
X, y = Xte[sub], yte[sub]
Y1 = keras.utils.to_categorical(y, NCLS)
assert np.abs(X[:, FROZEN] - np.round(X[:, FROZEN])).max() == 0, "one-hot columns are not exactly 0/1 - snapping would be invalid"
OUT = f"{RES}/art_crosscheck_results.json"
results = json.load(open(OUT))

# %% [1] Snapping wrapper, our attack, ART wrapper
M_T = tf.constant(MASK_NUM)
def snapped(model):
    inp = keras.Input(shape=(D,))
    x = layers.Lambda(lambda t: M_T * t + (1. - M_T) * tf.round(t))(inp)
    return keras.Model(inp, model(x))

def loss_fn(z, yy, margin):
    if margin:
        oh = tf.one_hot(yy, NCLS)
        return tf.reduce_max(z - 1e9 * oh, axis=1) - tf.reduce_sum(z * oh, axis=1)
    return tf.nn.sparse_softmax_cross_entropy_with_logits(labels=yy, logits=z)

def our_pgd(model, X, y, eps, margin, steps=20, restarts=3):     # numeric-only via the mask, as in Stage 2
    E = tf.constant(eps * MASK_NUM); x0 = tf.constant(X); yb = tf.constant(y)
    best_x, best_l = x0, tf.fill(tf.shape(yb), -1e30)
    for r in range(restarts):
        x = x0 if r == 0 else tf.clip_by_value(x0 + tf.random.uniform(tf.shape(x0), -1., 1.) * E, 0., 1.)
        for s_ in range(steps):
            alpha = (1.0 if s_ < steps // 2 else 0.25) * E / 4.0
            with tf.GradientTape() as t:
                t.watch(x); s = tf.reduce_sum(loss_fn(model(x, training=False), yb, margin))
            x = tf.clip_by_value(tf.clip_by_value(x + alpha * tf.sign(t.gradient(s, x)), x0 - E, x0 + E), 0., 1.)
            l_new = loss_fn(model(x, training=False), yb, margin)
            upd = l_new > best_l
            best_x, best_l = tf.where(upd[:, None], x, best_x), tf.where(upd, l_new, best_l)
    return best_x.numpy()

import art
from art.estimators.classification import TensorFlowV2Classifier
from art.attacks.evasion import AutoProjectedGradientDescent
def make_apgd(clf, eps, lt):
    return AutoProjectedGradientDescent(estimator=clf, norm=np.inf, eps=eps, eps_step=eps / 4, max_iter=100, targeted=False,
                                        nb_random_init=2, batch_size=150, loss_type=lt, verbose=False)
correct = lambda m, Xa: np.argmax(m.predict(Xa, verbose=0), 1) == y

# %% [2] Run
for name, path in MODELS.items():
    model = keras.models.load_model(path, compile=False); wm = snapped(model)
    assert (np.argmax(wm.predict(X, verbose=0), 1) == np.argmax(model.predict(X, verbose=0), 1)).all()
    clf = TensorFlowV2Classifier(model=wm, nb_classes=NCLS, input_shape=(D,),
                                 loss_object=tf.keras.losses.CategoricalCrossentropy(from_logits=True), clip_values=(0.0, 1.0))
    for eps in EPS_LIST:
        key = f"{name}|{eps}|numeric_only"
        if key in results: continue
        ce = correct(model, our_pgd(model, X, y, eps, False)); mg = correct(model, our_pgd(model, X, y, eps, True))
        row = {"ours_pgd_ce": float(ce.mean()), "ours_pgd_margin": float(mg.mean()), "ours_worst": float((ce & mg).mean()),
               "method": "ART on snapped wrapper (one-hot columns re-rounded)"}
        art_c = {}
        for lt in ("cross_entropy", "difference_logits_ratio"):
            t0 = time.time(); Xa = make_apgd(clf, eps, lt).generate(x=X, y=Y1)
            art_c[lt] = correct(wm, Xa); row[f"art_{lt}"] = float(art_c[lt].mean())
            snap = MASK_NUM * Xa + (1 - MASK_NUM) * np.round(Xa)
            row[f"art_{lt}_effective_frozen_change"] = float(np.abs((snap - X)[:, FROZEN]).max())
            row[f"art_{lt}_effective_numeric_delta"] = float(np.abs((snap - X)[:, ~FROZEN]).max())
            print(f"  {key} ART {lt}: acc={art_c[lt].mean():.4f} | effective change on frozen cols={row[f'art_{lt}_effective_frozen_change']} ({time.time()-t0:.0f}s)")
        row["art_worst"] = float((art_c["cross_entropy"] & art_c["difference_logits_ratio"]).mean())
        row["combined_worst"] = float((ce & mg & art_c["cross_entropy"] & art_c["difference_logits_ratio"]).mean())
        row["art_stronger_by"] = row["ours_worst"] - row["art_worst"]
        results[key] = row; json.dump(results, open(OUT, "w"), indent=1)

print(f"\n{'model|eps|protocol':<34}{'ours worst':>11}{'ART worst':>11}{'combined':>10}  note")
for key, r in results.items():
    note = "ART clearly stronger - our attack may be weak here" if r["art_stronger_by"] > 0.04 else ""
    print(f"{key:<34}{r['ours_worst']:>11.4f}{r['art_worst']:>11.4f}{r['combined_worst']:>10.4f}  {note}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  undefended|0.05|numeric_only ART cross_entropy: acc=0.4633 | effective change on frozen cols=0.0 (11s)
  undefended|0.05|numeric_only ART difference_logits_ratio: acc=0.5233 | effective change on frozen cols=0.0 (135s)
  undefended|0.1|numeric_only ART cross_entropy: acc=0.1067 | effective change on frozen cols=0.0 (11s)
  undefended|0.1|numeric_only ART difference_logits_ratio: acc=0.1433 | effective change on frozen cols=0.0 (100s)
  fgsm_eps0.05|0.05|numeric_only ART cross_entropy: acc=0.7500 | effective change on frozen cols=0.0 (14s)
  fgsm_eps0.05|0.05|numeric_only ART difference_logits_ratio: acc=0.7500 | effective change on frozen cols=0.0 (151s)
  fgsm_eps0.05|0.1|numeric_only ART cross_entropy: acc=0.7133 | effective change on frozen cols=0.0 (14s)
  fgsm_eps0.05|0.1|numeric_only ART difference_logits_ratio: acc=0.7267 | effective change on frozen

In [ ]:
# Stage 3 - A3IDPS on NSL-KDD under the frozen protocol (same data, attacks, selection rule as Stage 2).
# Implements what the paper describes:
#   (i)  wide teacher -> distilled student (alpha=0.7, T=3; KL taken teacher||student, the standard direction)
#   (ii) AdvGAN generator: delta = eps_max * mask * tanh(net(x)), untargeted margin objective, connected discriminator,
#        lambda_cls=20, lambda_l1=0.5, 1200 steps, trained on the TRAINING split only
#   (iii) feedback rounds: generator retrained against the current classifier, classifier retrained on clean + generated
# Ablations: with/without distillation x feedback vs fixed generator (control), rounds 0..3.
# The test set is used only for final reporting. Paste into Colab cell by cell. Resumable via files on Drive.

# %% [0] Setup (same conventions as Stage 2)
from google.colab import drive
drive.mount('/content/drive')
import os, json, random, time
import numpy as np, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight

BASE = "/content/drive/MyDrive/A3IDPS_COLAB"
DATA = f"{BASE}/data_processed/nslkdd_v2"
MOD  = f"{BASE}/models/nsl_v2"
RES  = f"{BASE}/results/nsl_v2"
os.makedirs(MOD, exist_ok=True); os.makedirs(RES, exist_ok=True)

d = np.load(f"{DATA}/nsl_v2_arrays.npz")
Xtr, Xva, Xte = [d[k].astype("float32") for k in ("Xtr", "Xva", "Xte")]
ytr, yva, yte = [d[k].astype("int32") for k in ("ytr", "yva", "yte")]
names = json.load(open(f"{DATA}/feature_names.json"))
D, NCLS = Xtr.shape[1], 5
CLASS_NAMES = ["Normal", "DoS", "Probe", "R2L", "U2R"]
MASK_NUM = np.array([0. if n.startswith(("protocol_type_", "service_", "flag_")) else 1. for n in names], "float32")
MASK_ALL = np.ones(D, "float32")
assert int(MASK_NUM.sum()) == 38

SEED = 0
EPS_EVAL, EPS_SELECT = [0.01, 0.02, 0.05, 0.10, 0.15], [0.01, 0.02, 0.05, 0.10]
EPS_TRAIN = [0.02, 0.05, 0.10]           # generator eps_max grid (same grid as the AT baselines)
CLASS_WEIGHT_CAP, VAL_CLEAN_TOL, BATCH = 20.0, 0.05, 512
KD_ALPHA, KD_T = 0.7, 3.0
GEN_STEPS, LAM_CLS, LAM_L1, LAM_GAN, KAPPA = 1200, 20.0, 0.5, 1.0, 1.0
ROUNDS = 3
FAST = True                              # tf.function training steps; set False to fall back to eager if tracing errors occur
cw = np.minimum(compute_class_weight("balanced", classes=np.arange(NCLS), y=ytr), CLASS_WEIGHT_CAP).astype("float32")

def maybe_tf(fn):
    return tf.function(fn, reduce_retracing=True) if FAST else fn

def set_seed(s):
    random.seed(s); np.random.seed(s); tf.keras.utils.set_random_seed(s)

def jsonable(o):
    if isinstance(o, dict): return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [jsonable(v) for v in o]
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return float(o)
    return o

# %% [1] Models and attacks (identical to Stage 2; models output LOGITS)
def build_model(d, widths=(256, 128, 64)):
    inp = keras.Input(shape=(d,))
    x = layers.Dense(widths[0])(inp); x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)
    x = layers.Dense(widths[1])(x);   x = layers.Dropout(0.3)(x);          x = layers.Activation("relu")(x)
    x = layers.Dense(widths[2])(x);   x = layers.Dropout(0.3)(x);          x = layers.Activation("relu")(x)
    return keras.Model(inp, layers.Dense(NCLS)(x))

def make_attacks(model, mask):
    M = tf.constant(mask, tf.float32)
    def loss_fn(z, y, margin):
        if margin:
            oh = tf.one_hot(y, NCLS)
            return tf.reduce_max(z - 1e9 * oh, axis=1) - tf.reduce_sum(z * oh, axis=1)
        return tf.nn.sparse_softmax_cross_entropy_with_logits(labels=y, logits=z)
    @tf.function
    def fgsm(x0, y, eps):
        with tf.GradientTape() as t:
            t.watch(x0); s = tf.reduce_sum(loss_fn(model(x0, training=False), y, False))
        return tf.clip_by_value(x0 + eps * M * tf.sign(t.gradient(s, x0)), 0., 1.)
    @tf.function
    def pgd(x0, y, eps, steps, restarts, margin):
        E = eps * M
        best_x, best_l = x0, tf.fill(tf.shape(y), -1e30)
        for r in range(restarts):
            x = x0 if r == 0 else tf.clip_by_value(x0 + tf.random.uniform(tf.shape(x0), -1., 1.) * E, 0., 1.)
            for s_ in range(steps):
                alpha = (1.0 if s_ < steps // 2 else 0.25) * E / 4.0
                with tf.GradientTape() as t:
                    t.watch(x); s = tf.reduce_sum(loss_fn(model(x, training=False), y, margin))
                x = tf.clip_by_value(tf.clip_by_value(x + alpha * tf.sign(t.gradient(s, x)), x0 - E, x0 + E), 0., 1.)
                l_new = loss_fn(model(x, training=False), y, margin)
                upd = l_new > best_l
                best_x = tf.where(upd[:, None], x, best_x); best_l = tf.where(upd, l_new, best_l)
        return best_x
    return {"fgsm": fgsm, "pgd": pgd}

def run_attack(fn, X, y, bs=4096, **kw):
    return np.vstack([fn(tf.constant(X[i:i+bs]), tf.constant(y[i:i+bs]), **kw).numpy() for i in range(0, len(X), bs)])

def predict(model, X):
    return np.argmax(model.predict(X, batch_size=4096, verbose=0), axis=1)

def clean_eval(model, X, y):
    p = predict(model, X)
    return {"acc": float(accuracy_score(y, p)),
            "macro_f1": float(f1_score(y, p, average="macro", labels=range(NCLS), zero_division=0)),
            "per_class_f1": dict(zip(CLASS_NAMES, f1_score(y, p, average=None, labels=range(NCLS), zero_division=0).tolist()))}

def robust_eval(model, X, y, mask, eps_list, steps, restarts):
    A = make_attacks(model, mask); out = {}
    for e in eps_list:
        et = tf.constant(e, tf.float32)
        c_f = predict(model, run_attack(A["fgsm"], X, y, eps=et)) == y
        c_c = predict(model, run_attack(A["pgd"], X, y, eps=et, steps=steps, restarts=restarts, margin=False)) == y
        c_m = predict(model, run_attack(A["pgd"], X, y, eps=et, steps=steps, restarts=restarts, margin=True)) == y
        w = c_f & c_c & c_m
        out[str(e)] = {"fgsm": float(c_f.mean()), "pgd_ce": float(c_c.mean()), "pgd_margin": float(c_m.mean()),
                       "worst": float(w.mean()), "worst_recall": {CLASS_NAMES[k]: float(w[y == k].mean()) for k in range(NCLS)}}
    return out

# %% [2] Generic classifier training: plain, distilled (zt given), or feedback (X_extra = generated samples, Eq. 4)
def fit_classifier(model, X, y, Xv, yv, X_extra=None, Xv_extra=None, zt=None,
                   lr=1e-3, max_epochs=40, patience=8, seed=0, tag=""):
    set_seed(seed)
    opt = keras.optimizers.Adam(lr); opt.build(model.trainable_variables)
    use_kd, cwt = zt is not None, tf.constant(cw)
    def wce(z, yy):
        return tf.reduce_mean(tf.nn.sparse_softmax_cross_entropy_with_logits(labels=yy, logits=z) * tf.gather(cwt, yy))

    @maybe_tf
    def step(xb, yb, ztb):
        with tf.GradientTape() as t:
            z = model(xb, training=True)
            hard = wce(z, yb)
            if use_kd:
                pt = tf.nn.softmax(ztb / KD_T)
                log_ps = tf.nn.log_softmax(z / KD_T)
                kl = tf.reduce_mean(tf.reduce_sum(pt * (tf.math.log(pt + 1e-8) - log_ps), axis=1))
                loss = KD_ALPHA * hard + (1. - KD_ALPHA) * KD_T * KD_T * kl
            else:
                loss = hard
        opt.apply_gradients(zip(t.gradient(loss, model.trainable_variables), model.trainable_variables))
        return loss

    rng, n = np.random.RandomState(seed), len(X)
    best, best_w, wait, cur_lr = 1e9, None, 0, lr
    for ep in range(max_epochs):
        t0 = time.time(); perm = rng.permutation(n)
        for i in range(0, n - BATCH + 1, BATCH):
            idx = perm[i:i+BATCH]; xb, yb = X[idx], y[idx]
            if X_extra is not None:
                xb, yb = np.concatenate([xb, X_extra[idx]]), np.concatenate([yb, y[idx]])
            ztb = zt[idx] if use_kd else np.zeros((len(idx), NCLS), "float32")
            step(tf.constant(xb), tf.constant(yb), tf.constant(ztb))
        vl = float(wce(model(tf.constant(Xv), training=False), tf.constant(yv)))
        if Xv_extra is not None:
            vl += float(wce(model(tf.constant(Xv_extra), training=False), tf.constant(yv)))
        if vl < best - 1e-4:
            best, best_w, wait = vl, model.get_weights(), 0
        else:
            wait += 1
            if wait % 4 == 0:
                cur_lr *= 0.5; opt.learning_rate = cur_lr
            if wait >= patience: break
        if (ep + 1) % 5 == 0 or ep == 0:
            print(f"[{tag}] epoch {ep+1} val_loss={vl:.4f} best={best:.4f} ({time.time()-t0:.1f}s)")
    model.set_weights(best_w)
    return model

# %% [3] AdvGAN generator (corrected): bounded delta on numeric features, untargeted margin loss, connected discriminator
def build_generator(d):
    inp = keras.Input(shape=(d,))
    x = layers.Dense(256, activation="relu")(inp); x = layers.Dense(256, activation="relu")(x)
    return keras.Model(inp, layers.Dense(d)(x))

def build_disc(d):
    inp = keras.Input(shape=(d,))
    x = layers.Dense(256)(inp); x = layers.LeakyReLU(0.2)(x)
    x = layers.Dense(256)(x);   x = layers.LeakyReLU(0.2)(x)
    return keras.Model(inp, layers.Dense(1)(x))

def apply_G(G, X, eps, bs=8192):
    E = eps * MASK_NUM
    return np.vstack([np.clip(X[i:i+bs] + E * np.tanh(G(tf.constant(X[i:i+bs]), training=False).numpy()), 0., 1.)
                      for i in range(0, len(X), bs)]).astype("float32")

def train_generator(clf, eps, seed=SEED, tag=""):
    set_seed(seed)
    G, Dn = build_generator(D), build_disc(D)
    g_opt = keras.optimizers.Adam(1e-3, beta_1=0.5); g_opt.build(G.trainable_variables)
    d_opt = keras.optimizers.Adam(1e-4, beta_1=0.5); d_opt.build(Dn.trainable_variables)
    E = tf.constant(eps * MASK_NUM)
    bce = lambda labels, logits: tf.reduce_mean(tf.nn.sigmoid_cross_entropy_with_logits(labels=labels, logits=logits))

    def gen(x):
        return tf.clip_by_value(x + E * tf.tanh(G(x, training=True)), 0., 1.)

    @maybe_tf
    def step(x, y):
        with tf.GradientTape() as td:
            xa = gen(x)
            ld = bce(tf.ones([tf.shape(x)[0], 1]), Dn(x, training=True)) + bce(tf.zeros([tf.shape(x)[0], 1]), Dn(xa, training=True))
        d_opt.apply_gradients(zip(td.gradient(ld, Dn.trainable_variables), Dn.trainable_variables))
        with tf.GradientTape() as tg:
            xa = gen(x)
            z = clf(xa, training=False)
            oh = tf.one_hot(y, NCLS)
            margin = tf.reduce_sum(z * oh, 1) - tf.reduce_max(z - 1e9 * oh, 1)       # >0: still correctly classified
            l_adv = tf.reduce_mean(tf.nn.relu(margin + KAPPA))                        # untargeted, bounded below
            l_gan = bce(tf.ones([tf.shape(x)[0], 1]), Dn(xa, training=False))
            l_l1 = tf.reduce_mean(tf.abs(xa - x))
            lg = LAM_CLS * l_adv + LAM_GAN * l_gan + LAM_L1 * l_l1
        g_opt.apply_gradients(zip(tg.gradient(lg, G.trainable_variables), G.trainable_variables))
        return lg

    rng = np.random.RandomState(seed)
    for s in range(GEN_STEPS):
        idx = rng.randint(0, len(Xtr), BATCH)
        lg = step(tf.constant(Xtr[idx]), tf.constant(ytr[idx]))
        if (s + 1) % 400 == 0: print(f"[{tag}] generator step {s+1}/{GEN_STEPS} loss={float(lg):.4f}")
    Xa = apply_G(G, Xva, eps)
    a0, a1 = float((predict(clf, Xva) == yva).mean()), float((predict(clf, Xa) == yva).mean())
    print(f"[{tag}] generator sanity (validation): acc clean={a0:.4f} -> under G={a1:.4f}; mean|delta|={np.abs(Xa - Xva).mean():.5f}")
    if a0 - a1 < 0.05:
        print(f"[{tag}] WARNING: generator lowers accuracy by < 5 pp - it is a weak attack; check objective/lr before trusting rounds")
    return G

# %% [4] Teacher, baselines (with / without distillation)
def get_teacher():
    path = f"{MOD}/s3_teacher_seed{SEED}.keras"
    if os.path.exists(path): return keras.models.load_model(path, compile=False)
    t = build_model(D, widths=(512, 256, 128))
    fit_classifier(t, Xtr, ytr, Xva, yva, seed=SEED, tag="teacher"); t.save(path); return t

teacher = get_teacher()
zt_tr = teacher.predict(Xtr, batch_size=4096, verbose=0).astype("float32")
teacher_report = {"val": clean_eval(teacher, Xva, yva), "test": clean_eval(teacher, Xte, yte),
                  "params": int(teacher.count_params())}
print("TEACHER clean: val acc", round(teacher_report["val"]["acc"], 4), "| test acc", round(teacher_report["test"]["acc"], 4),
      "| test macro-F1", round(teacher_report["test"]["macro_f1"], 4), "| params", teacher_report["params"])

def get_baseline(distil):
    path = f"{MOD}/s3_base_{'kd' if distil else 'nokd'}_seed{SEED}.keras"
    if os.path.exists(path): return keras.models.load_model(path, compile=False)
    m = build_model(D)
    fit_classifier(m, Xtr, ytr, Xva, yva, zt=zt_tr if distil else None, seed=SEED, tag=f"base_{'kd' if distil else 'nokd'}")
    m.save(path); return m

# %% [5] Feedback pipeline: mode "feedback" retrains the generator each round; "fixed_gen" is the control
def run_pipeline(distil, mode, eps, rounds=ROUNDS):
    ptag = f"{'kd' if distil else 'nokd'}_{mode}"
    prev = get_baseline(distil); clfs, G_fixed = {0: prev}, None
    for r in range(1, rounds + 1):
        cpath = f"{MOD}/s3_{ptag}_eps{eps}_r{r}_seed{SEED}.keras"
        gpath = f"{MOD}/s3_gen_{ptag}_eps{eps}_r{r}_seed{SEED}.keras"
        if os.path.exists(cpath):
            clfs[r] = prev = keras.models.load_model(cpath, compile=False)
            if mode == "fixed_gen" and r == 1: G_fixed = keras.models.load_model(gpath, compile=False)
            continue
        if mode == "feedback" or r == 1:
            G = train_generator(prev, eps, tag=f"{ptag} eps={eps} r{r}"); G.save(gpath)
            if r == 1: G_fixed = G
        else:
            G = G_fixed
        Xa_tr, Xa_va = apply_G(G, Xtr, eps), apply_G(G, Xva, eps)
        new = build_model(D); new.set_weights(prev.get_weights())
        fit_classifier(new, Xtr, ytr, Xva, yva, X_extra=Xa_tr, Xv_extra=Xa_va, lr=5e-4, max_epochs=30,
                       seed=SEED + r, tag=f"{ptag} eps={eps} r{r}")
        new.save(cpath); clfs[r] = prev = new
    return clfs

def val_score(m):
    vr = robust_eval(m, Xva, yva, MASK_NUM, EPS_SELECT, steps=10, restarts=1)
    return {"clean": clean_eval(m, Xva, yva)["acc"], "robust_mean": float(np.mean([vr[str(e)]["worst"] for e in EPS_SELECT]))}

# %% [6] Select the generator eps_max on VALIDATION (main pipeline: distilled + feedback, paper's R=2), same rule as Stage 2
und_val_clean = json.load(open(f"{RES}/stage2_val_seed{SEED}.json"))["none"]["val_clean"]["acc"]
grid = {}
for eps in EPS_TRAIN:
    clfs = run_pipeline(True, "feedback", eps)
    grid[eps] = {"models": clfs, "val": {r: val_score(clfs[r]) for r in clfs}}
    print(f"VAL eps_max={eps}: " + ", ".join(f"r{r}: clean={v['clean']:.4f} robust={v['robust_mean']:.4f}" for r, v in grid[eps]["val"].items()))
elig = [e for e in EPS_TRAIN if und_val_clean - grid[e]["val"][2]["clean"] <= VAL_CLEAN_TOL]
pool = elig if elig else EPS_TRAIN
if not elig: print("WARNING: no eps_max within the clean-accuracy tolerance at round 2")
eps_sel = max(pool, key=lambda e: (round(grid[e]["val"][2]["robust_mean"], 6), -e))
print("SELECTED generator eps_max:", eps_sel)
json.dump(jsonable({"selected_eps": eps_sel, "teacher": teacher_report, "val_grid": {e: grid[e]["val"] for e in EPS_TRAIN}}),
          open(f"{RES}/stage3_selection_seed{SEED}.json", "w"), indent=1)

# %% [7] Remaining pipelines at the selected eps_max, then TEST evaluation of every (pipeline, round)
all_models = {("kd_feedback", r): m for r, m in grid[eps_sel]["models"].items()}
for distil, mode in [(False, "feedback"), (True, "fixed_gen"), (False, "fixed_gen")]:
    for r, m in run_pipeline(distil, mode, eps_sel).items():
        all_models[(f"{'kd' if distil else 'nokd'}_{mode}", r)] = m

test_res = {}
for (ptag, r), m in sorted(all_models.items()):
    test_res[f"{ptag}_r{r}"] = {"clean": clean_eval(m, Xte, yte),
                                "numeric_only": robust_eval(m, Xte, yte, MASK_NUM, EPS_EVAL, steps=20, restarts=3),
                                "all_features": robust_eval(m, Xte, yte, MASK_ALL, EPS_EVAL, steps=20, restarts=3)}
    json.dump(jsonable({"eps_max": eps_sel, "test": test_res}), open(f"{RES}/stage3_test_seed{SEED}.json", "w"), indent=1)

s2 = json.load(open(f"{RES}/stage2_test_seed{SEED}.json"))
ref = {"undefended (S2)": "none", f"FGSM-AT eps={s2['selected']['fgsm']} (S2)": f"fgsm_eps{s2['selected']['fgsm']}",
       f"PGD-AT eps={s2['selected']['pgd']} (S2)": f"pgd_eps{s2['selected']['pgd']}"}
for prot in ("numeric_only", "all_features"):
    print(f"\n=== TEST worst-case robust accuracy, {prot} ===")
    print(f"{'model':<26}{'clean':>8}{'mF1':>8}" + "".join(f"{('e='+str(e)):>8}" for e in EPS_EVAL))
    for name, key in ref.items():
        r_, c_ = s2["test"][key][prot], s2["test"][key]["clean"]
        print(f"{name:<26}{c_['acc']:>8.4f}{c_['macro_f1']:>8.4f}" + "".join(f"{r_[str(e)]['worst']:>8.4f}" for e in EPS_EVAL))
    for key in sorted(test_res):
        r_, c_ = test_res[key][prot], test_res[key]["clean"]
        print(f"{key:<26}{c_['acc']:>8.4f}{c_['macro_f1']:>8.4f}" + "".join(f"{r_[str(e)]['worst']:>8.4f}" for e in EPS_EVAL))
print("\nSaved:", f"{RES}/stage3_test_seed{SEED}.json")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[teacher] epoch 1 val_loss=0.1918 best=0.1918 (2.7s)
[teacher] epoch 5 val_loss=0.0430 best=0.0388 (0.9s)
[teacher] epoch 10 val_loss=0.0362 best=0.0362 (0.9s)
[teacher] epoch 15 val_loss=0.0453 best=0.0362 (0.9s)
TEACHER clean: val acc 0.9779 | test acc 0.8007 | test macro-F1 0.6241 | params 229893
[base_kd] epoch 1 val_loss=0.1176 best=0.1176 (1.7s)
[base_kd] epoch 5 val_loss=0.0359 best=0.0359 (0.9s)
[base_kd] epoch 10 val_loss=0.0353 best=0.0353 (0.9s)
[base_kd] epoch 15 val_loss=0.0365 best=0.0353 (0.9s)
[base_kd] epoch 20 val_loss=0.0337 best=0.0337 (0.9s)
[base_kd] epoch 25 val_loss=0.0344 best=0.0325 (0.9s)
[base_kd] epoch 30 val_loss=0.0349 best=0.0312 (0.9s)
[kd_feedback eps=0.02 r1] generator step 400/1200 loss=279.8183
[kd_feedback eps=0.02 r1] generator step 800/1200 loss=274.9428
[kd_feedback eps=0.02 r1] generator step 1200/1200 loss=291.1619
[

VAL eps_max=0.02: r0: clean=0.9876 robust=0.7198, r1: clean=0.9844 robust=0.7225, r2: clean=0.9839 robust=0.7540, r3: clean=0.9863 robust=0.7629
[kd_feedback eps=0.05 r1] generator step 400/1200 loss=201.5370
[kd_feedback eps=0.05 r1] generator step 800/1200 loss=195.0512
[kd_feedback eps=0.05 r1] generator step 1200/1200 loss=205.4068
[kd_feedback eps=0.05 r1] generator sanity (validation): acc clean=0.9876 -> under G=0.7555; mean|delta|=0.00719
[kd_feedback eps=0.05 r1] epoch 1 val_loss=0.0872 best=0.0872 (1.5s)
[kd_feedback eps=0.05 r1] epoch 5 val_loss=0.0812 best=0.0732 (1.0s)
[kd_feedback eps=0.05 r1] epoch 10 val_loss=0.0912 best=0.0732 (0.9s)
[kd_feedback eps=0.05 r2] generator step 400/1200 loss=284.0813
[kd_feedback eps=0.05 r2] generator step 800/1200 loss=279.5421
[kd_feedback eps=0.05 r2] generator step 1200/1200 loss=292.0574
[kd_feedback eps=0.05 r2] generator sanity (validation): acc clean=0.9848 -> under G=0.8451; mean|delta|=0.00614
[kd_feedback eps=0.05 r2] epoch 1 v

In [1]:
# Stage 4a - NSL-KDD, 5 seeds: undefended, FGSM-AT, PGD-AT, teacher, and the A3IDPS pipelines up to round 5.
# Protocol frozen from Stages 2-3: same data split (seed 42), numeric-only main protocol, attack suite = worst case over
# FGSM / PGD-CE / PGD-margin (20 steps x 3 restarts). Hyper-parameters chosen on seed-0 VALIDATION data are reused for all seeds:
#   eps_train for FGSM-AT / PGD-AT  <- stage2_selected_seed0.json
#   generator eps_max               <- stage3_selection_seed0.json
# Only model / generator initialisation and batch order change with the seed. The test set is used only for final reporting.
# Resumable: models and per-seed results are cached in Drive. Run cell [0]-[5] once; then run cell [6] as often as you like.

# %% [0] Setup
from google.colab import drive
drive.mount('/content/drive')
import os, json, random, time
import numpy as np, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight

BASE = "/content/drive/MyDrive/A3IDPS_COLAB"
DATA = f"{BASE}/data_processed/nslkdd_v2"
MOD  = f"{BASE}/models/nsl_v2"
RES  = f"{BASE}/results/nsl_v2"
os.makedirs(MOD, exist_ok=True); os.makedirs(RES, exist_ok=True)

d = np.load(f"{DATA}/nsl_v2_arrays.npz")
Xtr, Xva, Xte = [d[k].astype("float32") for k in ("Xtr", "Xva", "Xte")]
ytr, yva, yte = [d[k].astype("int32") for k in ("ytr", "yva", "yte")]
names = json.load(open(f"{DATA}/feature_names.json"))
D, NCLS = Xtr.shape[1], 5
CLASS_NAMES = ["Normal", "DoS", "Probe", "R2L", "U2R"]
MASK_NUM = np.array([0. if n.startswith(("protocol_type_", "service_", "flag_")) else 1. for n in names], "float32")
MASK_ALL = np.ones(D, "float32")
assert int(MASK_NUM.sum()) == 38

SEEDS = [0, 1, 2, 3, 4]
ROUNDS = 5
sel2 = json.load(open(f"{RES}/stage2_selected_seed0.json"))
sel3 = json.load(open(f"{RES}/stage3_selection_seed0.json"))
EPS_AT  = {"fgsm": sel2["fgsm"], "pgd": sel2["pgd"]}
EPS_GEN = sel3["selected_eps"]
print("fixed from seed-0 validation:", EPS_AT, "| generator eps_max:", EPS_GEN)

EPS_NUM, EPS_ALL = [0.01, 0.02, 0.05, 0.10, 0.15], [0.05, 0.10]
KEY_ALL = {"undefended", "fgsm_at", "pgd_at", "kd_feedback_r3", "kd_feedback_r5"}   # tags that also get the all-features stress test
CLASS_WEIGHT_CAP, BATCH = 20.0, 512
KD_ALPHA, KD_T = 0.7, 3.0
GEN_STEPS, LAM_CLS, LAM_L1, LAM_GAN, KAPPA = 1200, 20.0, 0.5, 1.0, 1.0
FAST = True                                   # set False to fall back to eager steps if tf.function tracing errors occur
cw = np.minimum(compute_class_weight("balanced", classes=np.arange(NCLS), y=ytr), CLASS_WEIGHT_CAP).astype("float32")

def maybe_tf(fn): return tf.function(fn, reduce_retracing=True) if FAST else fn
def set_seed(s): random.seed(s); np.random.seed(s); tf.keras.utils.set_random_seed(s)
def jsonable(o):
    if isinstance(o, dict): return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [jsonable(v) for v in o]
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return float(o)
    return o

# %% [1] Models and attacks (identical to Stages 2-3)
def build_model(d, widths=(256, 128, 64)):
    inp = keras.Input(shape=(d,))
    x = layers.Dense(widths[0])(inp); x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)
    x = layers.Dense(widths[1])(x);   x = layers.Dropout(0.3)(x);          x = layers.Activation("relu")(x)
    x = layers.Dense(widths[2])(x);   x = layers.Dropout(0.3)(x);          x = layers.Activation("relu")(x)
    return keras.Model(inp, layers.Dense(NCLS)(x))

def make_attacks(model, mask):
    M = tf.constant(mask, tf.float32)
    def loss_fn(z, y, margin):
        if margin:
            oh = tf.one_hot(y, NCLS)
            return tf.reduce_max(z - 1e9 * oh, axis=1) - tf.reduce_sum(z * oh, axis=1)
        return tf.nn.sparse_softmax_cross_entropy_with_logits(labels=y, logits=z)
    @tf.function
    def fgsm(x0, y, eps):
        with tf.GradientTape() as t:
            t.watch(x0); s = tf.reduce_sum(loss_fn(model(x0, training=False), y, False))
        return tf.clip_by_value(x0 + eps * M * tf.sign(t.gradient(s, x0)), 0., 1.)
    @tf.function
    def pgd(x0, y, eps, steps, restarts, margin):
        E = eps * M
        best_x, best_l = x0, tf.fill(tf.shape(y), -1e30)
        for r in range(restarts):
            x = x0 if r == 0 else tf.clip_by_value(x0 + tf.random.uniform(tf.shape(x0), -1., 1.) * E, 0., 1.)
            for s_ in range(steps):
                alpha = (1.0 if s_ < steps // 2 else 0.25) * E / 4.0
                with tf.GradientTape() as t:
                    t.watch(x); s = tf.reduce_sum(loss_fn(model(x, training=False), y, margin))
                x = tf.clip_by_value(tf.clip_by_value(x + alpha * tf.sign(t.gradient(s, x)), x0 - E, x0 + E), 0., 1.)
                l_new = loss_fn(model(x, training=False), y, margin)
                upd = l_new > best_l
                best_x, best_l = tf.where(upd[:, None], x, best_x), tf.where(upd, l_new, best_l)
        return best_x
    return {"fgsm": fgsm, "pgd": pgd}

def run_attack(fn, X, y, bs=4096, **kw):
    return np.vstack([fn(tf.constant(X[i:i+bs]), tf.constant(y[i:i+bs]), **kw).numpy() for i in range(0, len(X), bs)])

def predict(model, X): return np.argmax(model.predict(X, batch_size=4096, verbose=0), axis=1)

def clean_eval(model, X, y):
    p = predict(model, X)
    return {"acc": float(accuracy_score(y, p)),
            "macro_f1": float(f1_score(y, p, average="macro", labels=range(NCLS), zero_division=0)),
            "per_class_f1": dict(zip(CLASS_NAMES, f1_score(y, p, average=None, labels=range(NCLS), zero_division=0).tolist()))}

def robust_eval(model, X, y, mask, eps_list, steps=20, restarts=3):
    A = make_attacks(model, mask); out = {}
    for e in eps_list:
        et = tf.constant(e, tf.float32)
        c_f = predict(model, run_attack(A["fgsm"], X, y, eps=et)) == y
        c_c = predict(model, run_attack(A["pgd"], X, y, eps=et, steps=steps, restarts=restarts, margin=False)) == y
        c_m = predict(model, run_attack(A["pgd"], X, y, eps=et, steps=steps, restarts=restarts, margin=True)) == y
        w = c_f & c_c & c_m
        out[str(e)] = {"fgsm": float(c_f.mean()), "pgd_ce": float(c_c.mean()), "pgd_margin": float(c_m.mean()),
                       "worst": float(w.mean()), "worst_recall": {CLASS_NAMES[k]: float(w[y == k].mean()) for k in range(NCLS)}}
    return out

# %% [2] Classifier training: plain / distilled / adversarial training on the fly (at) / feedback (X_extra)
def fit_classifier(model, X, y, Xv, yv, X_extra=None, Xv_extra=None, zt=None, at=None,
                   lr=1e-3, max_epochs=40, patience=8, seed=0, tag=""):
    set_seed(seed)
    opt = keras.optimizers.Adam(lr); opt.build(model.trainable_variables)
    use_kd, cwt = zt is not None, tf.constant(cw)
    A = make_attacks(model, MASK_NUM) if at is not None else None
    et = tf.constant(at[1] if at else 0., tf.float32)
    def wce(z, yy):
        return tf.reduce_mean(tf.nn.sparse_softmax_cross_entropy_with_logits(labels=yy, logits=z) * tf.gather(cwt, yy))

    @maybe_tf
    def step(xb, yb, ztb):
        with tf.GradientTape() as t:
            z = model(xb, training=True)
            hard = wce(z, yb)
            if use_kd:
                pt = tf.nn.softmax(ztb / KD_T); log_ps = tf.nn.log_softmax(z / KD_T)
                kl = tf.reduce_mean(tf.reduce_sum(pt * (tf.math.log(pt + 1e-8) - log_ps), axis=1))
                loss = KD_ALPHA * hard + (1. - KD_ALPHA) * KD_T * KD_T * kl
            else:
                loss = hard
        opt.apply_gradients(zip(t.gradient(loss, model.trainable_variables), model.trainable_variables))
        return loss

    rng, n = np.random.RandomState(seed), len(X)
    best, best_w, wait, cur_lr = 1e9, None, 0, lr
    for ep in range(max_epochs):
        t0 = time.time(); perm = rng.permutation(n)
        for i in range(0, n - BATCH + 1, BATCH):
            idx = perm[i:i+BATCH]; xb, yb = X[idx], y[idx]
            if X_extra is not None:
                xb, yb = np.concatenate([xb, X_extra[idx]]), np.concatenate([yb, y[idx]])
            ztb = zt[idx] if use_kd else np.zeros((len(idx), NCLS), "float32")
            xt, yt = tf.constant(xb), tf.constant(yb)
            if at is not None:
                xa = A["fgsm"](xt, yt, eps=et) if at[0] == "fgsm" else A["pgd"](xt, yt, eps=et, steps=10, restarts=1, margin=False)
                xt, yt = tf.concat([xt, xa], 0), tf.concat([yt, yt], 0)
            step(xt, yt, tf.constant(ztb))
        vl = float(wce(model(tf.constant(Xv), training=False), tf.constant(yv)))
        if Xv_extra is not None:
            vl += float(wce(model(tf.constant(Xv_extra), training=False), tf.constant(yv)))
        if at is not None:
            xv = run_attack(A["fgsm"], Xv, yv, eps=et)
            vl += float(wce(model(tf.constant(xv), training=False), tf.constant(yv)))
        if vl < best - 1e-4:
            best, best_w, wait = vl, model.get_weights(), 0
        else:
            wait += 1
            if wait % 4 == 0: cur_lr *= 0.5; opt.learning_rate = cur_lr
            if wait >= patience: break
        if (ep + 1) % 10 == 0 or ep == 0:
            print(f"[{tag}] epoch {ep+1} val_loss={vl:.4f} best={best:.4f} ({time.time()-t0:.1f}s)")
    model.set_weights(best_w)
    return model

# %% [3] AdvGAN generator (corrected objective, as in Stage 3)
def build_generator(d):
    inp = keras.Input(shape=(d,))
    x = layers.Dense(256, activation="relu")(inp); x = layers.Dense(256, activation="relu")(x)
    return keras.Model(inp, layers.Dense(d)(x))

def build_disc(d):
    inp = keras.Input(shape=(d,))
    x = layers.Dense(256)(inp); x = layers.LeakyReLU(0.2)(x)
    x = layers.Dense(256)(x);   x = layers.LeakyReLU(0.2)(x)
    return keras.Model(inp, layers.Dense(1)(x))

def apply_G(G, X, eps, bs=8192):
    E = eps * MASK_NUM
    return np.vstack([np.clip(X[i:i+bs] + E * np.tanh(G(tf.constant(X[i:i+bs]), training=False).numpy()), 0., 1.)
                      for i in range(0, len(X), bs)]).astype("float32")

def train_generator(clf, eps, seed, tag="", diag_path=None):
    set_seed(seed)
    G, Dn = build_generator(D), build_disc(D)
    g_opt = keras.optimizers.Adam(1e-3, beta_1=0.5); g_opt.build(G.trainable_variables)
    d_opt = keras.optimizers.Adam(1e-4, beta_1=0.5); d_opt.build(Dn.trainable_variables)
    E = tf.constant(eps * MASK_NUM)
    bce = lambda labels, logits: tf.reduce_mean(tf.nn.sigmoid_cross_entropy_with_logits(labels=labels, logits=logits))
    def gen(x): return tf.clip_by_value(x + E * tf.tanh(G(x, training=True)), 0., 1.)

    @maybe_tf
    def step(x, y):
        with tf.GradientTape() as td:
            xa = gen(x)
            ld = bce(tf.ones([tf.shape(x)[0], 1]), Dn(x, training=True)) + bce(tf.zeros([tf.shape(x)[0], 1]), Dn(xa, training=True))
        d_opt.apply_gradients(zip(td.gradient(ld, Dn.trainable_variables), Dn.trainable_variables))
        with tf.GradientTape() as tg:
            xa = gen(x); z = clf(xa, training=False); oh = tf.one_hot(y, NCLS)
            margin = tf.reduce_sum(z * oh, 1) - tf.reduce_max(z - 1e9 * oh, 1)
            lg = (LAM_CLS * tf.reduce_mean(tf.nn.relu(margin + KAPPA))
                  + LAM_GAN * bce(tf.ones([tf.shape(x)[0], 1]), Dn(xa, training=False))
                  + LAM_L1 * tf.reduce_mean(tf.abs(xa - x)))
        g_opt.apply_gradients(zip(tg.gradient(lg, G.trainable_variables), G.trainable_variables))
        return lg

    rng = np.random.RandomState(seed)
    for s in range(GEN_STEPS):
        idx = rng.randint(0, len(Xtr), BATCH)
        step(tf.constant(Xtr[idx]), tf.constant(ytr[idx]))
    Xa = apply_G(G, Xva, eps)
    a0, a1 = float((predict(clf, Xva) == yva).mean()), float((predict(clf, Xa) == yva).mean())
    print(f"[{tag}] generator sanity (validation): acc clean={a0:.4f} -> under G={a1:.4f}")
    if diag_path: json.dump({"acc_clean": a0, "acc_under_G": a1, "mean_abs_delta": float(np.abs(Xa - Xva).mean())}, open(diag_path, "w"))
    return G

# %% [4] Teacher, baselines, pipelines (per seed)
def get_teacher(seed):
    path = f"{MOD}/s4_teacher_seed{seed}.keras"
    if os.path.exists(path): return keras.models.load_model(path, compile=False)
    t = build_model(D, widths=(512, 256, 128))
    fit_classifier(t, Xtr, ytr, Xva, yva, seed=seed, tag=f"teacher s{seed}"); t.save(path); return t

def get_baseline(distil, seed):
    path = f"{MOD}/s4_base_{'kd' if distil else 'nokd'}_seed{seed}.keras"
    if os.path.exists(path): return keras.models.load_model(path, compile=False)
    zt = get_teacher(seed).predict(Xtr, batch_size=4096, verbose=0).astype("float32") if distil else None
    m = build_model(D)
    fit_classifier(m, Xtr, ytr, Xva, yva, zt=zt, seed=seed, tag=f"base_{'kd' if distil else 'nokd'} s{seed}")
    m.save(path); return m

def run_pipeline(distil, mode, eps, seed, rounds=ROUNDS):
    ptag = f"{'kd' if distil else 'nokd'}_{mode}"
    prev = get_baseline(distil, seed); clfs, G_fixed = {0: prev}, None
    for r in range(1, rounds + 1):
        cpath = f"{MOD}/s4_{ptag}_r{r}_seed{seed}.keras"; gpath = f"{MOD}/s4_gen_{ptag}_r{r}_seed{seed}.keras"
        if os.path.exists(cpath):
            clfs[r] = prev = keras.models.load_model(cpath, compile=False)
            if mode == "fixed_gen" and r == 1: G_fixed = keras.models.load_model(gpath, compile=False)
            continue
        if mode == "feedback" or r == 1:
            G = train_generator(prev, eps, seed * 100 + r, tag=f"{ptag} s{seed} r{r}", diag_path=gpath.replace(".keras", ".json"))
            G.save(gpath)
            if r == 1: G_fixed = G
        else:
            G = G_fixed
        Xa_tr, Xa_va = apply_G(G, Xtr, eps), apply_G(G, Xva, eps)
        new = build_model(D); new.set_weights(prev.get_weights())
        fit_classifier(new, Xtr, ytr, Xva, yva, X_extra=Xa_tr, Xv_extra=Xa_va, lr=5e-4, max_epochs=30,
                       seed=seed * 100 + r, tag=f"{ptag} s{seed} r{r}")
        new.save(cpath); clfs[r] = prev = new
    return clfs

# %% [5] Run every seed (resumable) and evaluate on TEST
def evaluate_tag(res, tag, m, path):
    have = res.get(tag, {})
    if "numeric_only" in have and (tag not in KEY_ALL or "all_features" in have): return
    r = {"clean": clean_eval(m, Xte, yte), "numeric_only": robust_eval(m, Xte, yte, MASK_NUM, EPS_NUM)}
    if tag in KEY_ALL: r["all_features"] = robust_eval(m, Xte, yte, MASK_ALL, EPS_ALL)
    res[tag] = r; json.dump(jsonable(res), open(path, "w"), indent=1)
    print(f"  evaluated {tag}: clean={r['clean']['acc']:.4f}, worst@0.05={r['numeric_only']['0.05']['worst']:.4f}, worst@0.10={r['numeric_only']['0.1']['worst']:.4f}")

def run_seed(seed):
    out = f"{RES}/stage4a_seed{seed}.json"
    res = json.load(open(out)) if os.path.exists(out) else {}
    print(f"\n########## seed {seed} ##########")
    for tag, at in [("undefended", None), ("fgsm_at", ("fgsm", EPS_AT["fgsm"])), ("pgd_at", ("pgd", EPS_AT["pgd"]))]:
        path = f"{MOD}/s4_{tag}_seed{seed}.keras"
        if os.path.exists(path): m = keras.models.load_model(path, compile=False)
        else:
            m = build_model(D); fit_classifier(m, Xtr, ytr, Xva, yva, at=at, seed=seed, tag=f"{tag} s{seed}"); m.save(path)
        evaluate_tag(res, tag, m, out)
    for distil, mode in [(True, "feedback"), (True, "fixed_gen"), (False, "feedback")]:
        clfs = run_pipeline(distil, mode, EPS_GEN, seed)
        base_tag = "kd_base" if distil else "nokd_base"
        if mode == "feedback": evaluate_tag(res, base_tag, clfs[0], out)
        for r in range(1, ROUNDS + 1):
            evaluate_tag(res, f"{'kd' if distil else 'nokd'}_{mode}_r{r}", clfs[r], out)
    keras.backend.clear_session()

for s in SEEDS:
    run_seed(s)

# %% [6] Aggregate (mean +- std over the seeds finished so far) and paired differences
def collect():
    return {s: json.load(open(f"{RES}/stage4a_seed{s}.json")) for s in SEEDS if os.path.exists(f"{RES}/stage4a_seed{s}.json")}
ORDER = (["undefended", "fgsm_at", "pgd_at", "kd_base"] + [f"kd_feedback_r{r}" for r in range(1, ROUNDS + 1)]
         + [f"kd_fixed_gen_r{r}" for r in range(1, ROUNDS + 1)] + ["nokd_base"] + [f"nokd_feedback_r{r}" for r in range(1, ROUNDS + 1)])
def ms(v):
    a = np.array(v); return f"{100*a.mean():5.1f}+-{100*(a.std(ddof=1) if len(a) > 1 else 0):4.1f}"

allres = collect(); n_seeds = len(allres)
print(f"seeds available: {sorted(allres)}")
for prot, eps_list in (("numeric_only", EPS_NUM), ("all_features", EPS_ALL)):
    print(f"\n=== TEST worst-case accuracy (%), {prot}, mean+-std over {n_seeds} seeds ===")
    print(f"{'model':<20}{'clean':>12}{'macroF1':>12}" + "".join(f"{('e='+str(e)):>12}" for e in eps_list))
    for tag in ORDER:
        rows = [allres[s][tag] for s in allres if tag in allres[s] and prot in allres[s][tag]]
        if not rows: continue
        print(f"{tag:<20}{ms([r['clean']['acc'] for r in rows]):>12}{ms([r['clean']['macro_f1'] for r in rows]):>12}"
              + "".join(f"{ms([r[prot][str(e)]['worst'] for r in rows]):>12}" for e in eps_list))

print("\n=== Paired differences in worst-case accuracy, numeric_only (percentage points, mean+-std over seeds) ===")
PAIRS = [("kd_feedback_r3", "kd_fixed_gen_r3"), ("kd_feedback_r5", "kd_fixed_gen_r5"),
         ("kd_feedback_r3", "nokd_feedback_r3"), ("kd_feedback_r5", "nokd_feedback_r5"),
         ("kd_feedback_r5", "undefended"), ("kd_feedback_r5", "pgd_at")]
for a, b in PAIRS:
    for e in (0.05, 0.10):
        diffs = [100 * (allres[s][a]["numeric_only"][str(e)]["worst"] - allres[s][b]["numeric_only"][str(e)]["worst"])
                 for s in allres if a in allres[s] and b in allres[s]]
        if diffs:
            print(f"{a} - {b} @ eps={e}: {np.mean(diffs):+6.1f} +- {(np.std(diffs, ddof=1) if len(diffs) > 1 else 0):4.1f}  (positive in {sum(x > 0 for x in diffs)}/{len(diffs)} seeds)")

Mounted at /content/drive
fixed from seed-0 validation: {'fgsm': 0.05, 'pgd': 0.1} | generator eps_max: 0.05

########## seed 0 ##########
[undefended s0] epoch 1 val_loss=0.1462 best=0.1462 (3.7s)
[undefended s0] epoch 10 val_loss=0.0365 best=0.0365 (0.8s)
  evaluated undefended: clean=0.7930, worst@0.05=0.4646, worst@0.10=0.1898
[fgsm_at s0] epoch 1 val_loss=0.3329 best=0.3329 (2.1s)
[fgsm_at s0] epoch 10 val_loss=0.1186 best=0.1186 (1.2s)
  evaluated fgsm_at: clean=0.7941, worst@0.05=0.7564, worst@0.10=0.7009
[pgd_at s0] epoch 1 val_loss=0.3702 best=0.3702 (4.5s)
[pgd_at s0] epoch 10 val_loss=0.1338 best=0.1338 (2.5s)
[pgd_at s0] epoch 20 val_loss=0.1462 best=0.1299 (2.5s)
  evaluated pgd_at: clean=0.7938, worst@0.05=0.7675, worst@0.10=0.7472
[teacher s0] epoch 1 val_loss=0.1867 best=0.1867 (1.4s)
[teacher s0] epoch 10 val_loss=0.0381 best=0.0351 (0.9s)
[base_kd s0] epoch 1 val_loss=0.1185 best=0.1185 (1.7s)
[base_kd s0] epoch 10 val_loss=0.0336 best=0.0336 (0.9s)
[base_kd s0] epoch

[kd_feedback s0 r2] generator sanity (validation): acc clean=0.9829 -> under G=0.8065
[kd_feedback s0 r2] epoch 1 val_loss=0.0922 best=0.0922 (1.5s)
[kd_feedback s0 r2] epoch 10 val_loss=0.0912 best=0.0742 (0.9s)
[kd_feedback s0 r3] generator sanity (validation): acc clean=0.9855 -> under G=0.8469
[kd_feedback s0 r3] epoch 1 val_loss=0.0812 best=0.0812 (1.5s)
[kd_feedback s0 r3] epoch 10 val_loss=0.0753 best=0.0698 (0.9s)
[kd_feedback s0 r4] generator sanity (validation): acc clean=0.9886 -> under G=0.8543
[kd_feedback s0 r4] epoch 1 val_loss=0.0874 best=0.0874 (1.4s)
[kd_feedback s0 r4] epoch 10 val_loss=0.0734 best=0.0687 (0.9s)
[kd_feedback s0 r5] generator sanity (validation): acc clean=0.9883 -> under G=0.9699
[kd_feedback s0 r5] epoch 1 val_loss=0.0769 best=0.0769 (1.5s)
[kd_feedback s0 r5] epoch 10 val_loss=0.0769 best=0.0754 (0.9s)
  evaluated kd_base: clean=0.7888, worst@0.05=0.4458, worst@0.10=0.1876
  evaluated kd_feedback_r1: clean=0.7864, worst@0.05=0.4965, worst@0.10=0.07

[kd_feedback s1 r2] generator sanity (validation): acc clean=0.9894 -> under G=0.8336
[kd_feedback s1 r2] epoch 1 val_loss=0.0936 best=0.0936 (1.5s)
[kd_feedback s1 r2] epoch 10 val_loss=0.0850 best=0.0677 (0.9s)
[kd_feedback s1 r3] generator sanity (validation): acc clean=0.9881 -> under G=0.8365
[kd_feedback s1 r3] epoch 1 val_loss=0.0873 best=0.0873 (1.5s)
[kd_feedback s1 r3] epoch 10 val_loss=0.0740 best=0.0691 (0.9s)
[kd_feedback s1 r4] generator sanity (validation): acc clean=0.9879 -> under G=0.8626
[kd_feedback s1 r4] epoch 1 val_loss=0.1019 best=0.1019 (1.5s)
[kd_feedback s1 r4] epoch 10 val_loss=0.0951 best=0.0794 (0.9s)
[kd_feedback s1 r5] generator sanity (validation): acc clean=0.9861 -> under G=0.9753
[kd_feedback s1 r5] epoch 1 val_loss=0.0878 best=0.0878 (1.5s)
[kd_feedback s1 r5] epoch 10 val_loss=0.0941 best=0.0843 (0.9s)
  evaluated kd_base: clean=0.7842, worst@0.05=0.4146, worst@0.10=0.2583
  evaluated kd_feedback_r1: clean=0.7759, worst@0.05=0.5259, worst@0.10=0.24

In [2]:
# Stage 4b - AdvGAN used as an ATTACK: fresh generators trained against each final model, sweeping generator
# capacity, lambda_cls and number of steps (Reviewer 2, point 6). Reports accuracy under each generator, the worst
# generator configuration, and the white-box worst case from Stage 4a for comparison.
# Integrity checks are built in: target models must reproduce Stage 4a's clean accuracy, and every generated perturbation
# must stay within eps and leave the one-hot columns untouched. Resumable. Start with SEEDS = [0], then extend.

# %% [0] Setup
from google.colab import drive
drive.mount('/content/drive')
import os, json, random, time
import numpy as np, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

BASE = "/content/drive/MyDrive/A3IDPS_COLAB"
DATA = f"{BASE}/data_processed/nslkdd_v2"; MOD = f"{BASE}/models/nsl_v2"; RES = f"{BASE}/results/nsl_v2"
d = np.load(f"{DATA}/nsl_v2_arrays.npz")
Xtr, Xte = d["Xtr"].astype("float32"), d["Xte"].astype("float32")
ytr, yte = d["ytr"].astype("int32"), d["yte"].astype("int32")
names = json.load(open(f"{DATA}/feature_names.json"))
D, NCLS, BATCH = Xtr.shape[1], 5, 512
MASK_NUM = np.array([0. if n.startswith(("protocol_type_", "service_", "flag_")) else 1. for n in names], "float32")
FROZEN = MASK_NUM == 0

SEEDS = [0, 1, 2, 3, 4]                       # first run: seed 0 only; extend to [0, 1, 2, 3, 4] once the output looks right
EPS_ATTACK = [0.05, 0.10]
TARGETS = {"undefended": "s4_undefended", "fgsm_at": "s4_fgsm_at", "pgd_at": "s4_pgd_at", "kd_feedback_r5": "s4_kd_feedback_r5"}
CONFIGS = {
    "default":   dict(width=256,  lam_cls=20., steps=1200),      # the paper's setting
    "width64":   dict(width=64,   lam_cls=20., steps=1200),
    "width1024": dict(width=1024, lam_cls=20., steps=1200),
    "lam5":      dict(width=256,  lam_cls=5.,  steps=1200),
    "lam80":     dict(width=256,  lam_cls=80., steps=1200),
    "steps400":  dict(width=256,  lam_cls=20., steps=400),
    "steps4800": dict(width=256,  lam_cls=20., steps=4800),
    "strong":    dict(width=1024, lam_cls=80., steps=4800),
}
LAM_L1, LAM_GAN, KAPPA = 0.5, 1.0, 1.0
FAST = True                        # set False to fall back to eager steps if tf.function tracing errors occur
def maybe_tf(fn): return tf.function(fn, reduce_retracing=True) if FAST else fn
def set_seed(s): random.seed(s); np.random.seed(s); tf.keras.utils.set_random_seed(s)
def predict(model, X): return np.argmax(model.predict(X, batch_size=4096, verbose=0), axis=1)

# %% [1] Generator (same objective as the defence generator) with configurable capacity / lambda_cls / steps
def build_generator(width):
    inp = keras.Input(shape=(D,))
    x = layers.Dense(width, activation="relu")(inp); x = layers.Dense(width, activation="relu")(x)
    return keras.Model(inp, layers.Dense(D)(x))

def build_disc():
    inp = keras.Input(shape=(D,))
    x = layers.Dense(256)(inp); x = layers.LeakyReLU(0.2)(x)
    x = layers.Dense(256)(x);   x = layers.LeakyReLU(0.2)(x)
    return keras.Model(inp, layers.Dense(1)(x))

def apply_G(G, X, eps, bs=8192):
    E = eps * MASK_NUM
    return np.vstack([np.clip(X[i:i+bs] + E * np.tanh(G(tf.constant(X[i:i+bs]), training=False).numpy()), 0., 1.)
                      for i in range(0, len(X), bs)]).astype("float32")

def train_generator(clf, eps, seed, width, lam_cls, steps):
    set_seed(seed)
    G, Dn = build_generator(width), build_disc()
    g_opt = keras.optimizers.Adam(1e-3, beta_1=0.5); g_opt.build(G.trainable_variables)
    d_opt = keras.optimizers.Adam(1e-4, beta_1=0.5); d_opt.build(Dn.trainable_variables)
    E = tf.constant(eps * MASK_NUM)
    bce = lambda labels, logits: tf.reduce_mean(tf.nn.sigmoid_cross_entropy_with_logits(labels=labels, logits=logits))
    def gen(x): return tf.clip_by_value(x + E * tf.tanh(G(x, training=True)), 0., 1.)

    @maybe_tf
    def step(x, y):
        with tf.GradientTape() as td:
            xa = gen(x)
            ld = bce(tf.ones([tf.shape(x)[0], 1]), Dn(x, training=True)) + bce(tf.zeros([tf.shape(x)[0], 1]), Dn(xa, training=True))
        d_opt.apply_gradients(zip(td.gradient(ld, Dn.trainable_variables), Dn.trainable_variables))
        with tf.GradientTape() as tg:
            xa = gen(x); z = clf(xa, training=False); oh = tf.one_hot(y, NCLS)
            margin = tf.reduce_sum(z * oh, 1) - tf.reduce_max(z - 1e9 * oh, 1)
            lg = (lam_cls * tf.reduce_mean(tf.nn.relu(margin + KAPPA))
                  + LAM_GAN * bce(tf.ones([tf.shape(x)[0], 1]), Dn(xa, training=False))
                  + LAM_L1 * tf.reduce_mean(tf.abs(xa - x)))
        g_opt.apply_gradients(zip(tg.gradient(lg, G.trainable_variables), G.trainable_variables))
        return lg

    rng = np.random.RandomState(seed)
    for _ in range(steps):
        idx = rng.randint(0, len(Xtr), BATCH)
        step(tf.constant(Xtr[idx]), tf.constant(ytr[idx]))
    return G

# %% [2] Run: fresh generator per (model, seed, eps, configuration); integrity checks on every one
OUT = f"{RES}/stage4b_advgan_attack.json"
res = json.load(open(OUT)) if os.path.exists(OUT) else {}
s4a = {s: json.load(open(f"{RES}/stage4a_seed{s}.json")) for s in SEEDS}

for seed in SEEDS:
    for tname, prefix in TARGETS.items():
        model = keras.models.load_model(f"{MOD}/{prefix}_seed{seed}.keras", compile=False)
        clean = float((predict(model, Xte) == yte).mean())
        ref_clean = s4a[seed][tname]["clean"]["acc"]
        assert abs(clean - ref_clean) < 1e-6, f"{tname} seed {seed}: clean acc {clean} != Stage 4a {ref_clean} (wrong model file?)"
        print(f"\n##### seed {seed} | {tname} | clean acc {clean:.4f} (matches Stage 4a)")
        for eps in EPS_ATTACK:
            for cname, cfg in CONFIGS.items():
                key = f"{tname}|seed{seed}|eps{eps}|{cname}"
                if key in res: continue
                t0 = time.time()
                G = train_generator(model, eps, 1000 + seed, **cfg)
                Xa = apply_G(G, Xte, eps)
                delta = Xa - Xte
                assert np.abs(delta).max() <= eps + 1e-6, "perturbation exceeds eps"
                assert np.abs(delta[:, FROZEN]).max() == 0.0, "one-hot columns were modified"
                acc = float((predict(model, Xa) == yte).mean())
                res[key] = {"acc_under_generator": acc, "mean_abs_delta": float(np.abs(delta).mean()), "config": cfg}
                json.dump(res, open(OUT, "w"), indent=1)
                print(f"  eps={eps} {cname:<10} acc under generator={acc:.4f} | mean|delta|={np.abs(delta).mean():.5f} ({time.time()-t0:.0f}s)")
        keras.backend.clear_session()

# %% [3] Summary (run again after more seeds finish)
res = json.load(open(OUT)); seeds_done = sorted({int(k.split("|")[1][4:]) for k in res})
def ms(v): a = np.array(v); return f"{100*a.mean():5.1f}+-{100*(a.std(ddof=1) if len(a) > 1 else 0):4.1f}"
print(f"\nseeds with results: {seeds_done}   (accuracy in %, lower = stronger attack; white-box = Stage 4a worst case)")
for eps in EPS_ATTACK:
    print(f"\n=== eps = {eps}: AdvGAN attack accuracy per model ===")
    print(f"{'model':<18}{'default cfg':>14}{'worst cfg':>14}{'white-box':>14}   worst config (per seed)")
    for tname in TARGETS:
        dflt, worst, wb, who = [], [], [], []
        for s in seeds_done:
            accs = {c: res[f"{tname}|seed{s}|eps{eps}|{c}"]["acc_under_generator"] for c in CONFIGS if f"{tname}|seed{s}|eps{eps}|{c}" in res}
            if len(accs) < len(CONFIGS): continue
            dflt.append(accs["default"]); c_min = min(accs, key=accs.get); worst.append(accs[c_min]); who.append(c_min)
            s4 = json.load(open(f"{RES}/stage4a_seed{s}.json"))[tname]["numeric_only"][str(eps)]["worst"]; wb.append(s4)
        if dflt:
            flag = "  <-- generator STRONGER than white-box: check for a bug" if np.mean(worst) < np.mean(wb) - 0.02 else ""
            print(f"{tname:<18}{ms(dflt):>14}{ms(worst):>14}{ms(wb):>14}   {who}{flag}")
    print(f"\n--- per-configuration mean accuracy at eps={eps} ---")
    print(f"{'config':<12}" + "".join(f"{t:>16}" for t in TARGETS))
    for c in CONFIGS:
        row = []
        for tname in TARGETS:
            v = [res[f"{tname}|seed{s}|eps{eps}|{c}"]["acc_under_generator"] for s in seeds_done if f"{tname}|seed{s}|eps{eps}|{c}" in res]
            row.append(ms(v) if v else "-")
        print(f"{c:<12}" + "".join(f"{r:>16}" for r in row))

Exception ignored in: <function NpzFile.__del__ at 0x7aefc0b158a0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/numpy/lib/_npyio_impl.py", line 228, in __del__
    self.close()
  File "/usr/local/lib/python3.13/dist-packages/numpy/lib/_npyio_impl.py", line 223, in close
    self.fid.close()
OSError: [Errno 107] Transport endpoint is not connected


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

##### seed 0 | undefended | clean acc 0.7930 (matches Stage 4a)



##### seed 0 | fgsm_at | clean acc 0.7941 (matches Stage 4a)



##### seed 0 | pgd_at | clean acc 0.7938 (matches Stage 4a)

##### seed 0 | kd_feedback_r5 | clean acc 0.7799 (matches Stage 4a)

##### seed 1 | undefended | clean acc 0.7909 (matches Stage 4a)
  eps=0.05 default    acc under generator=0.6133 | mean|delta|=0.00795 (10s)
  eps=0.05 width64    acc under generator=0.6207 | mean|delta|=0.00782 (7s)
  eps=0.05 width1024  acc under generator=0.6120 | mean|delta|=0.00689 (7s)
  eps=0.05 lam5       acc under generator=0.6157 | mean|delta|=0.00553 (7s)
  eps=0.05 lam80      acc under generator=0.6171 | mean|delta|=0.00835 (7s)
  eps=0.05 steps400   acc under generator=0.6214 | mean|delta|=0.00847 (3s)
  eps=0.05 steps4800  acc under generator=0.6038 | mean|delta|=0.00753 (26s)
  eps=0.05 strong     acc under generator=0.5706 | mean|delta|=0.00751 (25s)
  eps=0.1 default    acc under generator=0.3811 | mean|delta|=0.01067 (7s)
  eps=0.1 width64    acc under generator=0.4399 | mean|delta|=0.01227 (7s)
  eps=0.1 width1024  acc under generator=0.3

In [3]:
# ART APGD cross-check, all-features protocol - UPDATED to include the A3IDPS model.
# Identical to nsl_art_crosscheck.py except for the MODELS dict. Already-finished entries are skipped,
# so this only adds the four new (a3idps, eps, protocol) rows to the same results file.

# %% [0] Install, load, configuration
import subprocess
subprocess.run(["pip", "install", "-q", "adversarial-robustness-toolbox"], check=True)

from google.colab import drive
drive.mount('/content/drive')
import json, os, time, numpy as np, tensorflow as tf
from tensorflow import keras

BASE = "/content/drive/MyDrive/A3IDPS_COLAB"
RES = f"{BASE}/results/nsl_v2"; MOD = f"{BASE}/models/nsl_v2"
d = np.load(f"{BASE}/data_processed/nslkdd_v2/nsl_v2_arrays.npz")
Xte, yte = d["Xte"].astype("float32"), d["yte"].astype("int32")
names = json.load(open(f"{BASE}/data_processed/nslkdd_v2/feature_names.json"))
MASK_NUM = np.array([0. if n.startswith(("protocol_type_", "service_", "flag_")) else 1. for n in names], "float32")
MASK_ALL = np.ones_like(MASK_NUM)
D, NCLS = Xte.shape[1], 5

sel = json.load(open(f"{RES}/stage2_selected_seed0.json"))
MODELS = {
    "undefended": f"{MOD}/s2_none_seed0.keras",
    f"fgsm_eps{sel['fgsm']}": f"{MOD}/s2_fgsm_eps{sel['fgsm']}_seed0.keras",
    f"pgd_eps{sel['pgd']}": f"{MOD}/s2_pgd_eps{sel['pgd']}_seed0.keras",
    "a3idps_kd_feedback_r5": f"{MOD}/s4_kd_feedback_r5_seed0.keras",   # NEW
}
EPS_LIST = (0.05, 0.10)
N = 300
sub = np.array(json.load(open(f"{RES}/art_crosscheck_subset_idx.json")))   # same 300 rows as before
X, y = Xte[sub], yte[sub]
Y1 = keras.utils.to_categorical(y, NCLS)
OUT = f"{RES}/art_crosscheck_results.json"
results = json.load(open(OUT)) if os.path.exists(OUT) else {}

# %% [1] Our attack (unchanged)
def loss_fn(z, yy, margin):
    if margin:
        oh = tf.one_hot(yy, NCLS)
        return tf.reduce_max(z - 1e9 * oh, axis=1) - tf.reduce_sum(z * oh, axis=1)
    return tf.nn.sparse_softmax_cross_entropy_with_logits(labels=yy, logits=z)

def our_pgd(model, X, y, eps, mask, margin, steps=20, restarts=3):
    E = tf.constant(eps * mask); x0 = tf.constant(X); yb = tf.constant(y)
    best_x, best_l = x0, tf.fill(tf.shape(yb), -1e30)
    for r in range(restarts):
        x = x0 if r == 0 else tf.clip_by_value(x0 + tf.random.uniform(tf.shape(x0), -1., 1.) * E, 0., 1.)
        for s_ in range(steps):
            alpha = (1.0 if s_ < steps // 2 else 0.25) * E / 4.0
            with tf.GradientTape() as t:
                t.watch(x); s = tf.reduce_sum(loss_fn(model(x, training=False), yb, margin))
            x = tf.clip_by_value(tf.clip_by_value(x + alpha * tf.sign(t.gradient(s, x)), x0 - E, x0 + E), 0., 1.)
            l_new = loss_fn(model(x, training=False), yb, margin)
            upd = l_new > best_l
            best_x = tf.where(upd[:, None], x, best_x); best_l = tf.where(upd, l_new, best_l)
    return best_x.numpy()

def correct(model, Xa): return np.argmax(model.predict(Xa, verbose=0), 1) == y

# %% [2] ART wrapper
import art
from art.estimators.classification import TensorFlowV2Classifier
from art.attacks.evasion import AutoProjectedGradientDescent
print("ART version:", art.__version__)

def make_art(model):
    return TensorFlowV2Classifier(model=model, nb_classes=NCLS, input_shape=(D,),
                                  loss_object=tf.keras.losses.CategoricalCrossentropy(from_logits=True), clip_values=(0.0, 1.0))

def make_apgd(clf, eps, loss_type, n_init=2, iters=100):
    return AutoProjectedGradientDescent(estimator=clf, norm=np.inf, eps=eps, eps_step=eps / 4, max_iter=iters, targeted=False,
                                        nb_random_init=n_init, batch_size=150, loss_type=loss_type, verbose=False)

# %% [3] Run (all-features protocol only; numeric-only is handled by the companion _numeric_patch script)
for name, path in MODELS.items():
    model = keras.models.load_model(path, compile=False); clf = make_art(model)
    print(f"\n##### {name}  (clean acc on subset: {correct(model, X).mean():.4f})")
    for eps in EPS_LIST:
        key = f"{name}|{eps}|all_features"
        if key in results: continue
        ce = correct(model, our_pgd(model, X, y, eps, MASK_ALL, False)); mg = correct(model, our_pgd(model, X, y, eps, MASK_ALL, True))
        row = {"ours_pgd_ce": float(ce.mean()), "ours_pgd_margin": float(mg.mean()), "ours_worst": float((ce & mg).mean())}
        art_c = {}
        for lt in ("cross_entropy", "difference_logits_ratio"):
            t0 = time.time()
            Xa = make_apgd(clf, eps, lt).generate(x=X, y=Y1)
            art_c[lt] = correct(model, Xa); row[f"art_{lt}"] = float(art_c[lt].mean())
            row[f"art_{lt}_max_delta"] = float(np.abs(Xa - X).max())
            print(f"  {key} ART {lt}: acc={art_c[lt].mean():.4f} ({time.time()-t0:.0f}s)")
        row["art_worst"] = float((art_c["cross_entropy"] & art_c["difference_logits_ratio"]).mean())
        row["combined_worst"] = float((ce & mg & art_c["cross_entropy"] & art_c["difference_logits_ratio"]).mean())
        row["art_stronger_by"] = row["ours_worst"] - row["art_worst"]
        results[key] = row; json.dump(results, open(OUT, "w"), indent=1)

# %% [4] Summary
print(f"\n{'model|eps|protocol':<38}{'ours worst':>11}{'ART worst':>11}{'combined':>10}  note")
for key, r in results.items():
    if not key.endswith("|all_features"): continue
    note = "ART clearly stronger - our attack may be weak here" if r["art_stronger_by"] > 0.04 else ""
    print(f"{key:<38}{r['ours_worst']:>11.4f}{r['art_worst']:>11.4f}{r['combined_worst']:>10.4f}  {note}")
print("saved:", OUT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ART version: 1.20.1

##### undefended  (clean acc on subset: 0.8000)

##### fgsm_eps0.05  (clean acc on subset: 0.7767)

##### pgd_eps0.1  (clean acc on subset: 0.7800)

##### a3idps_kd_feedback_r5  (clean acc on subset: 0.7967)
  a3idps_kd_feedback_r5|0.05|all_features ART cross_entropy: acc=0.2767 (10s)


  a3idps_kd_feedback_r5|0.05|all_features ART difference_logits_ratio: acc=0.3100 (113s)


  a3idps_kd_feedback_r5|0.1|all_features ART cross_entropy: acc=0.1900 (7s)
  a3idps_kd_feedback_r5|0.1|all_features ART difference_logits_ratio: acc=0.0333 (87s)

model|eps|protocol                     ours worst  ART worst  combined  note
undefended|0.05|all_features               0.0700     0.1667    0.0633  
undefended|0.1|all_features                0.0000     0.0000    0.0000  
fgsm_eps0.05|0.05|all_features             0.5767     0.5633    0.5633  
fgsm_eps0.05|0.1|all_features              0.1100     0.1100    0.1067  
pgd_eps0.1|0.05|all_features               0.6467     0.6433    0.6433  
pgd_eps0.1|0.1|all_features                0.2067     0.2100    0.2033  
a3idps_kd_feedback_r5|0.05|all_features     0.2133     0.2567    0.2133  
a3idps_kd_feedback_r5|0.1|all_features     0.0000     0.0333    0.0000  
saved: /content/drive/MyDrive/A3IDPS_COLAB/results/nsl_v2/art_crosscheck_results.json


In [4]:
# ART APGD cross-check, numeric-only protocol via the snapping wrapper - UPDATED to include the A3IDPS model.
# Identical to nsl_art_numeric_patch.py except for the MODELS dict.

# %% [0] Setup
import subprocess
subprocess.run(["pip", "install", "-q", "adversarial-robustness-toolbox"], check=True)

from google.colab import drive
drive.mount('/content/drive')
import json, os, time, numpy as np, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

BASE = "/content/drive/MyDrive/A3IDPS_COLAB"
RES = f"{BASE}/results/nsl_v2"; MOD = f"{BASE}/models/nsl_v2"
d = np.load(f"{BASE}/data_processed/nslkdd_v2/nsl_v2_arrays.npz")
Xte, yte = d["Xte"].astype("float32"), d["yte"].astype("int32")
names = json.load(open(f"{BASE}/data_processed/nslkdd_v2/feature_names.json"))
MASK_NUM = np.array([0. if n.startswith(("protocol_type_", "service_", "flag_")) else 1. for n in names], "float32")
D, NCLS = Xte.shape[1], 5
FROZEN = MASK_NUM == 0

sel = json.load(open(f"{RES}/stage2_selected_seed0.json"))
MODELS = {"undefended": f"{MOD}/s2_none_seed0.keras",
          f"fgsm_eps{sel['fgsm']}": f"{MOD}/s2_fgsm_eps{sel['fgsm']}_seed0.keras",
          f"pgd_eps{sel['pgd']}": f"{MOD}/s2_pgd_eps{sel['pgd']}_seed0.keras",
          "a3idps_kd_feedback_r5": f"{MOD}/s4_kd_feedback_r5_seed0.keras"}   # NEW
EPS_LIST = (0.05, 0.10)
sub = np.array(json.load(open(f"{RES}/art_crosscheck_subset_idx.json")))
X, y = Xte[sub], yte[sub]
Y1 = keras.utils.to_categorical(y, NCLS)
assert np.abs(X[:, FROZEN] - np.round(X[:, FROZEN])).max() == 0, "one-hot columns are not exactly 0/1 - snapping would be invalid"
OUT = f"{RES}/art_crosscheck_results.json"
results = json.load(open(OUT))

# %% [1] Snapping wrapper, our attack, ART wrapper (unchanged)
M_T = tf.constant(MASK_NUM)
def snapped(model):
    inp = keras.Input(shape=(D,))
    x = layers.Lambda(lambda t: M_T * t + (1. - M_T) * tf.round(t))(inp)
    return keras.Model(inp, model(x))

def loss_fn(z, yy, margin):
    if margin:
        oh = tf.one_hot(yy, NCLS)
        return tf.reduce_max(z - 1e9 * oh, axis=1) - tf.reduce_sum(z * oh, axis=1)
    return tf.nn.sparse_softmax_cross_entropy_with_logits(labels=yy, logits=z)

def our_pgd(model, X, y, eps, margin, steps=20, restarts=3):
    E = tf.constant(eps * MASK_NUM); x0 = tf.constant(X); yb = tf.constant(y)
    best_x, best_l = x0, tf.fill(tf.shape(yb), -1e30)
    for r in range(restarts):
        x = x0 if r == 0 else tf.clip_by_value(x0 + tf.random.uniform(tf.shape(x0), -1., 1.) * E, 0., 1.)
        for s_ in range(steps):
            alpha = (1.0 if s_ < steps // 2 else 0.25) * E / 4.0
            with tf.GradientTape() as t:
                t.watch(x); s = tf.reduce_sum(loss_fn(model(x, training=False), yb, margin))
            x = tf.clip_by_value(tf.clip_by_value(x + alpha * tf.sign(t.gradient(s, x)), x0 - E, x0 + E), 0., 1.)
            l_new = loss_fn(model(x, training=False), yb, margin)
            upd = l_new > best_l
            best_x, best_l = tf.where(upd[:, None], x, best_x), tf.where(upd, l_new, best_l)
    return best_x.numpy()

import art
from art.estimators.classification import TensorFlowV2Classifier
from art.attacks.evasion import AutoProjectedGradientDescent
def make_apgd(clf, eps, lt):
    return AutoProjectedGradientDescent(estimator=clf, norm=np.inf, eps=eps, eps_step=eps / 4, max_iter=100, targeted=False,
                                        nb_random_init=2, batch_size=150, loss_type=lt, verbose=False)
correct = lambda m, Xa: np.argmax(m.predict(Xa, verbose=0), 1) == y

# %% [2] Run
for name, path in MODELS.items():
    model = keras.models.load_model(path, compile=False); wm = snapped(model)
    assert (np.argmax(wm.predict(X, verbose=0), 1) == np.argmax(model.predict(X, verbose=0), 1)).all()
    clf = TensorFlowV2Classifier(model=wm, nb_classes=NCLS, input_shape=(D,),
                                 loss_object=tf.keras.losses.CategoricalCrossentropy(from_logits=True), clip_values=(0.0, 1.0))
    for eps in EPS_LIST:
        key = f"{name}|{eps}|numeric_only"
        if key in results: continue
        ce = correct(model, our_pgd(model, X, y, eps, False)); mg = correct(model, our_pgd(model, X, y, eps, True))
        row = {"ours_pgd_ce": float(ce.mean()), "ours_pgd_margin": float(mg.mean()), "ours_worst": float((ce & mg).mean()),
               "method": "ART on snapped wrapper (one-hot columns re-rounded)"}
        art_c = {}
        for lt in ("cross_entropy", "difference_logits_ratio"):
            t0 = time.time(); Xa = make_apgd(clf, eps, lt).generate(x=X, y=Y1)
            art_c[lt] = correct(wm, Xa); row[f"art_{lt}"] = float(art_c[lt].mean())
            snap = MASK_NUM * Xa + (1 - MASK_NUM) * np.round(Xa)
            row[f"art_{lt}_effective_frozen_change"] = float(np.abs((snap - X)[:, FROZEN]).max())
            print(f"  {key} ART {lt}: acc={art_c[lt].mean():.4f} | effective change on frozen cols={row[f'art_{lt}_effective_frozen_change']} ({time.time()-t0:.0f}s)")
        row["art_worst"] = float((art_c["cross_entropy"] & art_c["difference_logits_ratio"]).mean())
        row["combined_worst"] = float((ce & mg & art_c["cross_entropy"] & art_c["difference_logits_ratio"]).mean())
        row["art_stronger_by"] = row["ours_worst"] - row["art_worst"]
        results[key] = row; json.dump(results, open(OUT, "w"), indent=1)

print(f"\n{'model|eps|protocol':<38}{'ours worst':>11}{'ART worst':>11}{'combined':>10}  note")
for key, r in results.items():
    if not key.endswith("|numeric_only"): continue
    note = "ART clearly stronger - our attack may be weak here" if r["art_stronger_by"] > 0.04 else ""
    print(f"{key:<38}{r['ours_worst']:>11.4f}{r['art_worst']:>11.4f}{r['combined_worst']:>10.4f}  {note}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  a3idps_kd_feedback_r5|0.05|numeric_only ART cross_entropy: acc=0.5933 | effective change on frozen cols=0.0 (15s)
  a3idps_kd_feedback_r5|0.05|numeric_only ART difference_logits_ratio: acc=0.6100 | effective change on frozen cols=0.0 (146s)
  a3idps_kd_feedback_r5|0.1|numeric_only ART cross_entropy: acc=0.3933 | effective change on frozen cols=0.0 (11s)
  a3idps_kd_feedback_r5|0.1|numeric_only ART difference_logits_ratio: acc=0.4600 | effective change on frozen cols=0.0 (126s)

model|eps|protocol                     ours worst  ART worst  combined  note
undefended|0.05|numeric_only               0.4367     0.4633    0.4367  
undefended|0.1|numeric_only                0.1067     0.1067    0.1033  
fgsm_eps0.05|0.05|numeric_only             0.7500     0.7500    0.7500  
fgsm_eps0.05|0.1|numeric_only              0.7167     0.7133    0.7133  
pgd_eps0.1|0.05|n

In [5]:
# CICIDS2017 v2 - Stage 1: corrected preprocessing.
# Fixes vs the old pipeline: features are actually normalised (old pipeline had values up to 2.07e9, 54.8% outside [0,1]);
# a stratified row-level split so BENIGN is ~80.3% in every split and all 5 classes appear in test (reviewers' complaint);
# exact-duplicate removal before splitting (CICIDS2017 has many near-identical rows per attack session - Engelen et al. [40]);
# a second, blocked/contiguous split kept as a leakage robustness check, reported alongside the primary result, not instead of it.
# Paste into Colab. Needs the raw CICIDS2017 CSVs already available at data_raw/CICIDS2017 (same as your original pipeline).

# %% [0] Setup
from google.colab import drive
drive.mount('/content/drive')
import os, glob, json, random, hashlib
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import joblib

BASE = "/content/drive/MyDrive/A3IDPS_COLAB"
RAW  = f"{BASE}/data_raw/CICIDS2017"
OUT  = f"{BASE}/data_processed/cicids_v2"
os.makedirs(OUT, exist_ok=True)

SPLIT_SEED = 42
CLASS_NAMES = ["BENIGN", "Brute Force", "DoS/DDoS", "PortScan", "Other"]
DROP_COLS = {"Flow ID", "Src IP", "Source IP", "Dst IP", "Destination IP", "Timestamp", "__source_file__"}

def map_5class(label):
    if label == "BENIGN": return 0
    if label in ("FTP-Patator", "SSH-Patator"): return 1
    if label in ("DoS Hulk", "DoS GoldenEye", "DoS slowloris", "DoS Slowhttptest", "DDoS"): return 2
    if label == "PortScan": return 3
    return 4   # Bot, Web Attack *, Infiltration, Heartbleed

# %% [1] Load all raw CSVs, clean, label -- NO leakage columns, NO scaling yet (fit must happen on train only)
csvs = sorted(glob.glob(RAW + "/**/*.csv", recursive=True))
print("raw CSVs found:", len(csvs))
assert csvs, f"no CSVs under {RAW}"

chunks = []
for path in csvs:
    print("reading:", os.path.basename(path))
    try:
        df = pd.read_csv(path, low_memory=False, encoding_errors="ignore")
    except TypeError:
        df = pd.read_csv(path, low_memory=False, encoding="latin1")
    df.columns = [c.strip() for c in df.columns]
    if "Label" not in df.columns:
        print("  skipped (no Label column)"); continue
    df = df.drop(columns=[c for c in df.columns if c in DROP_COLS], errors="ignore")
    df = df.replace([np.inf, -np.inf], np.nan).dropna()
    df["label_5class"] = df["Label"].map(map_5class).astype("int8")
    chunks.append(df.drop(columns=["Label"]))

full = pd.concat(chunks, ignore_index=True)
del chunks
print("total rows after cleaning:", len(full))
print("class counts:", full["label_5class"].value_counts().sort_index().to_dict())

feature_cols = [c for c in full.columns if c != "label_5class"]
for c in feature_cols:
    full[c] = pd.to_numeric(full[c], errors="coerce")
before = len(full)
full = full.dropna()
print(f"dropped {before - len(full)} rows with non-numeric feature values")

# %% [2] Remove exact duplicate feature rows (keep first occurrence; report how many were dropped, per class)
before = len(full)
dup_mask = full.duplicated(subset=feature_cols, keep="first")
dup_by_class = full.loc[dup_mask, "label_5class"].value_counts().sort_index().to_dict()
full = full.loc[~dup_mask].reset_index(drop=True)
print(f"removed {before - len(full)} exact-duplicate rows ({100*(before-len(full))/before:.1f}%); "
      f"by class: {dup_by_class}")
print("class counts after dedup:", full["label_5class"].value_counts().sort_index().to_dict())

y = full["label_5class"].values.astype("int32")
X_raw = full[feature_cols].values.astype("float64")   # float64 during log-transform to avoid overflow on huge raw values
del full

# %% [3] PRIMARY split: stratified, row-level, 70/15/15 (answers the reviewers' "describe the actual partitioning")
idx = np.arange(len(y))
idx_tr, idx_tmp, y_tr, y_tmp = train_test_split(idx, y, test_size=0.30, random_state=SPLIT_SEED, stratify=y)
idx_va, idx_te, y_va, y_te = train_test_split(idx_tmp, y_tmp, test_size=0.50, random_state=SPLIT_SEED, stratify=y_tmp)
print("primary split sizes:", len(idx_tr), len(idx_va), len(idx_te))
for name, yy in (("train", y_tr), ("val", y_va), ("test", y_te)):
    props = {CLASS_NAMES[c]: f"{100*np.mean(yy==c):.2f}%" for c in range(5)}
    print(f"  {name}: {props}")

# %% [4] Log-transform heavy-tailed features, then min-max scale, fit on TRAIN only
# log1p on the shifted-nonnegative feature (features can be negative here only due to upstream tool artefacts in a few columns)
shift = np.minimum(X_raw[idx_tr].min(axis=0), 0.0)          # per-feature: 0 if already >=0, else the negative min on train
X_shifted = X_raw - shift
X_log = np.log1p(X_shifted)
assert np.isfinite(X_log).all(), "non-finite values after log-transform - check DROP_COLS / raw data"

scaler = MinMaxScaler().fit(X_log[idx_tr])
X_all = scaler.transform(X_log).astype("float32")
frac_clip_va = float(((X_all[idx_va] < 0) | (X_all[idx_va] > 1)).mean())
frac_clip_te = float(((X_all[idx_te] < 0) | (X_all[idx_te] > 1)).mean())
print(f"share of val values outside [0,1] before clipping: {frac_clip_va:.6f}")
print(f"share of test values outside [0,1] before clipping: {frac_clip_te:.6f}")
X_all = np.clip(X_all, 0.0, 1.0)
print("post-scaling range check: min", float(X_all.min()), "max", float(X_all.max()))
assert 0.0 <= X_all.min() and X_all.max() <= 1.0

Xtr, Xva, Xte = X_all[idx_tr], X_all[idx_va], X_all[idx_te]

def hashes(X): return {hashlib.md5(r.tobytes()).hexdigest() for r in np.round(X, 6)}
overlap_te = len(hashes(Xte) & (hashes(Xtr) | hashes(Xva)))
print("exact test rows (post-scaling) also present in train/val (informational, dedup was pre-split):", overlap_te, "of", len(Xte))

np.savez_compressed(f"{OUT}/cicids_v2_primary.npz", Xtr=Xtr, ytr=y_tr, Xva=Xva, yva=y_va, Xte=Xte, yte=y_te)
joblib.dump(scaler, f"{OUT}/log_minmax_scaler.joblib")
np.save(f"{OUT}/log_shift.npy", shift)
json.dump(feature_cols, open(f"{OUT}/feature_names.json", "w"), indent=1)
print(f"input dim: {len(feature_cols)}")

# %% [5] SECONDARY split: blocked/contiguous per class, on ORIGINAL file order - the leakage robustness check
# Within each class, take the rows in the order they appear in the concatenated raw files (a rough time proxy),
# split 70/15/15 in blocks with a gap discarded between train and test, so temporally adjacent (near-duplicate) flows
# cannot straddle the train/test boundary the way a random split allows.
GAP_FRAC = 0.02   # discard 2% of each class's rows at each block boundary
blocked_tr, blocked_va, blocked_te = [], [], []
for c in range(5):
    rows = np.where(y == c)[0]         # already in original file order
    n = len(rows)
    if n < 20:
        print(f"class {CLASS_NAMES[c]}: only {n} rows, too few to block-split; assigning all to test only")
        blocked_te.extend(rows.tolist()); continue
    gap = max(1, int(GAP_FRAC * n))
    t_end = int(0.70 * n)
    v_end = t_end + gap + int(0.15 * n)
    blocked_tr.extend(rows[:t_end].tolist())
    blocked_va.extend(rows[t_end + gap: v_end].tolist())
    blocked_te.extend(rows[v_end + gap:].tolist())

blocked_tr, blocked_va, blocked_te = map(np.array, (blocked_tr, blocked_va, blocked_te))
print("\nblocked split sizes:", len(blocked_tr), len(blocked_va), len(blocked_te))
for name, ii in (("train", blocked_tr), ("val", blocked_va), ("test", blocked_te)):
    yy = y[ii]
    props = {CLASS_NAMES[c]: f"{100*np.mean(yy==c):.2f}%" for c in range(5)} if len(yy) else {}
    print(f"  {name}: n={len(ii)} {props}")

# Fit a SEPARATE scaler for the blocked split (its train rows differ from the primary split's train rows)
shift_b = np.minimum(X_raw[blocked_tr].min(axis=0), 0.0)
X_log_b = np.log1p(X_raw - shift_b)
scaler_b = MinMaxScaler().fit(X_log_b[blocked_tr])
X_all_b = np.clip(scaler_b.transform(X_log_b), 0.0, 1.0).astype("float32")

np.savez_compressed(f"{OUT}/cicids_v2_blocked.npz",
                    Xtr=X_all_b[blocked_tr], ytr=y[blocked_tr],
                    Xva=X_all_b[blocked_va], yva=y[blocked_va],
                    Xte=X_all_b[blocked_te], yte=y[blocked_te])
joblib.dump(scaler_b, f"{OUT}/log_minmax_scaler_blocked.joblib")
np.save(f"{OUT}/log_shift_blocked.npy", shift_b)

print("\nSaved:")
print(" ", f"{OUT}/cicids_v2_primary.npz", " (main protocol: stratified row-level)")
print(" ", f"{OUT}/cicids_v2_blocked.npz", " (leakage robustness check: blocked/contiguous)")
print(" ", f"{OUT}/feature_names.json")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
raw CSVs found: 8
reading: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
reading: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
reading: Friday-WorkingHours-Morning.pcap_ISCX.csv
reading: Monday-WorkingHours.pcap_ISCX.csv
reading: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
reading: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
reading: Tuesday-WorkingHours.pcap_ISCX.csv
reading: Wednesday-workingHours.pcap_ISCX.csv
total rows after cleaning: 2827876
class counts: {0: 2271320, 1: 13832, 2: 379737, 3: 158804, 4: 4183}
dropped 0 rows with non-numeric feature values
removed 307775 exact-duplicate rows (10.9%); by class: {0: 176862, 1: 4682, 2: 58076, 3: 68110, 4: 45}
class counts after dedup: {0: 2094458, 1: 9150, 2: 321661, 3: 90694, 4: 4138}
primary split sizes: 1764070 378015 378016
  train: {'BENIGN': '83.11%', 'Brute Force

/tmp/ipykernel_2501/3230806713.py:91: RuntimeWarning: divide by zero encountered in log1p
  X_log = np.log1p(X_shifted)


AssertionError: non-finite values after log-transform - check DROP_COLS / raw data

In [9]:
# CICIDS Stage 1 - FIX for cell [4]. Run this in place of the failing cell (everything before it, i.e. cells [0]-[3],
# has already run successfully and X_raw / idx_tr / idx_va / idx_te / y_tr / y_va / y_te are still in memory).
# Fix: clip X_shifted to >= 0 BEFORE log1p. Values below the train-observed minimum (only possible in val/test, since
# shift is computed from train) are floored to 0 post-shift, i.e. treated as equal to the smallest value seen in
# training for that feature - the same kind of out-of-range handling as the [0,1] clip after MinMaxScaler, just earlier
# in the pipeline. This does not use any val/test information to choose the shift itself.

import numpy as np

shift = np.minimum(X_raw[idx_tr].min(axis=0), 0.0)
X_shifted = np.maximum(X_raw - shift, 0.0)          # <-- the fix: floor at 0 instead of letting it go negative
X_log = np.log1p(X_shifted)
assert np.isfinite(X_log).all(), "still non-finite after clipping - inspect X_raw for inf/extremely large values"

n_floored_va = int((X_raw[idx_va] - shift < 0).sum())
n_floored_te = int((X_raw[idx_te] - shift < 0).sum())
print(f"val cells floored to train-min (out of {X_raw[idx_va].size}): {n_floored_va} ({100*n_floored_va/X_raw[idx_va].size:.4f}%)")
print(f"test cells floored to train-min (out of {X_raw[idx_te].size}): {n_floored_te} ({100*n_floored_te/X_raw[idx_te].size:.4f}%)")
print("log-transform OK, X_log is finite. Continue from cell [4]'s scaler.fit(...) line onward.")

from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler().fit(X_log[idx_tr])
X_all = scaler.transform(X_log).astype("float32")
frac_clip_va = float(((X_all[idx_va] < 0) | (X_all[idx_va] > 1)).mean())
frac_clip_te = float(((X_all[idx_te] < 0) | (X_all[idx_te] > 1)).mean())
print(f"share of val values outside [0,1] before clipping: {frac_clip_va:.6f}")
print(f"share of test values outside [0,1] before clipping: {frac_clip_te:.6f}")
X_all = np.clip(X_all, 0.0, 1.0)
print("post-scaling range check: min", float(X_all.min()), "max", float(X_all.max()))
assert 0.0 <= X_all.min() and X_all.max() <= 1.0

Xtr, Xva, Xte = X_all[idx_tr], X_all[idx_va], X_all[idx_te]
print("shapes:", Xtr.shape, Xva.shape, Xte.shape)
print("\nNow re-run the ORIGINAL cell [4]'s save lines (np.savez_compressed / joblib.dump / json.dump), "
      "then continue to cell [5] (the blocked split) as originally written.")

val cells floored to train-min (out of 29485170): 0 (0.0000%)
test cells floored to train-min (out of 29485248): 3 (0.0000%)
log-transform OK, X_log is finite. Continue from cell [4]'s scaler.fit(...) line onward.
share of val values outside [0,1] before clipping: 0.000000
share of test values outside [0,1] before clipping: 0.000000
post-scaling range check: min 0.0 max 1.0
shapes: (1764070, 78) (378015, 78) (378016, 78)

Now re-run the ORIGINAL cell [4]'s save lines (np.savez_compressed / joblib.dump / json.dump), then continue to cell [5] (the blocked split) as originally written.


In [10]:
np.savez_compressed(f"{OUT}/cicids_v2_primary.npz", Xtr=Xtr, ytr=y_tr, Xva=Xva, yva=y_va, Xte=Xte, yte=y_te)
joblib.dump(scaler, f"{OUT}/log_minmax_scaler.joblib")
np.save(f"{OUT}/log_shift.npy", shift)
json.dump(feature_cols, open(f"{OUT}/feature_names.json", "w"), indent=1)
print(f"input dim: {len(feature_cols)}")



input dim: 78


In [8]:
# Diagnostic - run this BEFORE re-attempting the log-transform fix, to find out exactly what's non-finite.
# Assumes X_raw, idx_tr, feature_cols are still in memory from the earlier cells.
import numpy as np

print("X_raw dtype:", X_raw.dtype, "shape:", X_raw.shape)
print("any +inf in X_raw:", bool(np.isinf(X_raw).any()), "| count:", int(np.isinf(X_raw).sum()))
print("any nan in X_raw:", bool(np.isnan(X_raw).any()), "| count:", int(np.isnan(X_raw).sum()))
print("X_raw min:", float(np.nanmin(X_raw)), "| X_raw max:", float(np.nanmax(X_raw)))

if np.isinf(X_raw).any():
    rows_inf, cols_inf = np.where(np.isinf(X_raw))
    bad_cols = sorted(set(cols_inf.tolist()))
    print("\ncolumns containing inf:", [feature_cols[c] for c in bad_cols[:20]], "..." if len(bad_cols) > 20 else "")
    print("total inf cells:", len(rows_inf))

shift = np.minimum(X_raw[idx_tr].min(axis=0), 0.0)
X_shifted = X_raw - shift
neg_after_shift = X_shifted < 0
print("\ncells still negative after shifting by train-min (expected only in val/test, if any):", int(neg_after_shift.sum()))
print("most negative value after shift:", float(X_shifted.min()))

# Which is it: inf in X_raw itself, or just values below the train shift (which my fix handles)?
if np.isinf(X_raw).any():
    print("\n==> DIAGNOSIS: X_raw itself contains literal inf. The earlier inf->nan->dropna cleaning step did not remove all of them.")
    print("    This needs a different fix: re-check the cleaning step, or clip X_raw to a large finite value before log-transform.")
elif np.isnan(X_shifted).any() or (X_shifted < -1).any():
    print("\n==> DIAGNOSIS: some val/test values are far enough below the train minimum that shifting still leaves them < -1,")
    print("    so log1p(x) for x < -1 is NaN even before my floor-at-0 fix would apply (the floor-at-0 fix WOULD handle this -")
    print("    if you are still seeing the error, you are running the OLD unfixed cell, not the corrected one).")
else:
    print("\n==> No inf/nan found under this check - if the assertion is still firing, you are almost certainly running the old,")
    print("    unfixed version of the cell. Search your notebook for 'X_shifted = X_raw - shift' WITHOUT 'np.maximum' around it.")

X_raw dtype: float64 shape: (2520101, 78)
any +inf in X_raw: False | count: 0
any nan in X_raw: False | count: 0
X_raw min: -32212234632.0 | X_raw max: 2071000000.0

cells still negative after shifting by train-min (expected only in val/test, if any): 3
most negative value after shift: -1.0

==> No inf/nan found under this check - if the assertion is still firing, you are almost certainly running the old,
    unfixed version of the cell. Search your notebook for 'X_shifted = X_raw - shift' WITHOUT 'np.maximum' around it.


In [11]:
# %% [5] SECONDARY split: blocked/contiguous per class, on ORIGINAL file order - the leakage robustness check
# Within each class, take the rows in the order they appear in the concatenated raw files (a rough time proxy),
# split 70/15/15 in blocks with a gap discarded between train and test, so temporally adjacent (near-duplicate) flows
# cannot straddle the train/test boundary the way a random split allows.
GAP_FRAC = 0.02   # discard 2% of each class's rows at each block boundary
blocked_tr, blocked_va, blocked_te = [], [], []
for c in range(5):
    rows = np.where(y == c)[0]         # already in original file order
    n = len(rows)
    if n < 20:
        print(f"class {CLASS_NAMES[c]}: only {n} rows, too few to block-split; assigning all to test only")
        blocked_te.extend(rows.tolist()); continue
    gap = max(1, int(GAP_FRAC * n))
    t_end = int(0.70 * n)
    v_end = t_end + gap + int(0.15 * n)
    blocked_tr.extend(rows[:t_end].tolist())
    blocked_va.extend(rows[t_end + gap: v_end].tolist())
    blocked_te.extend(rows[v_end + gap:].tolist())

blocked_tr, blocked_va, blocked_te = map(np.array, (blocked_tr, blocked_va, blocked_te))
print("\nblocked split sizes:", len(blocked_tr), len(blocked_va), len(blocked_te))
for name, ii in (("train", blocked_tr), ("val", blocked_va), ("test", blocked_te)):
    yy = y[ii]
    props = {CLASS_NAMES[c]: f"{100*np.mean(yy==c):.2f}%" for c in range(5)} if len(yy) else {}
    print(f"  {name}: n={len(ii)} {props}")

# Fit a SEPARATE scaler for the blocked split (its train rows differ from the primary split's train rows)
shift_b = np.minimum(X_raw[blocked_tr].min(axis=0), 0.0)
X_log_b = np.log1p(X_raw - shift_b)
scaler_b = MinMaxScaler().fit(X_log_b[blocked_tr])
X_all_b = np.clip(scaler_b.transform(X_log_b), 0.0, 1.0).astype("float32")

np.savez_compressed(f"{OUT}/cicids_v2_blocked.npz",
                    Xtr=X_all_b[blocked_tr], ytr=y[blocked_tr],
                    Xva=X_all_b[blocked_va], yva=y[blocked_va],
                    Xte=X_all_b[blocked_te], yte=y[blocked_te])
joblib.dump(scaler_b, f"{OUT}/log_minmax_scaler_blocked.joblib")
np.save(f"{OUT}/log_shift_blocked.npy", shift_b)

print("\nSaved:")
print(" ", f"{OUT}/cicids_v2_primary.npz", " (main protocol: stratified row-level)")
print(" ", f"{OUT}/cicids_v2_blocked.npz", " (leakage robustness check: blocked/contiguous)")
print(" ", f"{OUT}/feature_names.json")


blocked split sizes: 1764068 378013 277220
  train: n=1764068 {'BENIGN': '83.11%', 'Brute Force': '0.36%', 'DoS/DDoS': '12.76%', 'PortScan': '3.60%', 'Other': '0.16%'}
  val: n=378013 {'BENIGN': '83.11%', 'Brute Force': '0.36%', 'DoS/DDoS': '12.76%', 'PortScan': '3.60%', 'Other': '0.16%'}
  test: n=277220 {'BENIGN': '83.11%', 'Brute Force': '0.36%', 'DoS/DDoS': '12.76%', 'PortScan': '3.60%', 'Other': '0.17%'}


/tmp/ipykernel_2501/2211688745.py:29: RuntimeWarning: invalid value encountered in log1p
  X_log_b = np.log1p(X_raw - shift_b)



Saved:
  /content/drive/MyDrive/A3IDPS_COLAB/data_processed/cicids_v2/cicids_v2_primary.npz  (main protocol: stratified row-level)
  /content/drive/MyDrive/A3IDPS_COLAB/data_processed/cicids_v2/cicids_v2_blocked.npz  (leakage robustness check: blocked/contiguous)
  /content/drive/MyDrive/A3IDPS_COLAB/data_processed/cicids_v2/feature_names.json


In [12]:
# CICIDS Stage 1 - FIX for the blocked-split scaler in cell [5] (same bug as cell [4]: values below the
# blocked-train minimum went negative enough that log1p produced NaN). Run this to redo just the blocked-split
# scaling and re-save cicids_v2_blocked.npz correctly. Assumes X_raw, y, blocked_tr/va/te are still in memory.

import numpy as np, json, joblib

BASE = "/content/drive/MyDrive/A3IDPS_COLAB"
OUT  = f"{BASE}/data_processed/cicids_v2"

shift_b = np.minimum(X_raw[blocked_tr].min(axis=0), 0.0)
X_shifted_b = np.maximum(X_raw - shift_b, 0.0)          # <-- the same fix: floor at 0 before log1p
X_log_b = np.log1p(X_shifted_b)
assert np.isfinite(X_log_b).all(), "still non-finite in the blocked split - inspect further"

n_floored = int((X_raw[blocked_te] - shift_b < 0).sum())
print(f"blocked-test cells floored to blocked-train-min: {n_floored} (out of {X_raw[blocked_te].size})")

from sklearn.preprocessing import MinMaxScaler
scaler_b = MinMaxScaler().fit(X_log_b[blocked_tr])
X_all_b = scaler_b.transform(X_log_b).astype("float32")
frac_out = float(((X_all_b[blocked_te] < 0) | (X_all_b[blocked_te] > 1)).mean())
print(f"share of blocked-test values outside [0,1] before clipping: {frac_out:.6f}")
X_all_b = np.clip(X_all_b, 0.0, 1.0)
assert 0.0 <= X_all_b.min() and X_all_b.max() <= 1.0
print("blocked post-scaling range check OK: min", float(X_all_b.min()), "max", float(X_all_b.max()))

np.savez_compressed(f"{OUT}/cicids_v2_blocked.npz",
                    Xtr=X_all_b[blocked_tr], ytr=y[blocked_tr],
                    Xva=X_all_b[blocked_va], yva=y[blocked_va],
                    Xte=X_all_b[blocked_te], yte=y[blocked_te])
joblib.dump(scaler_b, f"{OUT}/log_minmax_scaler_blocked.joblib")
np.save(f"{OUT}/log_shift_blocked.npy", shift_b)
print("re-saved cicids_v2_blocked.npz, log_minmax_scaler_blocked.joblib, log_shift_blocked.npy correctly.")

# sanity re-check: reload from disk and confirm no NaN made it into the saved file
chk = np.load(f"{OUT}/cicids_v2_blocked.npz")
for k in ("Xtr", "Xva", "Xte"):
    assert np.isfinite(chk[k]).all(), f"{k} in the saved file still has non-finite values!"
print("re-load check passed: saved blocked-split arrays are all finite.")

blocked-test cells floored to blocked-train-min: 0 (out of 21623160)
share of blocked-test values outside [0,1] before clipping: 0.000001
blocked post-scaling range check OK: min 0.0 max 1.0
re-saved cicids_v2_blocked.npz, log_minmax_scaler_blocked.joblib, log_shift_blocked.npy correctly.
re-load check passed: saved blocked-split arrays are all finite.


In [13]:
# CICIDS2017 v2 - Stage 2: shared architecture, undefended baseline, attack sanity checks.
# Same structure and same frozen conventions as the NSL-KDD stages: logits-output model, worst-case attacks
# (FGSM / PGD-CE / PGD-margin), class-weighted training, and Carlini et al. sanity checks across an eps sweep.
# Primary protocol = the stratified row-level split (cicids_v2_primary.npz). The blocked split is evaluated
# separately at the end, purely as a robustness check, never used for training or selection.

# %% [0] Setup
from google.colab import drive
drive.mount('/content/drive')
import os, json, random, time
import numpy as np, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight

BASE = "/content/drive/MyDrive/A3IDPS_COLAB"
DATA = f"{BASE}/data_processed/cicids_v2"
MOD  = f"{BASE}/models/cicids_v2"
RES  = f"{BASE}/results/cicids_v2"
os.makedirs(MOD, exist_ok=True); os.makedirs(RES, exist_ok=True)

d = np.load(f"{DATA}/cicids_v2_primary.npz")
Xtr, Xva, Xte = [d[k].astype("float32") for k in ("Xtr", "Xva", "Xte")]
ytr, yva, yte = [d[k].astype("int32") for k in ("ytr", "yva", "yte")]
names = json.load(open(f"{DATA}/feature_names.json"))
D, NCLS = Xtr.shape[1], 5
CLASS_NAMES = ["BENIGN", "Brute Force", "DoS/DDoS", "PortScan", "Other"]
print("D =", D, "| train/val/test:", Xtr.shape, Xva.shape, Xte.shape)
print("class counts (train):", np.bincount(ytr), "| (test):", np.bincount(yte))

# No categorical one-hot columns here (unlike NSL-KDD), so the "numeric-only" vs "all-features" distinction
# does not apply - every one of the 78 CICIDS features is a genuine numeric flow statistic. All eps budgets
# below perturb every feature.
MASK_ALL = np.ones(D, "float32")

SEED = 0
EPS_EVAL = [0.01, 0.02, 0.05, 0.10, 0.15]
CLASS_WEIGHT_CAP, BATCH, MAX_EPOCHS = 20.0, 2048, 60          # larger batch: CICIDS train set is ~50x NSL-KDD's
cw = np.minimum(compute_class_weight("balanced", classes=np.arange(NCLS), y=ytr), CLASS_WEIGHT_CAP).astype("float32")
print("class weights:", cw.tolist())

def set_seed(s): random.seed(s); np.random.seed(s); tf.keras.utils.set_random_seed(s)
def jsonable(o):
    if isinstance(o, dict): return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [jsonable(v) for v in o]
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return float(o)
    return o

# %% [1] Model (LOGITS output) and attacks - identical structure to the NSL-KDD stages
def build_model(d, widths=(256, 128, 64)):
    inp = keras.Input(shape=(d,))
    x = layers.Dense(widths[0])(inp); x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)
    x = layers.Dense(widths[1])(x);   x = layers.Dropout(0.3)(x);          x = layers.Activation("relu")(x)
    x = layers.Dense(widths[2])(x);   x = layers.Dropout(0.3)(x);          x = layers.Activation("relu")(x)
    return keras.Model(inp, layers.Dense(NCLS)(x))

def make_attacks(model, mask):
    M = tf.constant(mask, tf.float32)
    def loss_fn(z, y, margin):
        if margin:
            oh = tf.one_hot(y, NCLS)
            return tf.reduce_max(z - 1e9 * oh, axis=1) - tf.reduce_sum(z * oh, axis=1)
        return tf.nn.sparse_softmax_cross_entropy_with_logits(labels=y, logits=z)
    @tf.function
    def fgsm(x0, y, eps):
        with tf.GradientTape() as t:
            t.watch(x0); s = tf.reduce_sum(loss_fn(model(x0, training=False), y, False))
        return tf.clip_by_value(x0 + eps * M * tf.sign(t.gradient(s, x0)), 0., 1.)
    @tf.function
    def pgd(x0, y, eps, steps, restarts, margin):
        E = eps * M
        best_x, best_l = x0, tf.fill(tf.shape(y), -1e30)
        for r in range(restarts):
            x = x0 if r == 0 else tf.clip_by_value(x0 + tf.random.uniform(tf.shape(x0), -1., 1.) * E, 0., 1.)
            for s_ in range(steps):
                alpha = (1.0 if s_ < steps // 2 else 0.25) * E / 4.0
                with tf.GradientTape() as t:
                    t.watch(x); s = tf.reduce_sum(loss_fn(model(x, training=False), y, margin))
                x = tf.clip_by_value(tf.clip_by_value(x + alpha * tf.sign(t.gradient(s, x)), x0 - E, x0 + E), 0., 1.)
                l_new = loss_fn(model(x, training=False), y, margin)
                upd = l_new > best_l
                best_x, best_l = tf.where(upd[:, None], x, best_x), tf.where(upd, l_new, best_l)
        return best_x
    return {"fgsm": fgsm, "pgd": pgd}

def run_attack(fn, X, y, bs=4096, **kw):
    return np.vstack([fn(tf.constant(X[i:i+bs]), tf.constant(y[i:i+bs]), **kw).numpy() for i in range(0, len(X), bs)])

def predict(model, X): return np.argmax(model.predict(X, batch_size=8192, verbose=0), axis=1)

def clean_eval(model, X, y):
    p = predict(model, X)
    present = sorted(np.unique(y).tolist())
    return {"acc": float(accuracy_score(y, p)),
            "macro_f1": float(f1_score(y, p, average="macro", labels=present, zero_division=0)),
            "per_class_f1": {CLASS_NAMES[c]: float(f1_score(y == c, p == c, zero_division=0)) for c in present},
            "per_class_support": {CLASS_NAMES[c]: int((y == c).sum()) for c in present}}

def robust_eval(model, X, y, mask, eps_list, steps=20, restarts=3):
    A = make_attacks(model, mask); out = {}
    present = sorted(np.unique(y).tolist())
    for e in eps_list:
        et = tf.constant(e, tf.float32)
        c_f = predict(model, run_attack(A["fgsm"], X, y, eps=et)) == y
        c_c = predict(model, run_attack(A["pgd"], X, y, eps=et, steps=steps, restarts=restarts, margin=False)) == y
        c_m = predict(model, run_attack(A["pgd"], X, y, eps=et, steps=steps, restarts=restarts, margin=True)) == y
        w = c_f & c_c & c_m
        out[str(e)] = {"fgsm": float(c_f.mean()), "pgd_ce": float(c_c.mean()), "pgd_margin": float(c_m.mean()),
                       "worst": float(w.mean()),
                       "worst_recall": {CLASS_NAMES[c]: float(w[y == c].mean()) for c in present}}
    return out

def train(model, X, y, Xv, yv, epochs=MAX_EPOCHS):
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    cw_dict = {i: float(w) for i, w in enumerate(cw)}
    model.fit(X, y, validation_data=(Xv, yv), epochs=epochs, batch_size=BATCH,
              class_weight=cw_dict, verbose=2,
              callbacks=[keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True),
                         keras.callbacks.ReduceLROnPlateau(patience=4, factor=0.5)])
    return model

# %% [2] Train the undefended baseline
set_seed(SEED)
model = build_model(D)
print("parameters (total):", model.count_params())
train(model, Xtr, ytr, Xva, yva)
model.save(f"{MOD}/undefended_seed{SEED}.keras")

clean = clean_eval(model, Xte, yte)
print("CLEAN test:", {k: clean[k] for k in ("acc", "macro_f1")})
print("per-class F1:", clean["per_class_f1"])
print("per-class support:", clean["per_class_support"])

# %% [3] Attack sanity checks on the PRIMARY test set (Carlini et al. red flags), same as NSL-KDD Stage 1
rows = []
for e in [0.0] + EPS_EVAL:
    if e == 0:
        rows.append({"eps": 0.0, "fgsm": clean["acc"], "pgd_ce": clean["acc"], "pgd_margin": clean["acc"], "worst": clean["acc"]}); continue
    A = make_attacks(model, MASK_ALL)
    c_f = predict(model, run_attack(A["fgsm"], Xte, yte, eps=tf.constant(e, tf.float32))) == yte
    c_c = predict(model, run_attack(A["pgd"], Xte, yte, eps=tf.constant(e, tf.float32), steps=20, restarts=3, margin=False)) == yte
    c_m = predict(model, run_attack(A["pgd"], Xte, yte, eps=tf.constant(e, tf.float32), steps=20, restarts=3, margin=True)) == yte
    w = c_f & c_c & c_m
    rows.append({"eps": e, "fgsm": float(c_f.mean()), "pgd_ce": float(c_c.mean()), "pgd_margin": float(c_m.mean()), "worst": float(w.mean())})
    print(f"eps={e:<5} FGSM={rows[-1]['fgsm']:.4f}  PGD-CE={rows[-1]['pgd_ce']:.4f}  PGD-margin={rows[-1]['pgd_margin']:.4f}  worst={rows[-1]['worst']:.4f}")

flags = []
for i in range(len(rows) - 1):
    if rows[i + 1]["worst"] > rows[i]["worst"] + 0.01:
        flags.append(f"worst-case accuracy rises from eps={rows[i]['eps']} to {rows[i+1]['eps']}")
    if rows[i + 1]["pgd_ce"] > rows[i]["fgsm"] + 0.02:
        flags.append(f"PGD-CE weaker than FGSM going from eps={rows[i]['eps']} to {rows[i+1]['eps']}")
if rows[-1]["worst"] > 0.20:
    flags.append(f"worst-case at eps={rows[-1]['eps']} (largest tested) still above 20% - may need a larger eps for full saturation")
print("RED FLAGS:", flags if flags else "none")

# %% [4] Robustness eval at the standard eps grid (primary test set)
primary_robust = robust_eval(model, Xte, yte, MASK_ALL, EPS_EVAL)
for e in EPS_EVAL:
    r = primary_robust[str(e)]
    print(f"eps={e}: worst={r['worst']:.4f} | per-class recall under worst-case attack: {r['worst_recall']}")

# %% [5] Blocked-split robustness check: evaluate the SAME model (trained only on the primary split's train data)
# on the blocked split's test set, to see whether performance differs meaningfully from the primary test set.
# This is NOT retraining - it's checking whether the primary split's row-level randomisation is inflating results
# relative to a split that respects (approximate) temporal/session locality.
db = np.load(f"{DATA}/cicids_v2_blocked.npz")
Xte_b, yte_b = db["Xte"].astype("float32"), db["yte"].astype("int32")
print(f"\nblocked test set: {Xte_b.shape}, classes present: {sorted(np.unique(yte_b).tolist())}")
clean_b = clean_eval(model, Xte_b, yte_b)
print("CLEAN on blocked test:", {k: clean_b[k] for k in ("acc", "macro_f1")}, "| per-class F1:", clean_b["per_class_f1"])
blocked_robust = robust_eval(model, Xte_b, yte_b, MASK_ALL, [0.05, 0.10])
for e in (0.05, 0.10):
    print(f"blocked test, eps={e}: worst={blocked_robust[str(e)]['worst']:.4f}  (primary test at same eps: {primary_robust[str(e)]['worst']:.4f})")
gap = abs(clean_b["acc"] - clean["acc"])
if gap > 0.05:
    print(f"NOTE: clean accuracy differs by {gap:.4f} between primary and blocked test sets - "
          f"this is evidence worth discussing in Limitations (possible temporal / near-duplicate leakage in the primary split).")
else:
    print(f"Clean accuracy differs by only {gap:.4f} between primary and blocked test sets - no strong evidence of leakage inflating the primary split's numbers.")

# %% [6] Save everything
out = {
    "params": model.count_params(),
    "clean_primary": clean,
    "attack_sanity_primary": rows,
    "flags": flags,
    "robust_primary": primary_robust,
    "clean_blocked": clean_b,
    "robust_blocked": blocked_robust,
    "clean_accuracy_gap_primary_vs_blocked": gap,
}
json.dump(jsonable(out), open(f"{RES}/stage2_undefended_seed{SEED}.json", "w"), indent=1)
print("\nsaved:", f"{RES}/stage2_undefended_seed{SEED}.json")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
D = 78 | train/val/test: (1764070, 78) (378015, 78) (378016, 78)
class counts (train): [1466120    6405  225163   63486    2896] | (test): [314169   1373  48249  13604    621]
class weights: [0.2406446933746338, 20.0, 1.5669270753860474, 5.557351112365723, 20.0]
parameters (total): 62725
Epoch 1/60
862/862 - 22s - 25ms/step - accuracy: 0.7482 - loss: 0.9905 - val_accuracy: 0.8658 - val_loss: 1.0776 - learning_rate: 0.0010
Epoch 2/60
862/862 - 3s - 3ms/step - accuracy: 0.7046 - loss: 1.0128 - val_accuracy: 0.8519 - val_loss: 1.0702 - learning_rate: 0.0010
Epoch 3/60
862/862 - 2s - 3ms/step - accuracy: 0.5982 - loss: 1.0184 - val_accuracy: 0.4782 - val_loss: 1.0720 - learning_rate: 0.0010
Epoch 4/60
862/862 - 2s - 3ms/step - accuracy: 0.5357 - loss: 1.0153 - val_accuracy: 0.6074 - val_loss: 1.0718 - learning_rate: 0.0010
Epoch 5/60
862/862 - 2s - 3ms/step - acc

In [14]:
# CICIDS Stage 2 - FIX: add SMOTE (as originally agreed in the spec, which I omitted the first time), and
# reduce the initial learning rate to guard against the same early-training instability. SMOTE is applied to
# the TRAINING split only, on the already log+minmax-scaled features (never touches val/test).
# Run this as a replacement for cells [2] onward of cicids_stage2_undefended.py (cell [0]/[1] - data loading and
# model/attack definitions - should already be in memory; if not, re-run those first).

# %% [A] Install imblearn if needed, then apply SMOTE to the training split only
import subprocess
subprocess.run(["pip", "install", "-q", "imbalanced-learn"], check=True)
from imblearn.over_sampling import SMOTE
import numpy as np

print("train class counts before SMOTE:", np.bincount(ytr))

# Target: bring the two smallest classes (Brute Force, Other) up to a workable size; leave BENIGN, DoS/DDoS,
# PortScan alone (already large enough that the model should be able to learn them from class weights alone).
TARGET_PER_CLASS = 50_000
counts = np.bincount(ytr)
strategy = {c: TARGET_PER_CLASS for c in range(len(counts)) if counts[c] < TARGET_PER_CLASS}
print("SMOTE strategy (classes to oversample -> target count):", strategy)

sm = SMOTE(random_state=0, k_neighbors=5, sampling_strategy=strategy)
Xtr_sm, ytr_sm = sm.fit_resample(Xtr, ytr)
print("train class counts after SMOTE:", np.bincount(ytr_sm))
print("SMOTE output range check: min", float(Xtr_sm.min()), "max", float(Xtr_sm.max()))
Xtr_sm = np.clip(Xtr_sm, 0.0, 1.0).astype("float32")   # SMOTE interpolation can occasionally land just outside [0,1]
ytr_sm = ytr_sm.astype("int32")

# %% [B] Recompute class weights on the SMOTE'd training set (should now need a much smaller cap, since the
# extreme classes are no longer extreme) and retrain with a lower initial learning rate
from sklearn.utils.class_weight import compute_class_weight
cw_sm = np.minimum(compute_class_weight("balanced", classes=np.arange(NCLS), y=ytr_sm), CLASS_WEIGHT_CAP).astype("float32")
print("class weights after SMOTE:", cw_sm.tolist())

set_seed(SEED)
model2 = build_model(D)
model2.compile(optimizer=keras.optimizers.Adam(3e-4),           # lower LR than before (was 1e-3)
               loss="sparse_categorical_crossentropy", metrics=["accuracy"])
cw_dict2 = {i: float(w) for i, w in enumerate(cw_sm)}
hist2 = model2.fit(Xtr_sm, ytr_sm, validation_data=(Xva, yva), epochs=MAX_EPOCHS, batch_size=BATCH,
                   class_weight=cw_dict2, verbose=2,
                   callbacks=[keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True),
                              keras.callbacks.ReduceLROnPlateau(patience=4, factor=0.5)])

# %% [C] Evaluate: check the model no longer collapses to majority-only before trusting anything else
clean2 = clean_eval(model2, Xte, yte)
print("\nCLEAN test (SMOTE + lower LR):", {k: clean2[k] for k in ("acc", "macro_f1")})
print("per-class F1:", clean2["per_class_f1"])

collapsed = all(clean2["per_class_f1"][c] == 0.0 for c in ("Brute Force", "DoS/DDoS", "PortScan", "Other"))
if collapsed:
    print("\n*** STILL COLLAPSED: every minority class F1 is 0.0. Do NOT proceed - paste this output back before continuing. ***")
else:
    print("\nModel is no longer predicting a single class - proceeding looks reasonable. Per-class support was:",
          clean2["per_class_support"])
    model2.save(f"{MOD}/undefended_smote_seed{SEED}.keras")
    print("saved:", f"{MOD}/undefended_smote_seed{SEED}.keras")

train class counts before SMOTE: [1466120    6405  225163   63486    2896]
SMOTE strategy (classes to oversample -> target count): {1: 50000, 4: 50000}
train class counts after SMOTE: [1466120   50000  225163   63486   50000]
SMOTE output range check: min 0.0 max 1.0
class weights after SMOTE: [0.2530173659324646, 7.419075965881348, 1.6474900245666504, 5.843080520629883, 7.419075965881348]
Epoch 1/60
906/906 - 17s - 19ms/step - accuracy: 0.8052 - loss: 0.6162 - val_accuracy: 0.8948 - val_loss: 0.2730 - learning_rate: 3.0000e-04
Epoch 2/60
906/906 - 3s - 3ms/step - accuracy: 0.8806 - loss: 0.4462 - val_accuracy: 0.9519 - val_loss: 0.3307 - learning_rate: 3.0000e-04
Epoch 3/60
906/906 - 3s - 3ms/step - accuracy: 0.8059 - loss: 1.0671 - val_accuracy: 0.8130 - val_loss: 0.8383 - learning_rate: 3.0000e-04
Epoch 4/60
906/906 - 3s - 3ms/step - accuracy: 0.7699 - loss: 1.1913 - val_accuracy: 0.9305 - val_loss: 0.6387 - learning_rate: 3.0000e-04
Epoch 5/60
906/906 - 3s - 3ms/step - accuracy: 0.

In [15]:
# CICIDS Stage 2 - completion: attack sanity checks + blocked-split check on the CORRECTED (SMOTE'd) undefended
# model, replacing the stale results from the collapsed run. Needs Xte, yte, MASK_ALL, make_attacks, run_attack,
# predict, clean_eval, robust_eval, CLASS_NAMES, EPS_EVAL, RES, SEED still in memory from cicids_stage2_undefended.py,
# plus model2 (or reload from disk) from the SMOTE fix.

# %% [0] Load the corrected model (reload from disk in case the session was restarted)
import json, numpy as np
from tensorflow import keras

BASE = "/content/drive/MyDrive/A3IDPS_COLAB"
MOD = f"{BASE}/models/cicids_v2"; RES = f"{BASE}/results/cicids_v2"
model2 = keras.models.load_model(f"{MOD}/undefended_smote_seed{SEED}.keras", compile=False)

clean2 = clean_eval(model2, Xte, yte)
print("re-confirmed CLEAN test:", {k: clean2[k] for k in ("acc", "macro_f1")})
assert clean2["per_class_f1"]["Brute Force"] > 0.0 and clean2["per_class_f1"]["DoS/DDoS"] > 0.0, \
    "loaded model looks collapsed again - stop and check which model file this is"

# %% [1] Attack sanity checks (Carlini et al. red flags), same eps grid as before, on the corrected model
rows = []
for e in [0.0] + EPS_EVAL:
    if e == 0:
        rows.append({"eps": 0.0, "fgsm": clean2["acc"], "pgd_ce": clean2["acc"], "pgd_margin": clean2["acc"], "worst": clean2["acc"]}); continue
    A = make_attacks(model2, MASK_ALL)
    import tensorflow as tf
    c_f = predict(model2, run_attack(A["fgsm"], Xte, yte, eps=tf.constant(e, tf.float32))) == yte
    c_c = predict(model2, run_attack(A["pgd"], Xte, yte, eps=tf.constant(e, tf.float32), steps=20, restarts=3, margin=False)) == yte
    c_m = predict(model2, run_attack(A["pgd"], Xte, yte, eps=tf.constant(e, tf.float32), steps=20, restarts=3, margin=True)) == yte
    w = c_f & c_c & c_m
    rows.append({"eps": e, "fgsm": float(c_f.mean()), "pgd_ce": float(c_c.mean()), "pgd_margin": float(c_m.mean()), "worst": float(w.mean())})
    print(f"eps={e:<5} FGSM={rows[-1]['fgsm']:.4f}  PGD-CE={rows[-1]['pgd_ce']:.4f}  PGD-margin={rows[-1]['pgd_margin']:.4f}  worst={rows[-1]['worst']:.4f}")

flags = []
for i in range(len(rows) - 1):
    if rows[i + 1]["worst"] > rows[i]["worst"] + 0.01:
        flags.append(f"worst-case accuracy rises from eps={rows[i]['eps']} to {rows[i+1]['eps']}")
    if rows[i + 1]["pgd_ce"] > rows[i]["fgsm"] + 0.02:
        flags.append(f"PGD-CE weaker than FGSM going from eps={rows[i]['eps']} to {rows[i+1]['eps']}")
if rows[-1]["worst"] > 0.20:
    flags.append(f"worst-case at eps={rows[-1]['eps']} (largest tested) still above 20% - may need a larger eps, or this may just reflect the BENIGN majority floor")
print("RED FLAGS:", flags if flags else "none")

# %% [2] Robustness eval at the standard eps grid, with per-class recall (the number that actually matters for an IDS)
primary_robust = robust_eval(model2, Xte, yte, MASK_ALL, EPS_EVAL)
for e in EPS_EVAL:
    r = primary_robust[str(e)]
    print(f"eps={e}: worst={r['worst']:.4f} | per-class recall under worst-case attack: {r['worst_recall']}")

# %% [3] Blocked-split check on the corrected model
db = np.load(f"{BASE}/data_processed/cicids_v2/cicids_v2_blocked.npz")
Xte_b, yte_b = db["Xte"].astype("float32"), db["yte"].astype("int32")
clean_b = clean_eval(model2, Xte_b, yte_b)
print("\nCLEAN on blocked test:", {k: clean_b[k] for k in ("acc", "macro_f1")}, "| per-class F1:", clean_b["per_class_f1"])
blocked_robust = robust_eval(model2, Xte_b, yte_b, MASK_ALL, [0.05, 0.10])
for e in (0.05, 0.10):
    print(f"blocked test, eps={e}: worst={blocked_robust[str(e)]['worst']:.4f}  (primary test at same eps: {primary_robust[str(e)]['worst']:.4f})")
gap = abs(clean_b["acc"] - clean2["acc"])
macro_gap = abs(clean_b["macro_f1"] - clean2["macro_f1"])
print(f"clean accuracy gap (primary vs blocked): {gap:.4f} | macro-F1 gap: {macro_gap:.4f}")
if gap > 0.05 or macro_gap > 0.05:
    print("NOTE: meaningful gap between primary and blocked test performance - discuss in Limitations "
          "(possible temporal / near-duplicate leakage in the row-level primary split).")
else:
    print("No strong evidence of leakage inflating the primary split's numbers.")

# %% [4] Save (overwrites the stale collapsed-model JSON)
def jsonable(o):
    if isinstance(o, dict): return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [jsonable(v) for v in o]
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return float(o)
    return o

out = {
    "note": "corrected: trained with SMOTE (Brute Force, Other -> 50000) + lr=3e-4, replacing the collapsed run that used class weights only",
    "params": model2.count_params(),
    "clean_primary": clean2,
    "attack_sanity_primary": rows,
    "flags": flags,
    "robust_primary": primary_robust,
    "clean_blocked": clean_b,
    "robust_blocked": blocked_robust,
    "clean_accuracy_gap_primary_vs_blocked": gap,
    "macro_f1_gap_primary_vs_blocked": macro_gap,
}
json.dump(jsonable(out), open(f"{RES}/stage2_undefended_seed{SEED}.json", "w"), indent=1)
print("\nsaved (overwritten):", f"{RES}/stage2_undefended_seed{SEED}.json")

re-confirmed CLEAN test: {'acc': 0.8929383941420469, 'macro_f1': 0.7184019640657536}
eps=0.01  FGSM=0.8505  PGD-CE=0.8467  PGD-margin=0.8408  worst=0.8408
eps=0.02  FGSM=0.8043  PGD-CE=0.7904  PGD-margin=0.7764  worst=0.7754
eps=0.05  FGSM=0.6555  PGD-CE=0.5901  PGD-margin=0.5741  worst=0.5727
eps=0.1   FGSM=0.4904  PGD-CE=0.1865  PGD-margin=0.1123  worst=0.0928
eps=0.15  FGSM=0.2760  PGD-CE=0.0006  PGD-margin=0.0001  worst=0.0001
RED FLAGS: none
eps=0.01: worst=0.8408 | per-class recall under worst-case attack: {'BENIGN': 0.8161690045803374, 'Brute Force': 0.9708667152221413, 'DoS/DDoS': 0.9546104582478393, 'PortScan': 0.988900323434284, 'Other': 0.9001610305958132}
eps=0.02: worst=0.7754 | per-class recall under worst-case attack: {'BENIGN': 0.744494205348077, 'Brute Force': 0.6737072104879825, 'DoS/DDoS': 0.9206615681154013, 'PortScan': 0.9846368715083799, 'Other': 0.7793880837359098}
eps=0.05: worst=0.5727 | per-class recall under worst-case attack: {'BENIGN': 0.5399323294150601, '

In [16]:
# CICIDS2017 v2 - Stage 2b: FGSM-AT / PGD-AT with an eps_train sweep, validation-based selection (same rule as
# NSL-KDD), and worst-case test evaluation. Self-contained (re-defines everything needed in case of a fresh session).
# Carries forward two fixes learned from the undefended baseline: SMOTE on the training pool, and lr=3e-4.
# A collapse-guard checks per-class F1 after every training run before accepting it.

# %% [0] Setup
from google.colab import drive
drive.mount('/content/drive')
import os, json, random, time
import numpy as np, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight

BASE = "/content/drive/MyDrive/A3IDPS_COLAB"
DATA = f"{BASE}/data_processed/cicids_v2"; MOD = f"{BASE}/models/cicids_v2"; RES = f"{BASE}/results/cicids_v2"
os.makedirs(MOD, exist_ok=True); os.makedirs(RES, exist_ok=True)

d = np.load(f"{DATA}/cicids_v2_primary.npz")
Xtr, Xva, Xte = [d[k].astype("float32") for k in ("Xtr", "Xva", "Xte")]
ytr, yva, yte = [d[k].astype("int32") for k in ("ytr", "yva", "yte")]
D, NCLS = Xtr.shape[1], 5
CLASS_NAMES = ["BENIGN", "Brute Force", "DoS/DDoS", "PortScan", "Other"]
MASK_ALL = np.ones(D, "float32")

SEED = 0
EPS_EVAL, EPS_SELECT = [0.01, 0.02, 0.05, 0.10, 0.15], [0.01, 0.02, 0.05, 0.10]
EPS_TRAIN = [0.02, 0.05, 0.10]
CLASS_WEIGHT_CAP, VAL_CLEAN_TOL, BATCH, MAX_EPOCHS, LR = 20.0, 0.05, 2048, 60, 3e-4
FAST = True
def maybe_tf(fn): return tf.function(fn, reduce_retracing=True) if FAST else fn
def set_seed(s): random.seed(s); np.random.seed(s); tf.keras.utils.set_random_seed(s)
def jsonable(o):
    if isinstance(o, dict): return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [jsonable(v) for v in o]
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return float(o)
    return o

# %% [1] SMOTE the training pool ONCE (same target as the corrected undefended baseline)
from imblearn.over_sampling import SMOTE
SMOTE_TARGET = 50_000
counts = np.bincount(ytr)
strategy = {c: SMOTE_TARGET for c in range(len(counts)) if counts[c] < SMOTE_TARGET}
print("SMOTE strategy:", strategy)
Xtr_sm, ytr_sm = SMOTE(random_state=0, k_neighbors=5, sampling_strategy=strategy).fit_resample(Xtr, ytr)
Xtr_sm = np.clip(Xtr_sm, 0.0, 1.0).astype("float32"); ytr_sm = ytr_sm.astype("int32")
print("post-SMOTE train class counts:", np.bincount(ytr_sm))
cw = np.minimum(compute_class_weight("balanced", classes=np.arange(NCLS), y=ytr_sm), CLASS_WEIGHT_CAP).astype("float32")
print("class weights:", cw.tolist())

# %% [2] Model, attacks, eval (same structure as NSL-KDD / CICIDS Stage 2)
def build_model(d, widths=(256, 128, 64)):
    inp = keras.Input(shape=(d,))
    x = layers.Dense(widths[0])(inp); x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)
    x = layers.Dense(widths[1])(x);   x = layers.Dropout(0.3)(x);          x = layers.Activation("relu")(x)
    x = layers.Dense(widths[2])(x);   x = layers.Dropout(0.3)(x);          x = layers.Activation("relu")(x)
    return keras.Model(inp, layers.Dense(NCLS)(x))

def make_attacks(model, mask):
    M = tf.constant(mask, tf.float32)
    def loss_fn(z, y, margin):
        if margin:
            oh = tf.one_hot(y, NCLS)
            return tf.reduce_max(z - 1e9 * oh, axis=1) - tf.reduce_sum(z * oh, axis=1)
        return tf.nn.sparse_softmax_cross_entropy_with_logits(labels=y, logits=z)
    @tf.function
    def fgsm(x0, y, eps):
        with tf.GradientTape() as t:
            t.watch(x0); s = tf.reduce_sum(loss_fn(model(x0, training=False), y, False))
        return tf.clip_by_value(x0 + eps * M * tf.sign(t.gradient(s, x0)), 0., 1.)
    @tf.function
    def pgd(x0, y, eps, steps, restarts, margin):
        E = eps * M
        best_x, best_l = x0, tf.fill(tf.shape(y), -1e30)
        for r in range(restarts):
            x = x0 if r == 0 else tf.clip_by_value(x0 + tf.random.uniform(tf.shape(x0), -1., 1.) * E, 0., 1.)
            for s_ in range(steps):
                alpha = (1.0 if s_ < steps // 2 else 0.25) * E / 4.0
                with tf.GradientTape() as t:
                    t.watch(x); s = tf.reduce_sum(loss_fn(model(x, training=False), y, margin))
                x = tf.clip_by_value(tf.clip_by_value(x + alpha * tf.sign(t.gradient(s, x)), x0 - E, x0 + E), 0., 1.)
                l_new = loss_fn(model(x, training=False), y, margin)
                upd = l_new > best_l
                best_x, best_l = tf.where(upd[:, None], x, best_x), tf.where(upd, l_new, best_l)
        return best_x
    return {"fgsm": fgsm, "pgd": pgd}

def run_attack(fn, X, y, bs=4096, **kw):
    return np.vstack([fn(tf.constant(X[i:i+bs]), tf.constant(y[i:i+bs]), **kw).numpy() for i in range(0, len(X), bs)])
def predict(model, X): return np.argmax(model.predict(X, batch_size=8192, verbose=0), axis=1)

def clean_eval(model, X, y):
    p = predict(model, X); present = sorted(np.unique(y).tolist())
    return {"acc": float(accuracy_score(y, p)),
            "macro_f1": float(f1_score(y, p, average="macro", labels=present, zero_division=0)),
            "per_class_f1": {CLASS_NAMES[c]: float(f1_score(y == c, p == c, zero_division=0)) for c in present}}

def robust_eval(model, X, y, mask, eps_list, steps=20, restarts=3):
    A = make_attacks(model, mask); out = {}; present = sorted(np.unique(y).tolist())
    for e in eps_list:
        et = tf.constant(e, tf.float32)
        c_f = predict(model, run_attack(A["fgsm"], X, y, eps=et)) == y
        c_c = predict(model, run_attack(A["pgd"], X, y, eps=et, steps=steps, restarts=restarts, margin=False)) == y
        c_m = predict(model, run_attack(A["pgd"], X, y, eps=et, steps=steps, restarts=restarts, margin=True)) == y
        w = c_f & c_c & c_m
        out[str(e)] = {"fgsm": float(c_f.mean()), "pgd_ce": float(c_c.mean()), "pgd_margin": float(c_m.mean()),
                       "worst": float(w.mean()), "worst_recall": {CLASS_NAMES[c]: float(w[y == c].mean()) for c in present}}
    return out

def collapse_check(clean, tag):
    bad = [c for c in ("Brute Force", "DoS/DDoS", "PortScan") if clean["per_class_f1"].get(c, 0.0) == 0.0]
    if bad:
        raise RuntimeError(f"[{tag}] COLLAPSED: F1=0 for {bad} - stop and inspect before continuing")
    print(f"[{tag}] collapse check passed: {clean['per_class_f1']}")

# %% [3] Training loop: undefended reload, or FGSM-AT / PGD-AT with adversarial examples generated on the fly
def fit_classifier(model, X, y, Xv, yv, at=None, lr=LR, max_epochs=MAX_EPOCHS, patience=8, seed=SEED, tag=""):
    set_seed(seed)
    opt = keras.optimizers.Adam(lr); opt.build(model.trainable_variables)
    A = make_attacks(model, MASK_ALL) if at is not None else None
    et = tf.constant(at[1] if at else 0., tf.float32)
    cwt = tf.constant(cw)
    def wce(z, yy): return tf.reduce_mean(tf.nn.sparse_softmax_cross_entropy_with_logits(labels=yy, logits=z) * tf.gather(cwt, yy))

    @maybe_tf
    def step(xb, yb):
        with tf.GradientTape() as t:
            loss = wce(model(xb, training=True), yb)
        opt.apply_gradients(zip(t.gradient(loss, model.trainable_variables), model.trainable_variables))
        return loss

    rng, n = np.random.RandomState(seed), len(X)
    best, best_w, wait, cur_lr = 1e9, None, 0, lr
    for ep in range(max_epochs):
        t0 = time.time(); perm = rng.permutation(n)
        for i in range(0, n - BATCH + 1, BATCH):
            idx = perm[i:i+BATCH]; xt, yt = tf.constant(X[idx]), tf.constant(y[idx])
            if at is not None:
                xa = A["fgsm"](xt, yt, eps=et) if at[0] == "fgsm" else A["pgd"](xt, yt, eps=et, steps=10, restarts=1, margin=False)
                xt, yt = tf.concat([xt, xa], 0), tf.concat([yt, yt], 0)
            step(xt, yt)
        vl = float(wce(model(tf.constant(Xv), training=False), tf.constant(yv)))
        if at is not None:
            xv = run_attack(A["fgsm"], Xv, yv, eps=et)
            vl += float(wce(model(tf.constant(xv), training=False), tf.constant(yv)))
        if vl < best - 1e-4: best, best_w, wait = vl, model.get_weights(), 0
        else:
            wait += 1
            if wait % 4 == 0: cur_lr *= 0.5; opt.learning_rate = cur_lr
            if wait >= patience: break
        if (ep + 1) % 5 == 0 or ep == 0:
            print(f"[{tag}] epoch {ep+1} val_loss={vl:.4f} best={best:.4f} ({time.time()-t0:.1f}s)")
    model.set_weights(best_w)
    return model

# %% [4] Train / load undefended (already done in Stage 2) + FGSM-AT / PGD-AT sweep, score on VALIDATION
und = keras.models.load_model(f"{MOD}/undefended_smote_seed{SEED}.keras", compile=False)
und_val = clean_eval(und, Xva, yva); collapse_check(und_val, "undefended(val)")
print("undefended val clean acc:", und_val["acc"])

SPEC = [("fgsm", e) for e in EPS_TRAIN] + [("pgd", e) for e in EPS_TRAIN]
models, val_res = {}, {}
for kind, e in SPEC:
    tag = f"{kind}_eps{e}"
    path = f"{MOD}/{tag}_seed{SEED}.keras"
    if os.path.exists(path):
        m = keras.models.load_model(path, compile=False)
    else:
        m = build_model(D)
        fit_classifier(m, Xtr_sm, ytr_sm, Xva, yva, at=(kind, e), tag=tag)
        m.save(path)
    models[tag] = m
    vc = clean_eval(m, Xva, yva); collapse_check(vc, f"{tag}(val)")
    vr = robust_eval(m, Xva, yva, MASK_ALL, EPS_SELECT, steps=10, restarts=1)
    val_res[tag] = {"val_clean": vc, "val_rob": vr, "val_rob_mean": float(np.mean([vr[str(x)]["worst"] for x in EPS_SELECT]))}
    print(f"VAL {tag:<12} clean acc={vc['acc']:.4f}  mean worst-case robust acc={val_res[tag]['val_rob_mean']:.4f}")
    json.dump(jsonable(val_res), open(f"{RES}/stage2b_val_seed{SEED}.json", "w"), indent=1)

# %% [5] Select eps_train per method (same rule as NSL-KDD, validation only)
selected = {}
for method in ("fgsm", "pgd"):
    cands = [(e, val_res[f"{method}_eps{e}"]) for e in EPS_TRAIN]
    elig = [(e, v) for e, v in cands if und_val["acc"] - v["val_clean"]["acc"] <= VAL_CLEAN_TOL]
    pool = elig if elig else cands
    if not elig: print(f"WARNING: no {method} candidate within {VAL_CLEAN_TOL:.0%} clean-accuracy tolerance")
    e_best = max(pool, key=lambda ev: (round(ev[1]["val_rob_mean"], 6), -ev[0]))[0]
    selected[method] = e_best
    for e, v in cands:
        print(f"{method:<5} eps_train={e:<5} val clean={v['val_clean']['acc']:.4f} mean robust={v['val_rob_mean']:.4f} "
              f"eligible={(e, v) in elig} {'<== selected' if e == e_best else ''}")
json.dump(selected, open(f"{RES}/stage2b_selected_seed{SEED}.json", "w"))

# %% [6] TEST evaluation (undefended + the two selected AT models)
test_res = {"undefended": {"clean": clean_eval(und, Xte, yte), "robust": robust_eval(und, Xte, yte, MASK_ALL, EPS_EVAL)}}
for method in ("fgsm", "pgd"):
    tag = f"{method}_eps{selected[method]}"; m = models[tag]
    tc = clean_eval(m, Xte, yte); collapse_check(tc, f"{tag}(test)")
    test_res[tag] = {"clean": tc, "robust": robust_eval(m, Xte, yte, MASK_ALL, EPS_EVAL)}
json.dump(jsonable({"selected": selected, "test": test_res}), open(f"{RES}/stage2b_test_seed{SEED}.json", "w"), indent=1)

print(f"\n=== TEST worst-case robust accuracy (selected models) ===")
print(f"{'model':<16}{'clean':>8}{'macroF1':>9}" + "".join(f"{('e='+str(e)):>9}" for e in EPS_EVAL))
for name, key in [("undefended", "undefended"), (f"fgsm eps={selected['fgsm']}", f"fgsm_eps{selected['fgsm']}"),
                  (f"pgd eps={selected['pgd']}", f"pgd_eps{selected['pgd']}")]:
    r, c = test_res[key]["robust"], test_res[key]["clean"]
    print(f"{name:<16}{c['acc']:>8.4f}{c['macro_f1']:>9.4f}" + "".join(f"{r[str(e)]['worst']:>9.4f}" for e in EPS_EVAL))
print("\nsaved:", f"{RES}/stage2b_test_seed{SEED}.json")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
SMOTE strategy: {1: 50000, 4: 50000}
post-SMOTE train class counts: [1466120   50000  225163   63486   50000]
class weights: [0.2530173659324646, 7.419075965881348, 1.6474900245666504, 5.843080520629883, 7.419075965881348]
[undefended(val)] collapse check passed: {'BENIGN': 0.9329077479420929, 'Brute Force': 0.7889314100677068, 'DoS/DDoS': 0.9323415934131498, 'PortScan': 0.9190936309199782, 'Other': 0.0382320867278209}
undefended val clean acc: 0.8947528537227358
[fgsm_eps0.02] epoch 1 val_loss=0.0829 best=0.0829 (7.7s)
[fgsm_eps0.02] epoch 5 val_loss=0.0365 best=0.0365 (6.0s)
[fgsm_eps0.02] epoch 10 val_loss=0.0316 best=0.0316 (5.8s)
[fgsm_eps0.02] epoch 15 val_loss=0.0265 best=0.0265 (5.9s)
[fgsm_eps0.02] epoch 20 val_loss=0.0266 best=0.0241 (5.9s)
[fgsm_eps0.02] epoch 25 val_loss=0.0288 best=0.0228 (5.7s)
[fgsm_eps0.02] epoch 30 val_loss=0.0239 best=0.0209

In [17]:
# CICIDS2017 v2 - Stage 3: A3IDPS (teacher -> distilled student -> corrected AdvGAN generator -> feedback rounds
# 0-5), same structure as NSL-KDD Stage 3, adapted for CICIDS: trains on the SMOTE'd pool (Stage 2b), no
# categorical-mask distinction (all 78 features are numeric, so MASK = all-ones), and an extra collapse-guard
# after every single training stage (teacher / baseline / each feedback round) given what happened in Stage 2.
# Self-contained. Ablations: distilled+feedback (main), distilled+fixed-generator (control), no-distillation+feedback.

# %% [0] Setup
from google.colab import drive
drive.mount('/content/drive')
import os, json, random, time
import numpy as np, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight

BASE = "/content/drive/MyDrive/A3IDPS_COLAB"
DATA = f"{BASE}/data_processed/cicids_v2"; MOD = f"{BASE}/models/cicids_v2"; RES = f"{BASE}/results/cicids_v2"
os.makedirs(MOD, exist_ok=True); os.makedirs(RES, exist_ok=True)

d = np.load(f"{DATA}/cicids_v2_primary.npz")
Xtr, Xva, Xte = [d[k].astype("float32") for k in ("Xtr", "Xva", "Xte")]
ytr, yva, yte = [d[k].astype("int32") for k in ("ytr", "yva", "yte")]
D, NCLS = Xtr.shape[1], 5
CLASS_NAMES = ["BENIGN", "Brute Force", "DoS/DDoS", "PortScan", "Other"]
MASK = np.ones(D, "float32")   # all 78 CICIDS features are numeric - no categorical columns to freeze

SEED = 0
EPS_EVAL, EPS_SELECT = [0.01, 0.02, 0.05, 0.10, 0.15], [0.01, 0.02, 0.05, 0.10]
EPS_TRAIN = [0.02, 0.05, 0.10]      # generator eps_max grid, same as the AT sweep
CLASS_WEIGHT_CAP, VAL_CLEAN_TOL, BATCH = 20.0, 0.05, 2048
BASE_LR, FEEDBACK_LR = 3e-4, 1.5e-4   # both lower than NSL's, carried forward from the Stage 2 collapse fix
KD_ALPHA, KD_T = 0.7, 3.0
GEN_STEPS, LAM_CLS, LAM_L1, LAM_GAN, KAPPA = 1200, 20.0, 0.5, 1.0, 1.0
ROUNDS = 5
FAST = True
def maybe_tf(fn): return tf.function(fn, reduce_retracing=True) if FAST else fn
def set_seed(s): random.seed(s); np.random.seed(s); tf.keras.utils.set_random_seed(s)
def jsonable(o):
    if isinstance(o, dict): return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [jsonable(v) for v in o]
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return float(o)
    return o

# %% [1] SMOTE the training pool (identical to Stage 2b - must match so all models are trained on the same data)
from imblearn.over_sampling import SMOTE
SMOTE_TARGET = 50_000
counts = np.bincount(ytr)
strategy = {c: SMOTE_TARGET for c in range(len(counts)) if counts[c] < SMOTE_TARGET}
Xtr_sm, ytr_sm = SMOTE(random_state=0, k_neighbors=5, sampling_strategy=strategy).fit_resample(Xtr, ytr)
Xtr_sm = np.clip(Xtr_sm, 0.0, 1.0).astype("float32"); ytr_sm = ytr_sm.astype("int32")
print("post-SMOTE train class counts:", np.bincount(ytr_sm))
cw = np.minimum(compute_class_weight("balanced", classes=np.arange(NCLS), y=ytr_sm), CLASS_WEIGHT_CAP).astype("float32")

# %% [2] Model, attacks, eval, collapse-guard (same structure as Stage 2b)
def build_model(d, widths=(256, 128, 64)):
    inp = keras.Input(shape=(d,))
    x = layers.Dense(widths[0])(inp); x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)
    x = layers.Dense(widths[1])(x);   x = layers.Dropout(0.3)(x);          x = layers.Activation("relu")(x)
    x = layers.Dense(widths[2])(x);   x = layers.Dropout(0.3)(x);          x = layers.Activation("relu")(x)
    return keras.Model(inp, layers.Dense(NCLS)(x))

def make_attacks(model, mask):
    M = tf.constant(mask, tf.float32)
    def loss_fn(z, y, margin):
        if margin:
            oh = tf.one_hot(y, NCLS)
            return tf.reduce_max(z - 1e9 * oh, axis=1) - tf.reduce_sum(z * oh, axis=1)
        return tf.nn.sparse_softmax_cross_entropy_with_logits(labels=y, logits=z)
    @tf.function
    def fgsm(x0, y, eps):
        with tf.GradientTape() as t:
            t.watch(x0); s = tf.reduce_sum(loss_fn(model(x0, training=False), y, False))
        return tf.clip_by_value(x0 + eps * M * tf.sign(t.gradient(s, x0)), 0., 1.)
    @tf.function
    def pgd(x0, y, eps, steps, restarts, margin):
        E = eps * M
        best_x, best_l = x0, tf.fill(tf.shape(y), -1e30)
        for r in range(restarts):
            x = x0 if r == 0 else tf.clip_by_value(x0 + tf.random.uniform(tf.shape(x0), -1., 1.) * E, 0., 1.)
            for s_ in range(steps):
                alpha = (1.0 if s_ < steps // 2 else 0.25) * E / 4.0
                with tf.GradientTape() as t:
                    t.watch(x); s = tf.reduce_sum(loss_fn(model(x, training=False), y, margin))
                x = tf.clip_by_value(tf.clip_by_value(x + alpha * tf.sign(t.gradient(s, x)), x0 - E, x0 + E), 0., 1.)
                l_new = loss_fn(model(x, training=False), y, margin)
                upd = l_new > best_l
                best_x, best_l = tf.where(upd[:, None], x, best_x), tf.where(upd, l_new, best_l)
        return best_x
    return {"fgsm": fgsm, "pgd": pgd}

def run_attack(fn, X, y, bs=4096, **kw):
    return np.vstack([fn(tf.constant(X[i:i+bs]), tf.constant(y[i:i+bs]), **kw).numpy() for i in range(0, len(X), bs)])
def predict(model, X): return np.argmax(model.predict(X, batch_size=8192, verbose=0), axis=1)

def clean_eval(model, X, y):
    p = predict(model, X); present = sorted(np.unique(y).tolist())
    return {"acc": float(accuracy_score(y, p)),
            "macro_f1": float(f1_score(y, p, average="macro", labels=present, zero_division=0)),
            "per_class_f1": {CLASS_NAMES[c]: float(f1_score(y == c, p == c, zero_division=0)) for c in present}}

def robust_eval(model, X, y, mask, eps_list, steps=20, restarts=3):
    A = make_attacks(model, mask); out = {}; present = sorted(np.unique(y).tolist())
    for e in eps_list:
        et = tf.constant(e, tf.float32)
        c_f = predict(model, run_attack(A["fgsm"], X, y, eps=et)) == y
        c_c = predict(model, run_attack(A["pgd"], X, y, eps=et, steps=steps, restarts=restarts, margin=False)) == y
        c_m = predict(model, run_attack(A["pgd"], X, y, eps=et, steps=steps, restarts=restarts, margin=True)) == y
        w = c_f & c_c & c_m
        out[str(e)] = {"fgsm": float(c_f.mean()), "pgd_ce": float(c_c.mean()), "pgd_margin": float(c_m.mean()),
                       "worst": float(w.mean()), "worst_recall": {CLASS_NAMES[c]: float(w[y == c].mean()) for c in present}}
    return out

def collapse_check(clean, tag):
    bad = [c for c in ("Brute Force", "DoS/DDoS", "PortScan") if clean["per_class_f1"].get(c, 0.0) == 0.0]
    if bad: raise RuntimeError(f"[{tag}] COLLAPSED: F1=0 for {bad} - stop and inspect before continuing")
    print(f"[{tag}] collapse check passed")

# %% [3] Classifier training: plain / distilled (zt) / with extra generated samples (feedback)
def fit_classifier(model, X, y, Xv, yv, X_extra=None, Xv_extra=None, zt=None,
                   lr=BASE_LR, max_epochs=60, patience=8, seed=SEED, tag=""):
    set_seed(seed)
    opt = keras.optimizers.Adam(lr); opt.build(model.trainable_variables)
    use_kd, cwt = zt is not None, tf.constant(cw)
    def wce(z, yy): return tf.reduce_mean(tf.nn.sparse_softmax_cross_entropy_with_logits(labels=yy, logits=z) * tf.gather(cwt, yy))

    @maybe_tf
    def step(xb, yb, ztb):
        with tf.GradientTape() as t:
            z = model(xb, training=True)
            hard = wce(z, yb)
            if use_kd:
                pt = tf.nn.softmax(ztb / KD_T); log_ps = tf.nn.log_softmax(z / KD_T)
                kl = tf.reduce_mean(tf.reduce_sum(pt * (tf.math.log(pt + 1e-8) - log_ps), axis=1))
                loss = KD_ALPHA * hard + (1. - KD_ALPHA) * KD_T * KD_T * kl
            else:
                loss = hard
        opt.apply_gradients(zip(t.gradient(loss, model.trainable_variables), model.trainable_variables))
        return loss

    rng, n = np.random.RandomState(seed), len(X)
    best, best_w, wait, cur_lr = 1e9, None, 0, lr
    for ep in range(max_epochs):
        t0 = time.time(); perm = rng.permutation(n)
        for i in range(0, n - BATCH + 1, BATCH):
            idx = perm[i:i+BATCH]; xb, yb = X[idx], y[idx]
            if X_extra is not None:
                xb, yb = np.concatenate([xb, X_extra[idx]]), np.concatenate([yb, y[idx]])
            ztb = zt[idx] if use_kd else np.zeros((len(idx), NCLS), "float32")
            step(tf.constant(xb), tf.constant(yb), tf.constant(ztb))
        vl = float(wce(model(tf.constant(Xv), training=False), tf.constant(yv)))
        if Xv_extra is not None:
            vl += float(wce(model(tf.constant(Xv_extra), training=False), tf.constant(yv)))
        if vl < best - 1e-4: best, best_w, wait = vl, model.get_weights(), 0
        else:
            wait += 1
            if wait % 4 == 0: cur_lr *= 0.5; opt.learning_rate = cur_lr
            if wait >= patience: break
        if (ep + 1) % 10 == 0 or ep == 0:
            print(f"[{tag}] epoch {ep+1} val_loss={vl:.4f} best={best:.4f} ({time.time()-t0:.1f}s)")
    model.set_weights(best_w)
    return model

# %% [4] AdvGAN generator (corrected objective, same as NSL Stage 3), sampled from the SMOTE'd training pool
def build_generator(d):
    inp = keras.Input(shape=(d,))
    x = layers.Dense(256, activation="relu")(inp); x = layers.Dense(256, activation="relu")(x)
    return keras.Model(inp, layers.Dense(d)(x))
def build_disc(d):
    inp = keras.Input(shape=(d,))
    x = layers.Dense(256)(inp); x = layers.LeakyReLU(0.2)(x); x = layers.Dense(256)(x); x = layers.LeakyReLU(0.2)(x)
    return keras.Model(inp, layers.Dense(1)(x))

def apply_G(G, X, eps, bs=8192):
    E = eps * MASK
    return np.vstack([np.clip(X[i:i+bs] + E * np.tanh(G(tf.constant(X[i:i+bs]), training=False).numpy()), 0., 1.)
                      for i in range(0, len(X), bs)]).astype("float32")

def train_generator(clf, eps, seed, tag="", diag_path=None):
    set_seed(seed)
    G, Dn = build_generator(D), build_disc(D)
    g_opt = keras.optimizers.Adam(1e-3, beta_1=0.5); g_opt.build(G.trainable_variables)
    d_opt = keras.optimizers.Adam(1e-4, beta_1=0.5); d_opt.build(Dn.trainable_variables)
    E = tf.constant(eps * MASK)
    bce = lambda labels, logits: tf.reduce_mean(tf.nn.sigmoid_cross_entropy_with_logits(labels=labels, logits=logits))
    def gen(x): return tf.clip_by_value(x + E * tf.tanh(G(x, training=True)), 0., 1.)

    @maybe_tf
    def step(x, y):
        with tf.GradientTape() as td:
            xa = gen(x)
            ld = bce(tf.ones([tf.shape(x)[0], 1]), Dn(x, training=True)) + bce(tf.zeros([tf.shape(x)[0], 1]), Dn(xa, training=True))
        d_opt.apply_gradients(zip(td.gradient(ld, Dn.trainable_variables), Dn.trainable_variables))
        with tf.GradientTape() as tg:
            xa = gen(x); z = clf(xa, training=False); oh = tf.one_hot(y, NCLS)
            margin = tf.reduce_sum(z * oh, 1) - tf.reduce_max(z - 1e9 * oh, 1)
            lg = (LAM_CLS * tf.reduce_mean(tf.nn.relu(margin + KAPPA))
                  + LAM_GAN * bce(tf.ones([tf.shape(x)[0], 1]), Dn(xa, training=False))
                  + LAM_L1 * tf.reduce_mean(tf.abs(xa - x)))
        g_opt.apply_gradients(zip(tg.gradient(lg, G.trainable_variables), G.trainable_variables))
        return lg

    rng = np.random.RandomState(seed)
    for s in range(GEN_STEPS):
        idx = rng.randint(0, len(Xtr_sm), BATCH)
        step(tf.constant(Xtr_sm[idx]), tf.constant(ytr_sm[idx]))
    Xa = apply_G(G, Xva, eps)
    a0, a1 = float((predict(clf, Xva) == yva).mean()), float((predict(clf, Xa) == yva).mean())
    print(f"[{tag}] generator sanity (validation): acc clean={a0:.4f} -> under G={a1:.4f}")
    if diag_path: json.dump({"acc_clean": a0, "acc_under_G": a1}, open(diag_path, "w"))
    return G

# %% [5] Teacher, baselines, pipelines
def get_teacher():
    path = f"{MOD}/s3_teacher_seed{SEED}.keras"
    if os.path.exists(path): return keras.models.load_model(path, compile=False)
    t = build_model(D, widths=(512, 256, 128))
    fit_classifier(t, Xtr_sm, ytr_sm, Xva, yva, seed=SEED, tag="teacher")
    collapse_check(clean_eval(t, Xva, yva), "teacher(val)"); t.save(path); return t

teacher = get_teacher()
zt_tr = teacher.predict(Xtr_sm, batch_size=8192, verbose=0).astype("float32")
teacher_report = {"val": clean_eval(teacher, Xva, yva), "test": clean_eval(teacher, Xte, yte), "params": int(teacher.count_params())}
print("TEACHER:", "val acc", round(teacher_report["val"]["acc"], 4), "| test acc", round(teacher_report["test"]["acc"], 4),
      "| test macro-F1", round(teacher_report["test"]["macro_f1"], 4))

def get_baseline(distil):
    path = f"{MOD}/s3_base_{'kd' if distil else 'nokd'}_seed{SEED}.keras"
    if os.path.exists(path): return keras.models.load_model(path, compile=False)
    m = build_model(D)
    fit_classifier(m, Xtr_sm, ytr_sm, Xva, yva, zt=zt_tr if distil else None, seed=SEED, tag=f"base_{'kd' if distil else 'nokd'}")
    collapse_check(clean_eval(m, Xva, yva), f"base_{'kd' if distil else 'nokd'}(val)"); m.save(path); return m

def run_pipeline(distil, mode, eps, rounds=ROUNDS):
    ptag = f"{'kd' if distil else 'nokd'}_{mode}"
    prev = get_baseline(distil); clfs, G_fixed = {0: prev}, None
    for r in range(1, rounds + 1):
        cpath = f"{MOD}/s3_{ptag}_eps{eps}_r{r}_seed{SEED}.keras"; gpath = f"{MOD}/s3_gen_{ptag}_eps{eps}_r{r}_seed{SEED}.keras"
        if os.path.exists(cpath):
            clfs[r] = prev = keras.models.load_model(cpath, compile=False)
            if mode == "fixed_gen" and r == 1: G_fixed = keras.models.load_model(gpath, compile=False)
            continue
        if mode == "feedback" or r == 1:
            G = train_generator(prev, eps, tag=f"{ptag} eps={eps} r{r}", seed=SEED * 100 + r,
                                diag_path=gpath.replace(".keras", ".json")); G.save(gpath)
            if r == 1: G_fixed = G
        else:
            G = G_fixed
        Xa_tr, Xa_va = apply_G(G, Xtr_sm, eps), apply_G(G, Xva, eps)
        new = build_model(D); new.set_weights(prev.get_weights())
        fit_classifier(new, Xtr_sm, ytr_sm, Xva, yva, X_extra=Xa_tr, Xv_extra=Xa_va, lr=FEEDBACK_LR, max_epochs=30,
                       seed=SEED * 100 + r, tag=f"{ptag} eps={eps} r{r}")
        collapse_check(clean_eval(new, Xva, yva), f"{ptag} r{r}(val)")
        new.save(cpath); clfs[r] = prev = new
    return clfs

def val_score(m):
    vr = robust_eval(m, Xva, yva, MASK, EPS_SELECT, steps=10, restarts=1)
    return {"clean": clean_eval(m, Xva, yva)["acc"], "robust_mean": float(np.mean([vr[str(e)]["worst"] for e in EPS_SELECT]))}

# %% [6] Select eps_max on VALIDATION (main pipeline: distilled + feedback, evaluated at round 2)
und_val = json.load(open(f"{RES}/stage2b_val_seed{SEED}.json")) if os.path.exists(f"{RES}/stage2b_val_seed{SEED}.json") else None
und_val_clean = clean_eval(keras.models.load_model(f"{MOD}/undefended_smote_seed{SEED}.keras", compile=False), Xva, yva)["acc"]

grid = {}
for eps in EPS_TRAIN:
    clfs = run_pipeline(True, "feedback", eps)
    grid[eps] = {"models": clfs, "val": {r: val_score(clfs[r]) for r in clfs}}
    print(f"VAL eps_max={eps}: " + ", ".join(f"r{r}: clean={v['clean']:.4f} robust={v['robust_mean']:.4f}" for r, v in grid[eps]["val"].items()))
elig = [e for e in EPS_TRAIN if und_val_clean - grid[e]["val"][2]["clean"] <= VAL_CLEAN_TOL]
pool = elig if elig else EPS_TRAIN
if not elig: print("WARNING: no eps_max within tolerance at round 2")
eps_sel = max(pool, key=lambda e: (round(grid[e]["val"][2]["robust_mean"], 6), -e))
print("SELECTED generator eps_max:", eps_sel)
json.dump(jsonable({"selected_eps": eps_sel, "teacher": teacher_report, "val_grid": {e: grid[e]["val"] for e in EPS_TRAIN}}),
          open(f"{RES}/stage3_selection_seed{SEED}.json", "w"), indent=1)

# %% [7] Remaining pipelines at the selected eps_max, then TEST evaluation of every (pipeline, round)
all_models = {("kd_feedback", r): m for r, m in grid[eps_sel]["models"].items()}
for distil, mode in [(True, "fixed_gen"), (False, "feedback")]:
    for r, m in run_pipeline(distil, mode, eps_sel).items():
        all_models[(f"{'kd' if distil else 'nokd'}_{mode}", r)] = m

test_res = {}
for (ptag, r), m in sorted(all_models.items()):
    tc = clean_eval(m, Xte, yte)
    if r > 0: collapse_check(tc, f"{ptag}_r{r}(test)")
    test_res[f"{ptag}_r{r}"] = {"clean": tc, "robust": robust_eval(m, Xte, yte, MASK, EPS_EVAL)}
    json.dump(jsonable({"eps_max": eps_sel, "test": test_res}), open(f"{RES}/stage3_test_seed{SEED}.json", "w"), indent=1)

s2b = json.load(open(f"{RES}/stage2b_test_seed{SEED}.json"))
print(f"\n=== TEST worst-case robust accuracy ===")
print(f"{'model':<22}{'clean':>8}{'mF1':>8}" + "".join(f"{('e='+str(e)):>8}" for e in EPS_EVAL))
for name, key in [("undefended (S2)", "undefended"), (f"FGSM-AT eps={s2b['selected']['fgsm']}", f"fgsm_eps{s2b['selected']['fgsm']}"),
                  (f"PGD-AT eps={s2b['selected']['pgd']}", f"pgd_eps{s2b['selected']['pgd']}")]:
    r_, c_ = s2b["test"][key]["robust"], s2b["test"][key]["clean"]
    print(f"{name:<22}{c_['acc']:>8.4f}{c_['macro_f1']:>8.4f}" + "".join(f"{r_[str(e)]['worst']:>8.4f}" for e in EPS_EVAL))
for key in sorted(test_res):
    r_, c_ = test_res[key]["robust"], test_res[key]["clean"]
    print(f"{key:<22}{c_['acc']:>8.4f}{c_['macro_f1']:>8.4f}" + "".join(f"{r_[str(e)]['worst']:>8.4f}" for e in EPS_EVAL))
print("\nSaved:", f"{RES}/stage3_test_seed{SEED}.json")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
post-SMOTE train class counts: [1466120   50000  225163   63486   50000]
[teacher] epoch 1 val_loss=0.0195 best=0.0195 (5.4s)
[teacher] epoch 10 val_loss=0.0081 best=0.0081 (4.3s)
[teacher] epoch 20 val_loss=0.0079 best=0.0078 (4.3s)
[teacher] epoch 30 val_loss=0.0077 best=0.0069 (4.3s)
[teacher(val)] collapse check passed
TEACHER: val acc 0.996 | test acc 0.9961 | test macro-F1 0.9175
[base_kd] epoch 1 val_loss=0.0251 best=0.0251 (5.5s)
[base_kd] epoch 10 val_loss=0.0093 best=0.0093 (4.7s)
[base_kd] epoch 20 val_loss=0.0085 best=0.0077 (4.7s)
[base_kd(val)] collapse check passed
[kd_feedback eps=0.02 r1] generator sanity (validation): acc clean=0.9952 -> under G=0.9510
[kd_feedback eps=0.02 r1] epoch 1 val_loss=0.0602 best=0.0602 (6.0s)
[kd_feedback eps=0.02 r1] epoch 10 val_loss=0.0178 best=0.0170 (5.2s)
[kd_feedback r1(val)] collapse check passed
[kd_feedb

In [3]:
# CICIDS2017 v2 - Stage 4a: 5-seed robustness study (seeds 0-4), mirroring nsl_stage4a_seeds.py exactly in
# protocol: the SAME train/val/test split and the SAME SMOTE'd training pool (random_state=0, fixed) are used
# for every seed - only model/generator weight initialization and minibatch order vary by seed. Reuses the
# eps_train (fgsm/pgd) selected on seed-0 validation in Stage 2b and the eps_max selected on seed-0 validation
# in Stage 3 - selection is NOT repeated per seed (same rule as NSL-KDD). Trains, per seed: undefended, fgsm_at,
# pgd_at, teacher, kd_feedback (r0-5), kd_fixed_gen (r0-5), nokd_feedback (r0-5). Collapse-guard after every
# training stage. Fully resumable (every artifact checks os.path.exists before retraining/regenerating).
#
# fit_classifier_at and fit_classifier_kd below are copied verbatim (only SEED->seed made an explicit arg) from
# cicids_stage2b_at_sweep.py and cicids_stage3_a3idps.py respectively, and are NOT merged into one function -
# merging AT and KD training paths would be new, untested logic, and neither Stage 2b nor Stage 3 ever combined
# them, so keeping two faithful copies is the safer choice given we cannot afford to introduce a new bug here.

# %% [0] Setup
from google.colab import drive
drive.mount('/content/drive')
import os, json, random, time
import numpy as np, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
from imblearn.over_sampling import SMOTE

BASE = "/content/drive/MyDrive/A3IDPS_COLAB"
DATA = f"{BASE}/data_processed/cicids_v2"; MOD = f"{BASE}/models/cicids_v2"; RES = f"{BASE}/results/cicids_v2"
os.makedirs(MOD, exist_ok=True); os.makedirs(RES, exist_ok=True)

d = np.load(f"{DATA}/cicids_v2_primary.npz")
Xtr, Xva, Xte = [d[k].astype("float32") for k in ("Xtr", "Xva", "Xte")]
ytr, yva, yte = [d[k].astype("int32") for k in ("ytr", "yva", "yte")]
D, NCLS = Xtr.shape[1], 5
CLASS_NAMES = ["BENIGN", "Brute Force", "DoS/DDoS", "PortScan", "Other"]
MASK = np.ones(D, "float32")   # all 78 CICIDS features are numeric - no categorical columns to freeze

SEEDS = [0, 1, 2, 3, 4]
ROUNDS = 5
EPS_EVAL, EPS_SELECT = [0.01, 0.02, 0.05, 0.10, 0.15], [0.01, 0.02, 0.05, 0.10]
CLASS_WEIGHT_CAP, VAL_CLEAN_TOL, BATCH = 20.0, 0.05, 2048
BASE_LR, FEEDBACK_LR = 3e-4, 1.5e-4
KD_ALPHA, KD_T = 0.7, 3.0
GEN_STEPS, LAM_CLS, LAM_L1, LAM_GAN, KAPPA = 1200, 20.0, 0.5, 1.0, 1.0
FAST = True

def maybe_tf(fn): return tf.function(fn, reduce_retracing=True) if FAST else fn
def set_seed(s): random.seed(s); np.random.seed(s); tf.keras.utils.set_random_seed(s)
def jsonable(o):
    if isinstance(o, dict): return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [jsonable(v) for v in o]
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return float(o)
    return o

# %% [1] SMOTE the training pool ONCE - identical across all seeds (random_state=0, fixed), so every seed trains
# on exactly the same data; only weight init / batch order differ by seed. This matches the NSL-KDD 5-seed
# protocol, where the data split was fixed at seed 42 for every seed and only model randomness varied.
SMOTE_TARGET = 50_000
counts = np.bincount(ytr)
strategy = {c: SMOTE_TARGET for c in range(len(counts)) if counts[c] < SMOTE_TARGET}
Xtr_sm, ytr_sm = SMOTE(random_state=0, k_neighbors=5, sampling_strategy=strategy).fit_resample(Xtr, ytr)
Xtr_sm = np.clip(Xtr_sm, 0.0, 1.0).astype("float32"); ytr_sm = ytr_sm.astype("int32")
print("post-SMOTE train class counts (shared across all 5 seeds):", np.bincount(ytr_sm))
cw = np.minimum(compute_class_weight("balanced", classes=np.arange(NCLS), y=ytr_sm), CLASS_WEIGHT_CAP).astype("float32")

# %% [2] Reuse seed-0-selected hyperparameters (frozen; NOT re-selected per seed)
sel_at = json.load(open(f"{RES}/stage2b_selected_seed0.json"))          # {"fgsm": e, "pgd": e}
sel_gen = json.load(open(f"{RES}/stage3_selection_seed0.json"))        # {"selected_eps": e, ...}
EPS_FGSM, EPS_PGD, EPS_MAX = sel_at["fgsm"], sel_at["pgd"], sel_gen["selected_eps"]
print(f"Reusing seed-0 selections: fgsm eps_train={EPS_FGSM}  pgd eps_train={EPS_PGD}  generator eps_max={EPS_MAX}")

# %% [3] Model / attack / eval / collapse-guard - identical to Stage 2b and Stage 3
def build_model(d, widths=(256, 128, 64)):
    inp = keras.Input(shape=(d,))
    x = layers.Dense(widths[0])(inp); x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)
    x = layers.Dense(widths[1])(x);   x = layers.Dropout(0.3)(x);          x = layers.Activation("relu")(x)
    x = layers.Dense(widths[2])(x);   x = layers.Dropout(0.3)(x);          x = layers.Activation("relu")(x)
    return keras.Model(inp, layers.Dense(NCLS)(x))

def make_attacks(model, mask):
    M = tf.constant(mask, tf.float32)
    def loss_fn(z, y, margin):
        if margin:
            oh = tf.one_hot(y, NCLS)
            return tf.reduce_max(z - 1e9 * oh, axis=1) - tf.reduce_sum(z * oh, axis=1)
        return tf.nn.sparse_softmax_cross_entropy_with_logits(labels=y, logits=z)
    @tf.function
    def fgsm(x0, y, eps):
        with tf.GradientTape() as t:
            t.watch(x0); s = tf.reduce_sum(loss_fn(model(x0, training=False), y, False))
        return tf.clip_by_value(x0 + eps * M * tf.sign(t.gradient(s, x0)), 0., 1.)
    @tf.function
    def pgd(x0, y, eps, steps, restarts, margin):
        E = eps * M
        best_x, best_l = x0, tf.fill(tf.shape(y), -1e30)
        for r in range(restarts):
            x = x0 if r == 0 else tf.clip_by_value(x0 + tf.random.uniform(tf.shape(x0), -1., 1.) * E, 0., 1.)
            for s_ in range(steps):
                alpha = (1.0 if s_ < steps // 2 else 0.25) * E / 4.0
                with tf.GradientTape() as t:
                    t.watch(x); s = tf.reduce_sum(loss_fn(model(x, training=False), y, margin))
                x = tf.clip_by_value(tf.clip_by_value(x + alpha * tf.sign(t.gradient(s, x)), x0 - E, x0 + E), 0., 1.)
                l_new = loss_fn(model(x, training=False), y, margin)
                upd = l_new > best_l
                best_x, best_l = tf.where(upd[:, None], x, best_x), tf.where(upd, l_new, best_l)
        return best_x
    return {"fgsm": fgsm, "pgd": pgd}

def run_attack(fn, X, y, bs=4096, **kw):
    return np.vstack([fn(tf.constant(X[i:i+bs]), tf.constant(y[i:i+bs]), **kw).numpy() for i in range(0, len(X), bs)])
def predict(model, X): return np.argmax(model.predict(X, batch_size=8192, verbose=0), axis=1)

def clean_eval(model, X, y):
    p = predict(model, X); present = sorted(np.unique(y).tolist())
    return {"acc": float(accuracy_score(y, p)),
            "macro_f1": float(f1_score(y, p, average="macro", labels=present, zero_division=0)),
            "per_class_f1": {CLASS_NAMES[c]: float(f1_score(y == c, p == c, zero_division=0)) for c in present}}

def robust_eval(model, X, y, mask, eps_list, steps=20, restarts=3):
    A = make_attacks(model, mask); out = {}; present = sorted(np.unique(y).tolist())
    for e in eps_list:
        et = tf.constant(e, tf.float32)
        c_f = predict(model, run_attack(A["fgsm"], X, y, eps=et)) == y
        c_c = predict(model, run_attack(A["pgd"], X, y, eps=et, steps=steps, restarts=restarts, margin=False)) == y
        c_m = predict(model, run_attack(A["pgd"], X, y, eps=et, steps=steps, restarts=restarts, margin=True)) == y
        w = c_f & c_c & c_m
        out[str(e)] = {"fgsm": float(c_f.mean()), "pgd_ce": float(c_c.mean()), "pgd_margin": float(c_m.mean()),
                       "worst": float(w.mean()), "worst_recall": {CLASS_NAMES[c]: float(w[y == c].mean()) for c in present}}
    return out

def collapse_check(clean, tag):
    bad = [c for c in ("Brute Force", "DoS/DDoS", "PortScan") if clean["per_class_f1"].get(c, 0.0) == 0.0]
    if bad: raise RuntimeError(f"[{tag}] COLLAPSED: F1=0 for {bad} - stop and inspect before continuing")
    print(f"[{tag}] collapse check passed")

# %% [4a] Classifier training - undefended / FGSM-AT / PGD-AT (verbatim from Stage 2b, seed made an explicit arg)
def fit_classifier_at(model, X, y, Xv, yv, seed, at=None, lr=BASE_LR, max_epochs=60, patience=8, tag=""):
    set_seed(seed)
    opt = keras.optimizers.Adam(lr); opt.build(model.trainable_variables)
    A = make_attacks(model, MASK) if at is not None else None
    et = tf.constant(at[1] if at else 0., tf.float32)
    cwt = tf.constant(cw)
    def wce(z, yy): return tf.reduce_mean(tf.nn.sparse_softmax_cross_entropy_with_logits(labels=yy, logits=z) * tf.gather(cwt, yy))

    @maybe_tf
    def step(xb, yb):
        with tf.GradientTape() as t:
            loss = wce(model(xb, training=True), yb)
        opt.apply_gradients(zip(t.gradient(loss, model.trainable_variables), model.trainable_variables))
        return loss

    rng, n = np.random.RandomState(seed), len(X)
    best, best_w, wait, cur_lr = 1e9, None, 0, lr
    for ep in range(max_epochs):
        t0 = time.time(); perm = rng.permutation(n)
        for i in range(0, n - BATCH + 1, BATCH):
            idx = perm[i:i+BATCH]; xt, yt = tf.constant(X[idx]), tf.constant(y[idx])
            if at is not None:
                xa = A["fgsm"](xt, yt, eps=et) if at[0] == "fgsm" else A["pgd"](xt, yt, eps=et, steps=10, restarts=1, margin=False)
                xt, yt = tf.concat([xt, xa], 0), tf.concat([yt, yt], 0)
            step(xt, yt)
        vl = float(wce(model(tf.constant(Xv), training=False), tf.constant(yv)))
        if at is not None:
            xv = run_attack(A["fgsm"], Xv, yv, eps=et)
            vl += float(wce(model(tf.constant(xv), training=False), tf.constant(yv)))
        if vl < best - 1e-4: best, best_w, wait = vl, model.get_weights(), 0
        else:
            wait += 1
            if wait % 4 == 0: cur_lr *= 0.5; opt.learning_rate = cur_lr
            if wait >= patience: break
        if (ep + 1) % 10 == 0 or ep == 0:
            print(f"[{tag}] epoch {ep+1} val_loss={vl:.4f} best={best:.4f} ({time.time()-t0:.1f}s)")
    model.set_weights(best_w)
    return model

# %% [4b] Classifier training - plain / distilled / +feedback samples (verbatim from Stage 3, seed made an explicit arg)
def fit_classifier_kd(model, X, y, Xv, yv, seed, X_extra=None, Xv_extra=None, zt=None,
                      lr=BASE_LR, max_epochs=60, patience=8, tag=""):
    set_seed(seed)
    opt = keras.optimizers.Adam(lr); opt.build(model.trainable_variables)
    use_kd, cwt = zt is not None, tf.constant(cw)
    def wce(z, yy): return tf.reduce_mean(tf.nn.sparse_softmax_cross_entropy_with_logits(labels=yy, logits=z) * tf.gather(cwt, yy))

    @maybe_tf
    def step(xb, yb, ztb):
        with tf.GradientTape() as t:
            z = model(xb, training=True)
            hard = wce(z, yb)
            if use_kd:
                pt = tf.nn.softmax(ztb / KD_T); log_ps = tf.nn.log_softmax(z / KD_T)
                kl = tf.reduce_mean(tf.reduce_sum(pt * (tf.math.log(pt + 1e-8) - log_ps), axis=1))
                loss = KD_ALPHA * hard + (1. - KD_ALPHA) * KD_T * KD_T * kl
            else:
                loss = hard
        opt.apply_gradients(zip(t.gradient(loss, model.trainable_variables), model.trainable_variables))
        return loss

    rng, n = np.random.RandomState(seed), len(X)
    best, best_w, wait, cur_lr = 1e9, None, 0, lr
    for ep in range(max_epochs):
        t0 = time.time(); perm = rng.permutation(n)
        for i in range(0, n - BATCH + 1, BATCH):
            idx = perm[i:i+BATCH]; xb, yb = X[idx], y[idx]
            ztb = zt[idx] if use_kd else np.zeros((len(idx), NCLS), "float32")
            if X_extra is not None:
                xb, yb = np.concatenate([xb, X_extra[idx]]), np.concatenate([yb, y[idx]])
                # BUGFIX (found when seed 1 crashed with "2048 vs 4096"): xb/yb double to cover the clean +
                # generator-perturbed copies, so ztb (the teacher's soft labels used by the KD loss) must double
                # the same way, giving the perturbed copy the same teacher target as its clean counterpart.
                if use_kd:
                    ztb = np.concatenate([ztb, ztb])
            step(tf.constant(xb), tf.constant(yb), tf.constant(ztb))
        vl = float(wce(model(tf.constant(Xv), training=False), tf.constant(yv)))
        if Xv_extra is not None:
            vl += float(wce(model(tf.constant(Xv_extra), training=False), tf.constant(yv)))
        if vl < best - 1e-4: best, best_w, wait = vl, model.get_weights(), 0
        else:
            wait += 1
            if wait % 4 == 0: cur_lr *= 0.5; opt.learning_rate = cur_lr
            if wait >= patience: break
        if (ep + 1) % 10 == 0 or ep == 0:
            print(f"[{tag}] epoch {ep+1} val_loss={vl:.4f} best={best:.4f} ({time.time()-t0:.1f}s)")
    model.set_weights(best_w)
    return model

# %% [5] AdvGAN generator (verbatim from Stage 3, seed made an explicit arg)
def build_generator(d):
    inp = keras.Input(shape=(d,))
    x = layers.Dense(256, activation="relu")(inp); x = layers.Dense(256, activation="relu")(x)
    return keras.Model(inp, layers.Dense(d)(x))
def build_disc(d):
    inp = keras.Input(shape=(d,))
    x = layers.Dense(256)(inp); x = layers.LeakyReLU(0.2)(x); x = layers.Dense(256)(x); x = layers.LeakyReLU(0.2)(x)
    return keras.Model(inp, layers.Dense(1)(x))

def apply_G(G, X, eps, bs=8192):
    E = eps * MASK
    return np.vstack([np.clip(X[i:i+bs] + E * np.tanh(G(tf.constant(X[i:i+bs]), training=False).numpy()), 0., 1.)
                      for i in range(0, len(X), bs)]).astype("float32")

def train_generator(clf, eps, seed, tag="", diag_path=None):
    set_seed(seed)
    G, Dn = build_generator(D), build_disc(D)
    g_opt = keras.optimizers.Adam(1e-3, beta_1=0.5); g_opt.build(G.trainable_variables)
    d_opt = keras.optimizers.Adam(1e-4, beta_1=0.5); d_opt.build(Dn.trainable_variables)
    E = tf.constant(eps * MASK)
    bce = lambda labels, logits: tf.reduce_mean(tf.nn.sigmoid_cross_entropy_with_logits(labels=labels, logits=logits))
    def gen(x): return tf.clip_by_value(x + E * tf.tanh(G(x, training=True)), 0., 1.)

    @maybe_tf
    def step(x, y):
        with tf.GradientTape() as td:
            xa = gen(x)
            ld = bce(tf.ones([tf.shape(x)[0], 1]), Dn(x, training=True)) + bce(tf.zeros([tf.shape(x)[0], 1]), Dn(xa, training=True))
        d_opt.apply_gradients(zip(td.gradient(ld, Dn.trainable_variables), Dn.trainable_variables))
        with tf.GradientTape() as tg:
            xa = gen(x); z = clf(xa, training=False); oh = tf.one_hot(y, NCLS)
            margin = tf.reduce_sum(z * oh, 1) - tf.reduce_max(z - 1e9 * oh, 1)
            lg = (LAM_CLS * tf.reduce_mean(tf.nn.relu(margin + KAPPA))
                  + LAM_GAN * bce(tf.ones([tf.shape(x)[0], 1]), Dn(xa, training=False))
                  + LAM_L1 * tf.reduce_mean(tf.abs(xa - x)))
        g_opt.apply_gradients(zip(tg.gradient(lg, G.trainable_variables), G.trainable_variables))
        return lg

    rng = np.random.RandomState(seed)
    for s in range(GEN_STEPS):
        idx = rng.randint(0, len(Xtr_sm), BATCH)
        step(tf.constant(Xtr_sm[idx]), tf.constant(ytr_sm[idx]))
    Xa = apply_G(G, Xva, eps)
    a0, a1 = float((predict(clf, Xva) == yva).mean()), float((predict(clf, Xa) == yva).mean())
    print(f"[{tag}] generator sanity (validation): acc clean={a0:.4f} -> under G={a1:.4f}")
    if diag_path: json.dump({"acc_clean": a0, "acc_under_G": a1}, open(diag_path, "w"))
    return G

# %% [6] Per-seed pipelines - same model paths/naming as Stage 2b / Stage 3 so seed 0 loads the existing files
def get_undefended(seed):
    path = f"{MOD}/undefended_smote_seed{seed}.keras"
    if os.path.exists(path): return keras.models.load_model(path, compile=False)
    m = build_model(D)
    fit_classifier_at(m, Xtr_sm, ytr_sm, Xva, yva, seed=seed, at=None, tag=f"undefended seed{seed}")
    collapse_check(clean_eval(m, Xva, yva), f"undefended seed{seed}(val)"); m.save(path); return m

def get_at(method, seed):
    eps = EPS_FGSM if method == "fgsm" else EPS_PGD
    path = f"{MOD}/{method}_eps{eps}_seed{seed}.keras"
    if os.path.exists(path): return keras.models.load_model(path, compile=False)
    m = build_model(D)
    fit_classifier_at(m, Xtr_sm, ytr_sm, Xva, yva, seed=seed, at=(method, eps), tag=f"{method}_at seed{seed}")
    collapse_check(clean_eval(m, Xva, yva), f"{method}_at seed{seed}(val)"); m.save(path); return m

def get_teacher(seed):
    path = f"{MOD}/s3_teacher_seed{seed}.keras"
    if os.path.exists(path): return keras.models.load_model(path, compile=False)
    t = build_model(D, widths=(512, 256, 128))
    fit_classifier_kd(t, Xtr_sm, ytr_sm, Xva, yva, seed=seed, tag=f"teacher seed{seed}")
    collapse_check(clean_eval(t, Xva, yva), f"teacher seed{seed}(val)"); t.save(path); return t

def get_baseline(distil, seed, zt_tr):
    path = f"{MOD}/s3_base_{'kd' if distil else 'nokd'}_seed{seed}.keras"
    if os.path.exists(path): return keras.models.load_model(path, compile=False)
    m = build_model(D)
    fit_classifier_kd(m, Xtr_sm, ytr_sm, Xva, yva, seed=seed, zt=zt_tr if distil else None,
                      tag=f"base_{'kd' if distil else 'nokd'} seed{seed}")
    collapse_check(clean_eval(m, Xva, yva), f"base_{'kd' if distil else 'nokd'} seed{seed}(val)"); m.save(path); return m

def run_pipeline(distil, mode, eps, seed, zt_tr, rounds=ROUNDS):
    ptag = f"{'kd' if distil else 'nokd'}_{mode}"
    prev = get_baseline(distil, seed, zt_tr); clfs, G_fixed = {0: prev}, None
    for r in range(1, rounds + 1):
        cpath = f"{MOD}/s3_{ptag}_eps{eps}_r{r}_seed{seed}.keras"; gpath = f"{MOD}/s3_gen_{ptag}_eps{eps}_r{r}_seed{seed}.keras"
        if os.path.exists(cpath):
            clfs[r] = prev = keras.models.load_model(cpath, compile=False)
            if mode == "fixed_gen" and r == 1: G_fixed = keras.models.load_model(gpath, compile=False)
            continue
        if mode == "feedback" or r == 1:
            G = train_generator(prev, eps, tag=f"{ptag} eps={eps} r{r} seed{seed}", seed=seed * 100 + r,
                                diag_path=gpath.replace(".keras", ".json")); G.save(gpath)
            if r == 1: G_fixed = G
        else:
            G = G_fixed
        Xa_tr, Xa_va = apply_G(G, Xtr_sm, eps), apply_G(G, Xva, eps)
        new = build_model(D); new.set_weights(prev.get_weights())
        # NOTE: zt is deliberately NOT passed here, matching cicids_stage3_a3idps.py's run_pipeline (and
        # nsl_stage3.py's) exactly - in the original design, distillation only shapes the round-0 baseline's
        # warm-start weights; per-round feedback retraining is plain hard-label + class-weighted training on
        # clean+generated samples for every pipeline (kd_feedback included). An earlier draft of this script
        # added zt here to make KD apply continuously through the feedback rounds - that is a real, separate
        # design question worth deciding deliberately, but it must not be smuggled in as a silent deviation from
        # what already produced the existing seed-0 numbers, so it is reverted here to keep seed 0 comparable.
        fit_classifier_kd(new, Xtr_sm, ytr_sm, Xva, yva, seed=seed * 100 + r, X_extra=Xa_tr, Xv_extra=Xa_va,
                          lr=FEEDBACK_LR, max_epochs=30, tag=f"{ptag} eps={eps} r{r} seed{seed}")
        collapse_check(clean_eval(new, Xva, yva), f"{ptag} r{r} seed{seed}(val)")
        new.save(cpath); clfs[r] = prev = new
    return clfs

# %% [7] Run all 5 seeds, evaluate every model on TEST, save per-seed JSON as we go (resumable)
ALL = {}   # ALL[seed][tag] = {"clean":..., "robust":...}
for seed in SEEDS:
    seed_out_path = f"{RES}/stage4a_seed{seed}.json"
    if os.path.exists(seed_out_path):
        ALL[seed] = json.load(open(seed_out_path)); print(f"seed {seed}: loaded existing results"); continue

    print(f"\n========== SEED {seed} ==========")
    und = get_undefended(seed)
    fgsm_at = get_at("fgsm", seed)
    pgd_at = get_at("pgd", seed)
    teacher = get_teacher(seed)
    zt_tr = teacher.predict(Xtr_sm, batch_size=8192, verbose=0).astype("float32")

    kd_feedback = run_pipeline(True, "feedback", EPS_MAX, seed, zt_tr)
    kd_fixed_gen = run_pipeline(True, "fixed_gen", EPS_MAX, seed, zt_tr)
    nokd_feedback = run_pipeline(False, "feedback", EPS_MAX, seed, zt_tr)

    seed_res = {}
    for tag, m in [("undefended", und), (f"fgsm_at_eps{EPS_FGSM}", fgsm_at), (f"pgd_at_eps{EPS_PGD}", pgd_at),
                   ("teacher", teacher)]:
        tc = clean_eval(m, Xte, yte); collapse_check(tc, f"{tag} seed{seed}(test)")
        seed_res[tag] = {"clean": tc, "robust": robust_eval(m, Xte, yte, MASK, EPS_EVAL)}
    for ptag, clfs in [("kd_feedback", kd_feedback), ("kd_fixed_gen", kd_fixed_gen), ("nokd_feedback", nokd_feedback)]:
        for r, m in clfs.items():
            tc = clean_eval(m, Xte, yte)
            if r > 0: collapse_check(tc, f"{ptag}_r{r} seed{seed}(test)")
            seed_res[f"{ptag}_r{r}"] = {"clean": tc, "robust": robust_eval(m, Xte, yte, MASK, EPS_EVAL)}

    json.dump(jsonable(seed_res), open(seed_out_path, "w"), indent=1)
    ALL[seed] = seed_res
    print(f"seed {seed}: done, saved to {seed_out_path}")

# %% [8] Aggregate mean +- std across seeds, and paired differences (per-seed, then averaged)
TAGS = ["undefended", f"fgsm_at_eps{EPS_FGSM}", f"pgd_at_eps{EPS_PGD}"] + \
       [f"kd_feedback_r{r}" for r in range(ROUNDS + 1)] + \
       [f"kd_fixed_gen_r{r}" for r in range(ROUNDS + 1)] + \
       [f"nokd_feedback_r{r}" for r in range(ROUNDS + 1)]

def agg(tag, eps=None):
    vals = []
    for s in SEEDS:
        r = ALL[s][tag]
        vals.append(r["clean"]["acc"] if eps is None else r["robust"][str(eps)]["worst"])
    return float(np.mean(vals)), float(np.std(vals)), vals

print(f"\n=== 5-SEED TEST RESULTS (mean +- std over seeds {SEEDS}), worst-case robust accuracy ===")
print(f"{'model':<24}{'clean':>12}" + "".join(f"{('e='+str(e)):>13}" for e in EPS_EVAL))
agg_table = {}
for tag in TAGS:
    if tag not in ALL[SEEDS[0]]: continue
    cm, cs, _ = agg(tag)
    row = {"clean": (cm, cs)}
    line = f"{tag:<24}{cm*100:>8.1f}+-{cs*100:<3.1f}"
    for e in EPS_EVAL:
        m_, s_, _ = agg(tag, e); row[str(e)] = (m_, s_)
        line += f"  {m_*100:>6.1f}+-{s_*100:<3.1f}"
    print(line)
    agg_table[tag] = row

def paired(tag_a, tag_b, eps):
    diffs = [ALL[s][tag_a]["robust"][str(eps)]["worst"] - ALL[s][tag_b]["robust"][str(eps)]["worst"] for s in SEEDS]
    return float(np.mean(diffs)), float(np.std(diffs)), sum(1 for x in diffs if x > 0)

print(f"\n=== Paired differences at eps=0.05 and eps=0.10 (worst-case robust acc, kd_feedback_r{ROUNDS} vs others) ===")
comparisons = [(f"kd_feedback_r{ROUNDS}", "undefended"), (f"kd_feedback_r{ROUNDS}", f"pgd_at_eps{EPS_PGD}"),
               (f"kd_feedback_r{ROUNDS}", f"kd_fixed_gen_r{ROUNDS}"), (f"kd_feedback_r{ROUNDS}", f"nokd_feedback_r{ROUNDS}")]
paired_table = {}
for a, b in comparisons:
    entry = {}
    for e in (0.05, 0.10):
        m_, s_, npos = paired(a, b, e)
        entry[str(e)] = {"mean": m_, "std": s_, "n_positive": npos}
        print(f"{a} - {b}  @eps={e}: {m_*100:+.1f}+-{s_*100:.1f} pp  ({npos}/{len(SEEDS)} seeds positive)")
    paired_table[f"{a}__minus__{b}"] = entry

json.dump(jsonable({"eps_fgsm": EPS_FGSM, "eps_pgd": EPS_PGD, "eps_max": EPS_MAX, "seeds": SEEDS,
                    "aggregate": agg_table, "paired_differences": paired_table}),
          open(f"{RES}/stage4a_summary.json", "w"), indent=1)
print(f"\nSaved: {RES}/stage4a_summary.json")

Exception ignored in: <function NpzFile.__del__ at 0x7a45fdf418a0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/numpy/lib/_npyio_impl.py", line 228, in __del__
    self.close()
  File "/usr/local/lib/python3.13/dist-packages/numpy/lib/_npyio_impl.py", line 223, in close
    self.fid.close()
OSError: [Errno 107] Transport endpoint is not connected


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
post-SMOTE train class counts (shared across all 5 seeds): [1466120   50000  225163   63486   50000]
Reusing seed-0 selections: fgsm eps_train=0.05  pgd eps_train=0.1  generator eps_max=0.05
seed 0: loaded existing results

========== SEED 1 ==========
[kd_feedback eps=0.05 r1 seed1] generator sanity (validation): acc clean=0.9953 -> under G=0.2518
[kd_feedback eps=0.05 r1 seed1] epoch 1 val_loss=0.0225 best=0.0225 (5.8s)
[kd_feedback eps=0.05 r1 seed1] epoch 10 val_loss=0.0161 best=0.0154 (4.5s)
[kd_feedback r1 seed1(val)] collapse check passed
[kd_feedback eps=0.05 r2 seed1] generator sanity (validation): acc clean=0.9957 -> under G=0.6853
[kd_feedback eps=0.05 r2 seed1] epoch 1 val_loss=0.0249 best=0.0249 (5.3s)
[kd_feedback eps=0.05 r2 seed1] epoch 10 val_loss=0.0162 best=0.0149 (4.6s)
[kd_feedback r2 seed1(val)] collapse check passed
[kd_feedback eps=0.0

In [4]:
# CICIDS2017 v2 - Stage 4b: AdvGAN-as-attack sweep, mirroring nsl_stage4b_advgan_attack.py. Purpose: answer
# Reviewer 2's "sign-error / F1-contradiction" concern with a worst-case check - train a FRESH generator, with
# several architectures/hyperparameters, whose only job is to attack each already-finished classifier (it is not
# part of that classifier's own training loop), and confirm it never beats the gradient-based white-box attacks
# (FGSM/PGD) already reported in Stage 4a. If a generator config ever attacked a model MORE effectively than the
# white-box PGD attack, that would mean the white-box numbers understate real-world risk - this script checks
# for that directly, with integrity assertions, rather than asserting it never happens.
#
# NOTE ON THE 8 CONFIGS: these mirror the *names and intent* of the 8 configs used in nsl_stage4b_advgan_attack.py
# (default / width64 / width1024 / lam5 / lam80 / steps400 / steps4800 / strong), but the exact hyperparameter
# values below are freshly chosen for CICIDS (78-dim, no categorical mask) rather than copied from the NSL file,
# which was not available to reconstruct byte-for-byte. What matters for the paper is the SWEEP METHODOLOGY and
# the integrity checks, not that the two datasets share identical generator hyperparameters.
#
# Requires cicids_stage4a_seeds.py to have produced, per seed: undefended_smote_seed{s}.keras,
# {fgsm,pgd}_eps{e}_seed{s}.keras, and s3_kd_feedback_eps{EPS_MAX}_r5_seed{s}.keras. Fully resumable.

# %% [0] Setup
from google.colab import drive
drive.mount('/content/drive')
import os, json, time
import numpy as np, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import accuracy_score

BASE = "/content/drive/MyDrive/A3IDPS_COLAB"
DATA = f"{BASE}/data_processed/cicids_v2"; MOD = f"{BASE}/models/cicids_v2"; RES = f"{BASE}/results/cicids_v2"
os.makedirs(RES, exist_ok=True)

d = np.load(f"{DATA}/cicids_v2_primary.npz")
Xte, yte = d["Xte"].astype("float32"), d["yte"].astype("int32")
D, NCLS = Xte.shape[1], 5
MASK = np.ones(D, "float32")   # no categorical columns in CICIDS - nothing to freeze

SEEDS = [0, 1, 2, 3, 4]
EPS_LIST = [0.05, 0.10]
BATCH = 2048

sel_at = json.load(open(f"{RES}/stage2b_selected_seed0.json"))
sel_gen = json.load(open(f"{RES}/stage3_selection_seed0.json"))
EPS_FGSM, EPS_PGD, EPS_MAX = sel_at["fgsm"], sel_at["pgd"], sel_gen["selected_eps"]

# model tag -> path template (must match Stage 4a's saved filenames exactly)
MODEL_PATHS = {
    "undefended":  lambda s: f"{MOD}/undefended_smote_seed{s}.keras",
    "fgsm_at":     lambda s: f"{MOD}/fgsm_eps{EPS_FGSM}_seed{s}.keras",
    "pgd_at":      lambda s: f"{MOD}/pgd_eps{EPS_PGD}_seed{s}.keras",
    "kd_feedback_r5": lambda s: f"{MOD}/s3_kd_feedback_eps{EPS_MAX}_r5_seed{s}.keras",
}

def jsonable(o):
    if isinstance(o, dict): return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [jsonable(v) for v in o]
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return float(o)
    return o

def predict(model, X): return np.argmax(model.predict(X, batch_size=8192, verbose=0), axis=1)
def acc(model, X, y): return float(accuracy_score(y, predict(model, X)))

# %% [1] Generator/discriminator builders, parameterized by width - and the 8 config grid
def build_generator(d, width):
    inp = keras.Input(shape=(d,))
    x = layers.Dense(width, activation="relu")(inp); x = layers.Dense(width, activation="relu")(x)
    return keras.Model(inp, layers.Dense(d)(x))
def build_disc(d, width=256):
    inp = keras.Input(shape=(d,))
    x = layers.Dense(width)(inp); x = layers.LeakyReLU(0.2)(x); x = layers.Dense(width)(x); x = layers.LeakyReLU(0.2)(x)
    return keras.Model(inp, layers.Dense(1)(x))

# name -> dict(width, lam_cls, lam_gan, lam_l1, steps)
CONFIGS = {
    "default":   dict(width=256,  lam_cls=20., lam_gan=1.0, lam_l1=0.5, steps=1200),
    "width64":   dict(width=64,   lam_cls=20., lam_gan=1.0, lam_l1=0.5, steps=1200),
    "width1024": dict(width=1024, lam_cls=20., lam_gan=1.0, lam_l1=0.5, steps=1200),
    "lam5":      dict(width=256,  lam_cls=5.,  lam_gan=1.0, lam_l1=0.5, steps=1200),
    "lam80":     dict(width=256,  lam_cls=80., lam_gan=1.0, lam_l1=0.5, steps=1200),
    "steps400":  dict(width=256,  lam_cls=20., lam_gan=1.0, lam_l1=0.5, steps=400),
    "steps4800": dict(width=256,  lam_cls=20., lam_gan=1.0, lam_l1=0.5, steps=4800),
    "strong":    dict(width=1024, lam_cls=80., lam_gan=0.5, lam_l1=0.25, steps=4800),   # most aggressive combination
}
KAPPA = 1.0

def apply_G(G, X, eps, bs=8192):
    E = eps * MASK
    return np.vstack([np.clip(X[i:i+bs] + E * np.tanh(G(tf.constant(X[i:i+bs]), training=False).numpy()), 0., 1.)
                      for i in range(0, len(X), bs)]).astype("float32")

def train_attack_generator(clf, Xtr_for_gen, ytr_for_gen, eps, cfg, seed):
    tf.keras.utils.set_random_seed(seed)
    G = build_generator(D, cfg["width"]); Dn = build_disc(D)
    g_opt = keras.optimizers.Adam(1e-3, beta_1=0.5); g_opt.build(G.trainable_variables)
    d_opt = keras.optimizers.Adam(1e-4, beta_1=0.5); d_opt.build(Dn.trainable_variables)
    E = tf.constant(eps * MASK)
    bce = lambda labels, logits: tf.reduce_mean(tf.nn.sigmoid_cross_entropy_with_logits(labels=labels, logits=logits))
    def gen(x): return tf.clip_by_value(x + E * tf.tanh(G(x, training=True)), 0., 1.)

    @tf.function(reduce_retracing=True)
    def step(x, y):
        with tf.GradientTape() as td:
            xa = gen(x)
            ld = bce(tf.ones([tf.shape(x)[0], 1]), Dn(x, training=True)) + bce(tf.zeros([tf.shape(x)[0], 1]), Dn(xa, training=True))
        d_opt.apply_gradients(zip(td.gradient(ld, Dn.trainable_variables), Dn.trainable_variables))
        with tf.GradientTape() as tg:
            xa = gen(x); z = clf(xa, training=False); oh = tf.one_hot(y, NCLS)
            margin = tf.reduce_sum(z * oh, 1) - tf.reduce_max(z - 1e9 * oh, 1)
            lg = (cfg["lam_cls"] * tf.reduce_mean(tf.nn.relu(margin + KAPPA))
                  + cfg["lam_gan"] * bce(tf.ones([tf.shape(x)[0], 1]), Dn(xa, training=False))
                  + cfg["lam_l1"] * tf.reduce_mean(tf.abs(xa - x)))
        g_opt.apply_gradients(zip(tg.gradient(lg, G.trainable_variables), G.trainable_variables))
        return lg

    rng = np.random.RandomState(seed); n = len(Xtr_for_gen)
    for s in range(cfg["steps"]):
        idx = rng.randint(0, n, BATCH)
        step(tf.constant(Xtr_for_gen[idx]), tf.constant(ytr_for_gen[idx]))
    return G

# %% [2] Data the generator trains against - use a fixed subsample of TEST inputs' distribution is wrong (would
# leak test into training the attack); use the held-out VALIDATION split instead, exactly as Stage 3's
# train_generator() did, then attack the TEST set with the trained generator. Re-load val split from primary npz.
Xva, yva = d["Xva"].astype("float32"), d["yva"].astype("int32")

# %% [3] Run the sweep: per seed, per model tag, per eps, per config - fully resumable via a flat results dict
OUT_PATH = f"{RES}/stage4b_results.json"
results = json.load(open(OUT_PATH)) if os.path.exists(OUT_PATH) else {}

# integrity: baseline clean test accuracy per (seed, model tag), to catch any load mismatch immediately
def get_model(seed, tag):
    return keras.models.load_model(MODEL_PATHS[tag](seed), compile=False)

for seed in SEEDS:
    seed_key = str(seed)
    results.setdefault(seed_key, {})
    for tag in MODEL_PATHS:
        model = get_model(seed, tag)
        clean_acc = acc(model, Xte, yte)
        results[seed_key].setdefault(tag, {"clean_acc": clean_acc, "configs": {}})
        # integrity check 1: clean accuracy must be stable/deterministic for this saved model (no retraining happened)
        prev = results[seed_key][tag].get("clean_acc")
        if prev is not None and abs(prev - clean_acc) > 1e-6:
            raise RuntimeError(f"seed{seed} {tag}: clean acc changed between runs ({prev} -> {clean_acc}) - model file changed underfoot")
        results[seed_key][tag]["clean_acc"] = clean_acc

        for eps in EPS_LIST:
            eps_key = str(eps)
            for cfg_name, cfg in CONFIGS.items():
                done = results[seed_key][tag]["configs"].get(cfg_name, {}).get(eps_key)
                if done is not None:
                    continue
                t0 = time.time()
                G = train_attack_generator(model, Xva, yva, eps, cfg, seed=seed * 1000 + hash(cfg_name) % 1000)
                Xa = apply_G(G, Xte, eps)

                # integrity check 2: perturbation must not exceed eps anywhere (tanh bounds it by construction,
                # but verify numerically in case of a future code change)
                max_delta = float(np.max(np.abs(Xa - Xte)))
                if max_delta > eps + 1e-4:
                    raise RuntimeError(f"seed{seed} {tag} {cfg_name} eps={eps}: perturbation {max_delta:.5f} exceeds eps bound")

                acc_under_g = acc(model, Xa, yte)
                mean_abs_delta = float(np.mean(np.abs(Xa - Xte)))
                results[seed_key].setdefault(tag, {}).setdefault("configs", {}).setdefault(cfg_name, {})[eps_key] = {
                    "acc_under_generator": acc_under_g, "mean_abs_delta": mean_abs_delta, "max_abs_delta": max_delta,
                    "seconds": round(time.time() - t0, 1)}
                print(f"seed{seed} {tag:<14} eps={eps} {cfg_name:<10} acc_under_G={acc_under_g:.4f}  "
                      f"mean|d|={mean_abs_delta:.5f} max|d|={max_delta:.5f} ({time.time()-t0:.0f}s)")
                json.dump(jsonable(results), open(OUT_PATH, "w"), indent=1)

# %% [4] Compare against the white-box worst-case from Stage 4a - flag ANY case where the generator attack is
# stronger (lower accuracy) than the white-box PGD/FGSM worst-case, which would mean Stage 4a understated risk.
s4a = {s: json.load(open(f"{RES}/stage4a_seed{s}.json")) for s in SEEDS}
S4A_TAG = {"undefended": "undefended", "fgsm_at": f"fgsm_at_eps{EPS_FGSM}", "pgd_at": f"pgd_at_eps{EPS_PGD}",
           "kd_feedback_r5": "kd_feedback_r5"}

flags = []
print(f"\n=== AdvGAN-attack accuracy vs white-box worst-case (accuracy %, lower = stronger attack) ===")
for eps in EPS_LIST:
    print(f"\n--- eps={eps} ---")
    print(f"{'model':<16}{'default cfg':>12}{'worst cfg':>12}{'white-box':>12}   worst config per seed")
    for tag in MODEL_PATHS:
        default_accs, worst_accs, wb_accs, worst_cfgs = [], [], [], []
        for seed in SEEDS:
            cfgs = results[str(seed)][tag]["configs"]
            per_cfg = {c: cfgs[c][str(eps)]["acc_under_generator"] for c in CONFIGS if str(eps) in cfgs.get(c, {})}
            default_accs.append(per_cfg["default"])
            worst_name = min(per_cfg, key=per_cfg.get); worst_accs.append(per_cfg[worst_name]); worst_cfgs.append(worst_name)
            wb = s4a[seed][S4A_TAG[tag]]["robust"][str(eps)]["worst"]; wb_accs.append(wb)
            if per_cfg[worst_name] < wb - 1e-6:
                flags.append(f"seed{seed} {tag} eps={eps}: generator ({worst_name})={per_cfg[worst_name]:.4f} "
                             f"< white-box worst-case={wb:.4f}")
        print(f"{tag:<16}{np.mean(default_accs)*100:>9.1f}+-{np.std(default_accs)*100:<3.1f}"
              f"{np.mean(worst_accs)*100:>9.1f}+-{np.std(worst_accs)*100:<3.1f}"
              f"{np.mean(wb_accs)*100:>9.1f}+-{np.std(wb_accs)*100:<3.1f}   {worst_cfgs}")

if flags:
    print(f"\n*** {len(flags)} FLAG(S): generator attack beat the white-box worst-case - inspect before trusting Stage 4a numbers ***")
    for f in flags: print("  " + f)
else:
    print(f"\nNo flags: across {len(SEEDS)} seeds x {len(MODEL_PATHS)} models x {len(EPS_LIST)} eps x {len(CONFIGS)} configs "
          f"= {len(SEEDS)*len(MODEL_PATHS)*len(EPS_LIST)*len(CONFIGS)} generator-attack runs, the white-box "
          f"FGSM/PGD worst-case was never beaten by the AdvGAN attack sweep.")

json.dump({"flags": flags}, open(f"{RES}/stage4b_flags.json", "w"), indent=1)
print(f"\nSaved: {OUT_PATH} and {RES}/stage4b_flags.json")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
seed0 undefended     eps=0.05 default    acc_under_G=0.6893  mean|d|=0.03113 max|d|=0.05000 (9s)
seed0 undefended     eps=0.05 width64    acc_under_G=0.7491  mean|d|=0.03029 max|d|=0.05000 (9s)
seed0 undefended     eps=0.05 width1024  acc_under_G=0.6279  mean|d|=0.03119 max|d|=0.05000 (9s)
seed0 undefended     eps=0.05 lam5       acc_under_G=0.6689  mean|d|=0.02743 max|d|=0.05000 (9s)
seed0 undefended     eps=0.05 lam80      acc_under_G=0.6437  mean|d|=0.03373 max|d|=0.05000 (9s)
seed0 undefended     eps=0.05 steps400   acc_under_G=0.6869  mean|d|=0.03271 max|d|=0.05000 (4s)
seed0 undefended     eps=0.05 steps4800  acc_under_G=0.6563  mean|d|=0.02702 max|d|=0.05000 (29s)
seed0 undefended     eps=0.05 strong     acc_under_G=0.6202  mean|d|=0.03318 max|d|=0.05000 (29s)
seed0 undefended     eps=0.1 default    acc_under_G=0.1298  mean|d|=0.06012 max|d|=0.10000 (9

In [6]:
!pip install adversarial-robustness-toolbox --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 59.5 MB/s eta 0:00:00


In [7]:
# CICIDS2017 v2 - ART cross-check, mirroring nsl_art_crosscheck.py. Purpose: an independent, well-established
# attack library (Adversarial Robustness Toolbox) is run against the SAME models on the SAME fixed subset of
# test rows, to confirm our own hand-written PGD is not secretly weaker than it should be (the thing we'd want
# a reviewer to be able to check independently). CICIDS has no categorical columns, so - unlike the NSL-KDD
# version - there is no numeric-only/all-features distinction and no snapping-wrapper needed: every one of the
# 78 features is a real numeric flow statistic, so ART's mask (which NSL-KDD's cross-check discovered ART
# silently ignores) is not needed here since there is nothing to freeze in the first place.
#
# This runs on seed-0 models only (it is a sanity/validity check on the attack implementation, not part of the
# 5-seed statistical claims): undefended, fgsm_at, pgd_at, kd_feedback_r5. Uses ART's AutoProjectedGradientDescent
# (APGD-CE and APGD-DLR) - full AutoAttack failed on tabular data in the NSL-KDD check (SquareAttack requires
# image inputs), so APGD components are used and described in the paper as "components of AutoAttack", not
# "AutoAttack" itself, exactly as agreed for NSL-KDD.

# %% [0] Setup
from google.colab import drive
drive.mount('/content/drive')
import os, json
import numpy as np, tensorflow as tf
from tensorflow import keras

BASE = "/content/drive/MyDrive/A3IDPS_COLAB"
DATA = f"{BASE}/data_processed/cicids_v2"; MOD = f"{BASE}/models/cicids_v2"; RES = f"{BASE}/results/cicids_v2"

d = np.load(f"{DATA}/cicids_v2_primary.npz")
Xte, yte = d["Xte"].astype("float32"), d["yte"].astype("int32")
D, NCLS = Xte.shape[1], 5
CLASS_NAMES = ["BENIGN", "Brute Force", "DoS/DDoS", "PortScan", "Other"]

sel_at = json.load(open(f"{RES}/stage2b_selected_seed0.json"))
sel_gen = json.load(open(f"{RES}/stage3_selection_seed0.json"))
EPS_FGSM, EPS_PGD, EPS_MAX = sel_at["fgsm"], sel_at["pgd"], sel_gen["selected_eps"]

MODEL_PATHS = {
    "undefended":     f"{MOD}/undefended_smote_seed0.keras",
    "fgsm_at":         f"{MOD}/fgsm_eps{EPS_FGSM}_seed0.keras",
    "pgd_at":          f"{MOD}/pgd_eps{EPS_PGD}_seed0.keras",
    "kd_feedback_r5":  f"{MOD}/s3_kd_feedback_eps{EPS_MAX}_r5_seed0.keras",
}
EPS_LIST = [0.02, 0.05, 0.10]
N_SUBSET = 300

def jsonable(o):
    if isinstance(o, dict): return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [jsonable(v) for v in o]
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return float(o)
    return o

# %% [1] Fixed subset (same one used for every model, saved so it is reproducible / inspectable)
subset_path = f"{RES}/art_crosscheck_cicids_subset_idx.json"
if os.path.exists(subset_path):
    idx = np.array(json.load(open(subset_path)))
else:
    idx = np.random.RandomState(0).choice(len(Xte), size=N_SUBSET, replace=False)
    json.dump(idx.tolist(), open(subset_path, "w"))
Xs, ys = Xte[idx], yte[idx]
print(f"Fixed cross-check subset: {len(idx)} rows (seed 0, saved to {subset_path})")

# %% [2] Our own attack (verbatim logic from Stage 2b/3, re-run on this exact subset for a fair comparison)
def make_attacks(model):
    def loss_fn(z, y, margin):
        if margin:
            oh = tf.one_hot(y, NCLS)
            return tf.reduce_max(z - 1e9 * oh, axis=1) - tf.reduce_sum(z * oh, axis=1)
        return tf.nn.sparse_softmax_cross_entropy_with_logits(labels=y, logits=z)
    @tf.function
    def fgsm(x0, y, eps):
        with tf.GradientTape() as t:
            t.watch(x0); s = tf.reduce_sum(loss_fn(model(x0, training=False), y, False))
        return tf.clip_by_value(x0 + eps * tf.sign(t.gradient(s, x0)), 0., 1.)
    @tf.function
    def pgd(x0, y, eps, steps, restarts, margin):
        E = eps
        best_x, best_l = x0, tf.fill(tf.shape(y), -1e30)
        for r in range(restarts):
            x = x0 if r == 0 else tf.clip_by_value(x0 + tf.random.uniform(tf.shape(x0), -1., 1.) * E, 0., 1.)
            for s_ in range(steps):
                alpha = (1.0 if s_ < steps // 2 else 0.25) * E / 4.0
                with tf.GradientTape() as t:
                    t.watch(x); s = tf.reduce_sum(loss_fn(model(x, training=False), y, margin))
                x = tf.clip_by_value(tf.clip_by_value(x + alpha * tf.sign(t.gradient(s, x)), x0 - E, x0 + E), 0., 1.)
                l_new = loss_fn(model(x, training=False), y, margin)
                upd = l_new > best_l
                best_x, best_l = tf.where(upd[:, None], x, best_x), tf.where(upd, l_new, best_l)
        return best_x
    return {"fgsm": fgsm, "pgd": pgd}

def predict(model, X): return np.argmax(model.predict(X, batch_size=8192, verbose=0), axis=1)

def our_worst_case(model, X, y, eps):
    A = make_attacks(model)
    et = tf.constant(eps, tf.float32)
    c_f = predict(model, A["fgsm"](tf.constant(X), tf.constant(y), eps=et).numpy()) == y
    c_c = predict(model, A["pgd"](tf.constant(X), tf.constant(y), eps=et, steps=20, restarts=3, margin=False).numpy()) == y
    c_m = predict(model, A["pgd"](tf.constant(X), tf.constant(y), eps=et, steps=20, restarts=3, margin=True).numpy()) == y
    return float((c_f & c_c & c_m).mean())

# %% [3] ART cross-check (APGD-CE, APGD-DLR - components of AutoAttack; full AutoAttack fails on tabular data)
import art
from art.estimators.classification import TensorFlowV2Classifier
from art.attacks.evasion import AutoProjectedGradientDescent

print("ART version:", art.__version__)

def art_classifier(model):
    return TensorFlowV2Classifier(
        model=model, nb_classes=NCLS, input_shape=(D,),
        loss_object=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        clip_values=(0.0, 1.0))

results = {}
for tag, path in MODEL_PATHS.items():
    model = keras.models.load_model(path, compile=False)
    clf = art_classifier(model)
    results[tag] = {}
    for eps in EPS_LIST:
        ours = our_worst_case(model, Xs, ys, eps)
        row = {"our_worst_case": ours}
        for loss_type, label in [("cross_entropy", "apgd_ce"), ("difference_logits_ratio", "apgd_dlr")]:
            atk = AutoProjectedGradientDescent(estimator=clf, norm="inf", eps=eps, eps_step=eps / 4,
                                               max_iter=20, nb_random_init=3, loss_type=loss_type,
                                               batch_size=len(Xs), verbose=False)
            Xa = atk.generate(x=Xs, y=ys)
            max_delta = float(np.max(np.abs(Xa - Xs)))
            if max_delta > eps + 1e-3:
                print(f"  WARNING [{tag} eps={eps} {label}]: ART perturbation {max_delta:.4f} exceeds eps bound")
            art_acc = float((predict(model, Xa) == ys).mean())
            row[label] = art_acc
            flag = " <== ART STRONGER (>4pp)" if (ours - art_acc) > 0.04 else ""
            print(f"{tag:<16} eps={eps:<5} {label:<9} ART acc={art_acc:.4f}  ours worst-case={ours:.4f}{flag}")
        results[tag][str(eps)] = row
        json.dump(jsonable(results), open(f"{RES}/art_crosscheck_cicids_results.json", "w"), indent=1)

# %% [4] Summary
print("\n=== Summary: cases where ART's APGD beat our worst-case by more than 4pp ===")
any_flag = False
for tag, by_eps in results.items():
    for eps, row in by_eps.items():
        for label in ("apgd_ce", "apgd_dlr"):
            gap = row["our_worst_case"] - row[label]
            if gap > 0.04:
                any_flag = True
                print(f"  {tag} eps={eps} {label}: ART acc={row[label]:.4f} vs ours={row['our_worst_case']:.4f} (gap {gap*100:.1f}pp)")
if not any_flag:
    print("  none - our attack implementation is at least as strong as ART's APGD components on every model/eps checked.")
print(f"\nSaved: {RES}/art_crosscheck_cicids_results.json")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Fixed cross-check subset: 300 rows (seed 0, saved to /content/drive/MyDrive/A3IDPS_COLAB/results/cicids_v2/art_crosscheck_cicids_subset_idx.json)
ART version: 1.20.1
undefended       eps=0.02  apgd_ce   ART acc=0.7700  ours worst-case=0.7467


undefended       eps=0.02  apgd_dlr  ART acc=0.7467  ours worst-case=0.7467


undefended       eps=0.05  apgd_ce   ART acc=0.5500  ours worst-case=0.5400
undefended       eps=0.05  apgd_dlr  ART acc=0.5433  ours worst-case=0.5400


undefended       eps=0.1   apgd_ce   ART acc=0.1500  ours worst-case=0.1067
undefended       eps=0.1   apgd_dlr  ART acc=0.1900  ours worst-case=0.1067
fgsm_at          eps=0.02  apgd_ce   ART acc=0.9633  ours worst-case=0.9633


fgsm_at          eps=0.02  apgd_dlr  ART acc=0.9633  ours worst-case=0.9633
fgsm_at          eps=0.05  apgd_ce   ART acc=0.7933  ours worst-case=0.8033


fgsm_at          eps=0.05  apgd_dlr  ART acc=0.8133  ours worst-case=0.8033
fgsm_at          eps=0.1   apgd_ce   ART acc=0.4567  ours worst-case=0.4433
fgsm_at          eps=0.1   apgd_dlr  ART acc=0.5100  ours worst-case=0.4433
pgd_at           eps=0.02  apgd_ce   ART acc=0.9367  ours worst-case=0.9367
pgd_at           eps=0.02  apgd_dlr  ART acc=0.9367  ours worst-case=0.9367
pgd_at           eps=0.05  apgd_ce   ART acc=0.8633  ours worst-case=0.8633
pgd_at           eps=0.05  apgd_dlr  ART acc=0.8933  ours worst-case=0.8633
pgd_at           eps=0.1   apgd_ce   ART acc=0.8000  ours worst-case=0.7900
pgd_at           eps=0.1   apgd_dlr  ART acc=0.8033  ours worst-case=0.7900
kd_feedback_r5   eps=0.02  apgd_ce   ART acc=0.8867  ours worst-case=0.8833
kd_feedback_r5   eps=0.02  apgd_dlr  ART acc=0.8967  ours worst-case=0.8833
kd_feedback_r5   eps=0.05  apgd_ce   ART acc=0.7667  ours worst-case=0.5500
kd_feedback_r5   eps=0.05  apgd_dlr  ART acc=0.5700  ours worst-case=0.5500
kd_feedback_

In [8]:
# CICIDS2017 v2 - Final correction: fold the Stage 4b AdvGAN-attack sweep into the Stage 4a worst-case numbers.
# Rationale: Stage 4b found 3/320 cases (out of 5 seeds x 4 models x 2 eps x 8 generator configs) where the
# AdvGAN-attack sweep found a slightly stronger attack (lower accuracy) than the white-box FGSM/PGD ensemble
# Stage 4a reports as "worst-case". To make every worst-case number in the paper a true minimum over every
# attack actually tried (the strongest possible answer to "did you use a strong enough attack"), this script
# recomputes corrected_worst = min(stage4a_worst, best_advgan_attack_accuracy) for the 4 models Stage 4b covers
# (undefended, fgsm_at, pgd_at, kd_feedback_r5) at the 2 eps values Stage 4b covers (0.05, 0.1). Every other
# cell (other models, other eps) is copied through unchanged - Stage 4b never attacked them, so there is
# nothing to correct. Read-only over existing results; writes one new summary file, does not touch stage4a_*.

import json, os
import numpy as np

BASE = "/content/drive/MyDrive/A3IDPS_COLAB"
RES = f"{BASE}/results/cicids_v2"

SEEDS = [0, 1, 2, 3, 4]
sel_at = json.load(open(f"{RES}/stage2b_selected_seed0.json"))
EPS_FGSM, EPS_PGD = sel_at["fgsm"], sel_at["pgd"]

# maps Stage 4b's short model tags -> the tag names used in stage4a_seed{s}.json
S4B_TO_S4A_TAG = {
    "undefended": "undefended",
    "fgsm_at": f"fgsm_at_eps{EPS_FGSM}",
    "pgd_at": f"pgd_at_eps{EPS_PGD}",
    "kd_feedback_r5": "kd_feedback_r5",
}
CORRECTED_EPS = ["0.05", "0.1"]

s4a = {s: json.load(open(f"{RES}/stage4a_seed{s}.json")) for s in SEEDS}
s4b = json.load(open(f"{RES}/stage4b_results.json"))

# every tag present in seed 0's stage4a output, in a sensible print order
ALL_TAGS = list(s4a[0].keys())
EPS_ALL = sorted(s4a[0][ALL_TAGS[0]]["robust"].keys(), key=float)

def s4a_worst(seed, tag, eps):
    return s4a[seed][tag]["robust"][eps]["worst"]

def best_advgan_acc(seed, s4b_tag, eps):
    cfgs = s4b[str(seed)][s4b_tag]["configs"]
    return min(cfgs[c][eps]["acc_under_generator"] for c in cfgs if eps in cfgs.get(c, {}))

# %% Build the corrected per-seed, per-tag, per-eps worst-case table
corrections = []   # log of every cell actually changed, for transparency
corrected = {tag: {eps: [] for eps in EPS_ALL} for tag in ALL_TAGS}

for tag in ALL_TAGS:
    s4b_tag = next((k for k, v in S4B_TO_S4A_TAG.items() if v == tag), None)
    for eps in EPS_ALL:
        for seed in SEEDS:
            wb = s4a_worst(seed, tag, eps)
            if s4b_tag is not None and eps in CORRECTED_EPS:
                adv = best_advgan_acc(seed, s4b_tag, eps)
                final = min(wb, adv)
                if adv < wb - 1e-9:
                    corrections.append({"seed": seed, "tag": tag, "eps": eps,
                                        "white_box_worst": wb, "advgan_acc": adv, "corrected_worst": final})
            else:
                final = wb   # untouched - Stage 4b never attacked this (tag, eps) combination
            corrected[tag][eps].append(final)

# %% Aggregate mean +- std across seeds, and print the final corrected table (same format as Stage 4a's)
print(f"=== FINAL 5-SEED TEST RESULTS, corrected worst-case = min(white-box, AdvGAN-sweep) where both exist ===")
print(f"{'model':<24}{'clean':>12}" + "".join(f"{('e='+e):>13}" for e in EPS_ALL))
agg_table = {}
for tag in ALL_TAGS:
    clean_vals = [s4a[s][tag]["clean"]["acc"] for s in SEEDS]
    cm, cs = float(np.mean(clean_vals)), float(np.std(clean_vals))
    row = {"clean": (cm, cs)}
    line = f"{tag:<24}{cm*100:>8.1f}+-{cs*100:<3.1f}"
    for eps in EPS_ALL:
        vals = corrected[tag][eps]
        m_, s_ = float(np.mean(vals)), float(np.std(vals))
        row[eps] = (m_, s_)
        line += f"  {m_*100:>6.1f}+-{s_*100:<3.1f}"
    print(line)
    agg_table[tag] = row

print(f"\n=== Cells actually changed by the Stage 4b correction ({len(corrections)} of {len(ALL_TAGS)*len(CORRECTED_EPS)*len(SEEDS)} checked) ===")
if corrections:
    for c in corrections:
        print(f"  seed{c['seed']} {c['tag']:<18} eps={c['eps']}: white-box={c['white_box_worst']*100:.2f}%  "
              f"-> AdvGAN found {c['advgan_acc']*100:.2f}%  (corrected worst-case: {c['corrected_worst']*100:.2f}%)")
else:
    print("  none - AdvGAN sweep never beat the white-box worst-case for any checked cell.")

# %% Paired differences on the CORRECTED numbers (same comparisons as Stage 4a, recomputed)
def paired(tag_a, tag_b, eps):
    diffs = [corrected[tag_a][eps][i] - corrected[tag_b][eps][i] for i in range(len(SEEDS))]
    return float(np.mean(diffs)), float(np.std(diffs)), sum(1 for x in diffs if x > 0)

print(f"\n=== Paired differences on CORRECTED worst-case (kd_feedback_r5 vs others, eps=0.05 and 0.1) ===")
comparisons = [("kd_feedback_r5", "undefended"), ("kd_feedback_r5", f"pgd_at_eps{EPS_PGD}"),
               ("kd_feedback_r5", "kd_fixed_gen_r5"), ("kd_feedback_r5", "nokd_feedback_r5")]
paired_table = {}
for a, b in comparisons:
    entry = {}
    for eps in ("0.05", "0.1"):
        m_, s_, npos = paired(a, b, eps)
        entry[eps] = {"mean": m_, "std": s_, "n_positive": npos}
        print(f"{a} - {b}  @eps={eps}: {m_*100:+.1f}+-{s_*100:.1f} pp  ({npos}/{len(SEEDS)} seeds positive)")
    paired_table[f"{a}__minus__{b}"] = entry

def jsonable(o):
    if isinstance(o, dict): return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [jsonable(v) for v in o]
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return float(o)
    return o

json.dump(jsonable({"aggregate_corrected": agg_table, "corrections_applied": corrections,
                    "paired_differences_corrected": paired_table}),
          open(f"{RES}/stage4_final_corrected.json", "w"), indent=1)
print(f"\nSaved: {RES}/stage4_final_corrected.json")

=== FINAL 5-SEED TEST RESULTS, corrected worst-case = min(white-box, AdvGAN-sweep) where both exist ===
model                          clean       e=0.01       e=0.02       e=0.05        e=0.1       e=0.15
undefended                  97.4+-4.0    94.8+-5.3    82.6+-2.7    24.6+-16.4     3.2+-2.9     0.0+-0.0
fgsm_at_eps0.05             99.1+-0.2    98.0+-0.1    96.1+-0.3    81.4+-1.8    44.6+-5.4     8.0+-1.7
pgd_at_eps0.1               97.6+-0.1    97.1+-0.1    95.7+-0.3    89.1+-0.2    84.0+-0.4    56.2+-1.7
teacher                     99.5+-0.1    98.0+-0.2    87.7+-1.3    23.3+-6.7     4.4+-1.3     0.2+-0.2
kd_feedback_r0              99.5+-0.1    96.9+-0.2    81.2+-1.8    15.5+-3.5     1.2+-0.6     0.0+-0.0
kd_feedback_r1              99.6+-0.0    97.4+-0.1    83.4+-2.1    30.8+-7.9     5.6+-1.0     0.3+-0.2
kd_feedback_r2              99.7+-0.1    97.7+-0.2    86.6+-1.2    53.6+-3.1     8.4+-0.8     1.6+-0.6
kd_feedback_r3              99.6+-0.1    98.0+-0.1    88.0+-0.8    55.7+

In [12]:
# Confusion matrices for both datasets, clean AND under worst-case attack at eps=0.05 (the eps both papers'
# headline tables report). Covers 4 models per dataset (undefended, fgsm_at, pgd_at, kd_feedback_r5, seed 0 -
# the same seed used for any single-seed illustration in the manuscript; the 5-seed tables already give the
# statistical claim, this gives the per-class qualitative picture Reviewer 2 asked for). Saves both a PNG heatmap
# per (dataset, model, condition) and the raw counts as JSON (for building a LaTeX table instead, if preferred).
# Self-contained, read-only over existing saved models - no training.

# %% [0] Setup
from google.colab import drive
drive.mount('/content/drive')
import os, json
import numpy as np, tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

BASE = "/content/drive/MyDrive/A3IDPS_COLAB"
OUT = f"{BASE}/results/confusion_matrices"
os.makedirs(OUT, exist_ok=True)

DATASETS = {
    "nsl": {
        "data": f"{BASE}/data_processed/nslkdd_v2/nsl_v2_arrays.npz",
        "mod": f"{BASE}/models/nsl_v2", "res": f"{BASE}/results/nsl_v2",
        "class_names": ["Normal", "DoS", "Probe", "R2L", "U2R"],
        "mask_categorical": True,  # NSL has one-hot categorical columns to freeze during the attack
    },
    "cicids": {
        "data": f"{BASE}/data_processed/cicids_v2/cicids_v2_primary.npz",
        "mod": f"{BASE}/models/cicids_v2", "res": f"{BASE}/results/cicids_v2",
        "class_names": ["BENIGN", "Brute Force", "DoS/DDoS", "PortScan", "Other"],
        "mask_categorical": False,
    },
}
EPS = 0.05
SEED = 0

def make_attacks(model, mask, ncls):
    M = tf.constant(mask, tf.float32)
    def loss_fn(z, y):
        return tf.nn.sparse_softmax_cross_entropy_with_logits(labels=y, logits=z)
    @tf.function
    def pgd(x0, y, eps, steps=20, restarts=3):
        E = eps * M
        best_x, best_l = x0, tf.fill(tf.shape(y), -1e30)
        for r in range(restarts):
            x = x0 if r == 0 else tf.clip_by_value(x0 + tf.random.uniform(tf.shape(x0), -1., 1.) * E, 0., 1.)
            for s_ in range(steps):
                alpha = (1.0 if s_ < steps // 2 else 0.25) * E / 4.0
                with tf.GradientTape() as t:
                    t.watch(x); s = tf.reduce_sum(loss_fn(model(x, training=False), y))
                x = tf.clip_by_value(tf.clip_by_value(x + alpha * tf.sign(t.gradient(s, x)), x0 - E, x0 + E), 0., 1.)
                l_new = loss_fn(model(x, training=False), y)
                upd = l_new > best_l
                best_x, best_l = tf.where(upd[:, None], x, best_x), tf.where(upd, l_new, best_l)
        return best_x
    return pgd

def predict(model, X): return np.argmax(model.predict(X, batch_size=8192, verbose=0), axis=1)

def plot_cm(cm, class_names, title, path):
    cm_norm = cm.astype("float64") / cm.sum(axis=1, keepdims=True).clip(min=1)
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(class_names))); ax.set_xticklabels(class_names, rotation=45, ha="right")
    ax.set_yticks(range(len(class_names))); ax.set_yticklabels(class_names)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title)
    for i in range(len(class_names)):
        for j in range(len(class_names)):
            txt = f"{cm[i,j]}\n({cm_norm[i,j]*100:.1f}%)"
            ax.text(j, i, txt, ha="center", va="center", fontsize=7,
                    color="white" if cm_norm[i, j] > 0.5 else "black")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout(); fig.savefig(path, dpi=150); plt.close(fig)

# %% [1] Run for both datasets
all_results = {}
for ds_name, cfg in DATASETS.items():
    print(f"\n========== {ds_name.upper()} ==========")
    d = np.load(cfg["data"])
    Xte, yte = d["Xte"].astype("float32"), d["yte"].astype("int32")
    D, ncls = Xte.shape[1], len(cfg["class_names"])
    if cfg["mask_categorical"]:
        names = json.load(open(f"{BASE}/data_processed/nslkdd_v2/feature_names.json"))
        mask = np.array([0. if n.startswith(("protocol_type_", "service_", "flag_")) else 1. for n in names], "float32")
    else:
        mask = np.ones(D, "float32")

    model_paths = None
    if ds_name == "cicids":
        eps_fgsm, eps_pgd = json.load(open(f"{cfg['res']}/stage2b_selected_seed0.json")).values()
        eps_max = json.load(open(f"{cfg['res']}/stage3_selection_seed0.json"))["selected_eps"]
        model_paths = {
            "undefended": f"{cfg['mod']}/undefended_smote_seed0.keras",
            "fgsm_at": f"{cfg['mod']}/fgsm_eps{eps_fgsm}_seed0.keras",
            "pgd_at": f"{cfg['mod']}/pgd_eps{eps_pgd}_seed0.keras",
            "kd_feedback_r5": f"{cfg['mod']}/s3_kd_feedback_eps{eps_max}_r5_seed0.keras",
        }
    else:
        s2 = json.load(open(f"{cfg['res']}/stage2_test_seed0.json"))
        eps_fgsm, eps_pgd = s2["selected"]["fgsm"], s2["selected"]["pgd"]
        eps_max = json.load(open(f"{cfg['res']}/stage3_selection_seed0.json"))["selected_eps"]
        # NSL Stage 2 saves as f"{MOD}/s2_{tag}_seed{seed}.keras" where tag is "none" for undefended, or
        # "{kind}_eps{eps_train}" for fgsm/pgd - confirmed against nsl_stage2.py's train_model().
        # kd_feedback_r5 is saved by nsl_stage4a_seeds.py under its OWN naming ("s4_" prefix, no eps in the
        # name) rather than reusing Stage 3's "s3_..._eps{e}_..." pattern - confirmed via directory listing.
        model_paths = {
            "undefended": f"{cfg['mod']}/s2_none_seed0.keras",
            "fgsm_at": f"{cfg['mod']}/s2_fgsm_eps{eps_fgsm}_seed0.keras",
            "pgd_at": f"{cfg['mod']}/s2_pgd_eps{eps_pgd}_seed0.keras",
            "kd_feedback_r5": f"{cfg['mod']}/s4_kd_feedback_r5_seed0.keras",
        }

    ds_results = {}
    for tag, path in model_paths.items():
        if not os.path.exists(path):
            print(f"  SKIP {tag}: model file not found at {path} - check the path convention for this dataset")
            continue
        model = keras.models.load_model(path, compile=False)

        p_clean = predict(model, Xte)
        cm_clean = confusion_matrix(yte, p_clean, labels=range(ncls))
        plot_cm(cm_clean, cfg["class_names"], f"{ds_name.upper()} - {tag} - clean",
               f"{OUT}/{ds_name}_{tag}_clean.png")

        pgd = make_attacks(model, mask, ncls)
        Xa = np.vstack([pgd(tf.constant(Xte[i:i+4096]), tf.constant(yte[i:i+4096]), eps=EPS).numpy()
                        for i in range(0, len(Xte), 4096)])
        p_adv = predict(model, Xa)
        cm_adv = confusion_matrix(yte, p_adv, labels=range(ncls))
        plot_cm(cm_adv, cfg["class_names"], f"{ds_name.upper()} - {tag} - PGD eps={EPS}",
               f"{OUT}/{ds_name}_{tag}_adv_eps{EPS}.png")

        ds_results[tag] = {"clean_cm": cm_clean.tolist(), "adv_cm": cm_adv.tolist(),
                           "clean_acc": float((p_clean == yte).mean()), "adv_acc": float((p_adv == yte).mean())}
        print(f"  {tag:<16} clean_acc={ds_results[tag]['clean_acc']:.4f}  acc@eps={EPS}={ds_results[tag]['adv_acc']:.4f}")

    all_results[ds_name] = ds_results
    json.dump(all_results[ds_name], open(f"{OUT}/{ds_name}_confusion_matrices.json", "w"), indent=1)

print(f"\nSaved PNGs and JSON summaries to {OUT}/")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

========== NSL ==========
  undefended       clean_acc=0.7943  acc@eps=0.05=0.4592
  fgsm_at          clean_acc=0.7874  acc@eps=0.05=0.7543


  pgd_at           clean_acc=0.7946  acc@eps=0.05=0.7750


  kd_feedback_r5   clean_acc=0.7799  acc@eps=0.05=0.5646

========== CICIDS ==========
  undefended       clean_acc=0.8929  acc@eps=0.05=0.5900
  fgsm_at          clean_acc=0.9883  acc@eps=0.05=0.8251
  pgd_at           clean_acc=0.9757  acc@eps=0.05=0.8951
  kd_feedback_r5   clean_acc=0.9957  acc@eps=0.05=0.7456

Saved PNGs and JSON summaries to /content/drive/MyDrive/A3IDPS_COLAB/results/confusion_matrices/


In [11]:
import os
p = "/content/drive/MyDrive/A3IDPS_COLAB/models/nsl_v2"
print(sorted(f for f in os.listdir(p) if "kd_feedback" in f))

['s3_gen_kd_feedback_eps0.02_r1_seed0.keras', 's3_gen_kd_feedback_eps0.02_r2_seed0.keras', 's3_gen_kd_feedback_eps0.02_r3_seed0.keras', 's3_gen_kd_feedback_eps0.05_r1_seed0.keras', 's3_gen_kd_feedback_eps0.05_r2_seed0.keras', 's3_gen_kd_feedback_eps0.05_r3_seed0.keras', 's3_gen_kd_feedback_eps0.1_r1_seed0.keras', 's3_gen_kd_feedback_eps0.1_r2_seed0.keras', 's3_gen_kd_feedback_eps0.1_r3_seed0.keras', 's3_gen_nokd_feedback_eps0.05_r1_seed0.keras', 's3_gen_nokd_feedback_eps0.05_r2_seed0.keras', 's3_gen_nokd_feedback_eps0.05_r3_seed0.keras', 's3_kd_feedback_eps0.02_r1_seed0.keras', 's3_kd_feedback_eps0.02_r2_seed0.keras', 's3_kd_feedback_eps0.02_r3_seed0.keras', 's3_kd_feedback_eps0.05_r1_seed0.keras', 's3_kd_feedback_eps0.05_r2_seed0.keras', 's3_kd_feedback_eps0.05_r3_seed0.keras', 's3_kd_feedback_eps0.1_r1_seed0.keras', 's3_kd_feedback_eps0.1_r2_seed0.keras', 's3_kd_feedback_eps0.1_r3_seed0.keras', 's3_nokd_feedback_eps0.05_r1_seed0.keras', 's3_nokd_feedback_eps0.05_r2_seed0.keras', 's3_

In [14]:
# SHAP re-analysis on the CORRECTED models, for both datasets. Fixes the axis-averaging bug diagnosed early on
# (shap_vals came back shaped (n_samples, n_features, n_classes); the original code averaged over the wrong
# axis, producing meaningless "top features"). Uses KernelExplainer's model-agnostic wrapper around the Keras
# model's predicted-class probabilities via a small background sample (standard for tabular NNs; a
# DeepExplainer/GradientExplainer would need a TF1-style graph, not worth fighting for a logits-output tf.keras
# model here). Reports per-class mean(|SHAP value|) top-15 features for undefended vs kd_feedback_r5, so the
# paper can show whether/how A3IDPS's training changes which features the model actually relies on.

# %% [0] Setup
from google.colab import drive
drive.mount('/content/drive')
import os, json
import numpy as np, tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt

try:
    import shap
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "shap", "--quiet"])
    import shap

BASE = "/content/drive/MyDrive/A3IDPS_COLAB"
OUT = f"{BASE}/results/shap"
os.makedirs(OUT, exist_ok=True)

N_BACKGROUND = 100     # background sample for KernelExplainer (kept small - KernelExplainer is O(features x background))
N_EXPLAIN = 200         # test rows to explain per model (kept small for runtime; increase if Colab time allows)
TOP_K = 15

DATASETS = {
    "nsl": {
        "data": f"{BASE}/data_processed/nslkdd_v2/nsl_v2_arrays.npz",
        "mod": f"{BASE}/models/nsl_v2", "res": f"{BASE}/results/nsl_v2",
        "class_names": ["Normal", "DoS", "Probe", "R2L", "U2R"],
        "feature_names": json.load(open(f"{BASE}/data_processed/nslkdd_v2/feature_names.json")),
    },
    "cicids": {
        "data": f"{BASE}/data_processed/cicids_v2/cicids_v2_primary.npz",
        "mod": f"{BASE}/models/cicids_v2", "res": f"{BASE}/results/cicids_v2",
        "class_names": ["BENIGN", "Brute Force", "DoS/DDoS", "PortScan", "Other"],
        "feature_names": json.load(open(f"{BASE}/data_processed/cicids_v2/feature_names.json")),
    },
}

def get_model_paths(ds_name, cfg):
    if ds_name == "cicids":
        eps_fgsm, eps_pgd = json.load(open(f"{cfg['res']}/stage2b_selected_seed0.json")).values()
        eps_max = json.load(open(f"{cfg['res']}/stage3_selection_seed0.json"))["selected_eps"]
        return {
            "undefended": f"{cfg['mod']}/undefended_smote_seed0.keras",
            "kd_feedback_r5": f"{cfg['mod']}/s3_kd_feedback_eps{eps_max}_r5_seed0.keras",
        }
    else:
        # NSL Stage 2 saves undefended as f"{MOD}/s2_none_seed0.keras" (confirmed against nsl_stage2.py).
        # kd_feedback_r5 is saved by nsl_stage4a_seeds.py under its own naming ("s4_" prefix, no eps in the
        # name), not Stage 3's "s3_..._eps{e}_..." pattern - confirmed via directory listing.
        return {
            "undefended": f"{cfg['mod']}/s2_none_seed0.keras",
            "kd_feedback_r5": f"{cfg['mod']}/s4_kd_feedback_r5_seed0.keras",
        }

# %% [1] Per-dataset, per-model SHAP (KernelExplainer over predicted class probabilities, correct axis handling)
all_top_features = {}
for ds_name, cfg in DATASETS.items():
    print(f"\n========== {ds_name.upper()} ==========")
    d = np.load(cfg["data"])
    Xte = d["Xte"].astype("float32")
    D = Xte.shape[1]
    ncls = len(cfg["class_names"])
    feat_names = cfg["feature_names"] if cfg["feature_names"] else [f"f{i}" for i in range(D)]
    assert len(feat_names) == D, f"{ds_name}: feature name count ({len(feat_names)}) != D ({D})"

    rng = np.random.RandomState(0)
    bg_idx = rng.choice(len(Xte), N_BACKGROUND, replace=False)
    ex_idx = rng.choice(len(Xte), N_EXPLAIN, replace=False)
    background, X_explain = Xte[bg_idx], Xte[ex_idx]

    model_paths = get_model_paths(ds_name, cfg)
    ds_top = {}
    for tag, path in model_paths.items():
        if not os.path.exists(path):
            print(f"  SKIP {tag}: model not found at {path}"); continue
        model = keras.models.load_model(path, compile=False)
        def f(x):  # KernelExplainer needs a plain numpy-in numpy-out probability function
            return tf.nn.softmax(model.predict(x, batch_size=4096, verbose=0), axis=1).numpy()

        explainer = shap.KernelExplainer(f, background)
        shap_vals = explainer.shap_values(X_explain, nsamples="auto")
        # shap_vals can come back as either a list of (n_explain, D) arrays (one per class) or a single
        # (n_explain, D, n_classes) array depending on the shap version - handle both, and NEVER average over
        # the wrong axis (the bug that broke the original analysis): we want, per class c, mean over samples of
        # |shap value for feature j, class c| - i.e. mean over axis 0 only, keeping features and classes separate.
        if isinstance(shap_vals, list):
            sv = np.stack(shap_vals, axis=-1)   # (n_explain, D, n_classes)
        else:
            sv = shap_vals
        assert sv.shape == (len(X_explain), D, ncls), f"unexpected SHAP output shape {sv.shape}, expected {(len(X_explain), D, ncls)}"

        mean_abs = np.mean(np.abs(sv), axis=0)   # (D, n_classes) - correct axis: average over SAMPLES only
        ds_top[tag] = {}
        for c, cname in enumerate(cfg["class_names"]):
            order = np.argsort(-mean_abs[:, c])[:TOP_K]
            ds_top[tag][cname] = [{"feature": feat_names[j], "mean_abs_shap": float(mean_abs[j, c])} for j in order]

        # one summary plot per model: overall top-15 features by mean(|SHAP|) averaged across classes too
        overall = mean_abs.mean(axis=1)
        order = np.argsort(-overall)[:TOP_K]
        fig, ax = plt.subplots(figsize=(7, 5))
        ax.barh([feat_names[j] for j in order][::-1], overall[order][::-1])
        ax.set_xlabel("mean(|SHAP value|), averaged across classes")
        ax.set_title(f"{ds_name.upper()} - {tag} - top {TOP_K} features")
        fig.tight_layout(); fig.savefig(f"{OUT}/{ds_name}_{tag}_shap_top{TOP_K}.png", dpi=150); plt.close(fig)
        print(f"  {tag}: top-3 overall features = {[feat_names[j] for j in order[:3]]}")

    all_top_features[ds_name] = ds_top
    json.dump(all_top_features[ds_name], open(f"{OUT}/{ds_name}_shap_top_features.json", "w"), indent=1)

print(f"\nSaved plots and JSON to {OUT}/")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

========== NSL ==========


  0%|          | 0/200 [00:00<?, ?it/s]

  undefended: top-3 overall features = ['dst_host_rerror_rate', 'dst_host_srv_count', 'dst_host_same_srv_rate']


  0%|          | 0/200 [00:00<?, ?it/s]

  kd_feedback_r5: top-3 overall features = ['dst_host_rerror_rate', 'service_http', 'dst_host_same_srv_rate']

========== CICIDS ==========


  0%|          | 0/200 [00:00<?, ?it/s]

  undefended: top-3 overall features = ['Init_Win_bytes_forward', 'ACK Flag Count', 'Bwd Packet Length Min']


  0%|          | 0/200 [00:00<?, ?it/s]

  kd_feedback_r5: top-3 overall features = ['PSH Flag Count', 'Init_Win_bytes_forward', 'ACK Flag Count']

Saved plots and JSON to /content/drive/MyDrive/A3IDPS_COLAB/results/shap/
